<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_04_feature_engineering/stage_04_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_04_feature_engineering**

## **Introducción**

Esta notebook corresponde al stage_04a - Technical Indicators del pipeline neural_profit y tiene como objetivo generar, evaluar y consolidar indicadores técnicos intradía para el índice MNQ, a partir de datos minuto a minuto.

Partiendo del dataset intradía ya etiquetado con objetivos de retorno, se calculan indicadores técnicos de forma independiente por jornada, evitando la mezcla de información entre días. Esto garantiza consistencia temporal y previene leakage en etapas posteriores de modelado.

El proceso incluye la evaluación cuantitativa de los indicadores mediante Information Coefficient (IC), utilizando correlación de Spearman entre cada indicador y los targets de retorno definidos para distintos horizontes. Este análisis permite medir no solo la relación promedio con el target, sino también su estabilidad a lo largo del tiempo.

Como resultado final, se generan datasets consolidados y listos para modelado, que incluyen:

- Variables OHLCV
- Targets de retorno a distintos horizontes
- Indicadores técnicos seleccionados y validados

Estos artefactos serán utilizados en las siguientes etapas del pipeline para selección de features, entrenamiento y evaluación de modelos predictivos.

## **0. Configuración del Entorno**


### 0.1. Clonado de repositorio / Acceso a Drive

In [37]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [38]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [40]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

Librería instalada: technical-analysis


### 0.3. Importación de librerías


In [41]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

### 0.4. Definición de rutas



In [42]:
#/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [43]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/processed/mnq_intraday_labeled.parquet"))
IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_03b_target_definition_summary.json"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/features/mnq_features_target.parquet"))
OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/stage_04_feature_engineering_summary.json"))

In [44]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

### 0.5. Códigos auxiliares para carga de datos y visualización


In [45]:
def load_data():

    # Definir la URL del archivo Parquet en Drive
    data_path = f'{drive_path}/data/processed/mnq_intraday_labeled.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [46]:
def info_dataset(df):
  print("Información del dataset:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}")

  return primer_hora, ultima_hora

In [47]:
# Carga del JSON
with IN_ARTIFACT.open("r") as f:
    target_definition_summary = json.load(f)

In [48]:
def print_summary_console(summary: Dict[str, Any]) -> None:
    """Consola legible (sin depender de pandas display)."""
    print("\n" + "=" * 70)
    print(f"STAGE: {summary.get('stage')}")
    print(f"CREATED_AT_UTC: {summary.get('created_at_utc')}")
    print(f"VERSION: {summary.get('version')}")
    print("-" * 70)

    paths = summary.get("paths", {})
    print("[PATHS]")
    for group in ("inputs", "outputs", "reports"):
        print(f"  {group}:")
        for k, v in paths.get(group, {}).items():
            print(f"    - {k}: {v}")

    params = summary.get("params", {})
    print("\n[PARAMS]")
    for k, v in params.items():
        print(f"  - {k}: {v}")

    metrics = summary.get("metrics", {})
    print("\n[METRICS]")
    for k, v in metrics.items():
        print(f"  - {k}: {v}")

    details = summary.get("details", {})
    gw = details.get("gestation_window", {})
    if gw:
        print("\n[GESTATION WINDOW]")
        print(f"  - start: {gw.get('start_hhmm')}")
        print(f"  - end  : {gw.get('end_hhmm')}")
        print(f"  - top_n: {gw.get('top_n')}")
        print(f"  - minute_of_day_min: {gw.get('minute_of_day_min')}")
        print(f"  - minute_of_day_max: {gw.get('minute_of_day_max')}")

    print("=" * 70 + "\n")

In [49]:
def assign_indicator_family(indicator: str) -> str:
    """
    Asigna una familia económica a cada indicador técnico
    en función de su nombre.
    """
    name = indicator.lower()

    if "ema" in name or name.startswith("price_"):
        return "trend_price"

    if name.startswith("bb_"):
        return "volatility_extension"

    if name.startswith("roc") or name.startswith("momentum"):
        return "momentum"

    if name.startswith("rsi"):
        return "momentum_oscillator"

    if name.startswith("stoch"):
        return "momentum_oscillator"

    if name.startswith("atr"):
        return "volatility"

    if name.startswith("volume_ratio"):
        return "volume"

    if name == "macd":
        return "trend_momentum"

    return "other"

In [50]:
def select_top_by_family(
    ic_df: pd.DataFrame,
    top_n: int = 1
) -> pd.DataFrame:
    """
    Selecciona los mejores indicadores por familia
    según abs_IC_delta.
    """
    df = ic_df.copy()

    # Asignar familia
    df["family"] = df["indicator"].apply(assign_indicator_family)

    # Ordenar por fuerza de señal
    df = df.sort_values("abs_IC_delta", ascending=False)

    # Tomar top N por familia
    df_top = (
        df.groupby("family", as_index=False)
          .head(top_n)
          .reset_index(drop=True)
    )

    return df_top

In [51]:
def load_dataset():
  #Carga de dataset base:
  mnq_intraday_labeled = load_data()
  #info_dataset(mnq_intraday_labeled)
  # Eliminación de filas con NaN
  mnq_intraday_labeled = mnq_intraday_labeled.dropna()
  # Verificación posterior
  info_dataset(mnq_intraday_labeled)
  return mnq_intraday_labeled

In [54]:
mnq_intraday_labeled = load_dataset()

# Copiamos para trabajar de forma segura
df = mnq_intraday_labeled.copy()

# -----------------------------
# 1) Selección de columnas base
# -----------------------------
base_cols = ["date", "open", "high", "low", "close", "volume"]

# Targets: todas las columnas que empiezan con 'ret_'
target_cols = [c for c in df.columns if c.startswith("ret")]

# Subset final
df = df[base_cols + target_cols].copy()

# -----------------------------
# 2) Renombrar delta_pts_h -> ret_h
# -----------------------------
rename_map = {
    c: c.replace("ret_", "ret_")
    for c in target_cols
}

df.rename(columns=rename_map, inplace=True)

# -----------------------------
# 3) Sobrescribimos el dataset
# -----------------------------
mnq_intraday_labeled = df.copy()

mnq_intraday_labeled

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 481
	Hora diaria de inicio 06:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York


,date,open,high,low,close,volume,ret_60,ret_90
datetime,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7,0.001031,0.000687
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89,0.001060,0.000859
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34,0.001117,0.000917
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53,0.000974,0.000917
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3,0.000917,0.000888
...,...,...,...,...,...,...,...,...
2025-06-13 14:26:00-04:00,2025-06-13,21716.50,21722.75,21712.00,21719.50,1036,-0.004144,-0.004696
2025-06-13 14:27:00-04:00,2025-06-13,21719.00,21719.75,21695.75,21698.50,3542,-0.003191,-0.003445
2025-06-13 14:28:00-04:00,2025-06-13,21698.25,21700.25,21670.50,21679.25,5241,-0.002410,-0.002652


#**1. Relación entre indicadores técnicos, information coefficient y los targets definidos del stage_03**

### **1.1. Indicadores técnicos**

Los indicadores técnicos se construyen a partir de la serie de precios intradía, y en particular sobre la variable de cierre (close). Estas transformaciones matemáticas buscan capturar propiedades dinámicas del mercado tales como tendencia, momentum, reversión, volatilidad y estructura temporal del movimiento de precios.

En el marco actual del proyecto, la variable objetivo (target) se define como el retorno simple intradía a un horizonte futuro discreto, calculado sobre el precio de cierre, para horizontes temporales de 60 y 90 minutos. Este retorno representa la variación relativa del precio entre el instante actual y el instante futuro correspondiente al horizonte considerado.

En consecuencia, el problema de modelado se plantea como una tarea de predicción directa de retornos, sin discretización en umbrales ni definición de clases económicas. El análisis y la evaluación de los modelos se realizan íntegramente en el espacio continuo del retorno, preservando su signo y magnitud.

Esto implica que, aunque los indicadores técnicos se calculen directamente sobre el precio, su evaluación no se orienta a explicar la evolución instantánea del close, sino a medir su capacidad para anticipar retornos simples futuros en un horizonte temporal determinado.

En consecuencia, el vínculo entre indicadores y targets se establece en términos de poder predictivo sobre la magnitud y el signo del retorno intradía futuro, sin discretización en eventos ni definición de umbrales económicos. El análisis posterior se centra, por tanto, en identificar qué indicadores y configuraciones temporales contienen información relevante para explicar y predecir retornos futuros de manera consistente, en el marco de un problema de regresión continua.

### **1.2. Information Coefficient (IC) versus target `ret_h`**

En el dataset `mnq_intraday_labeled`, cada fila representa una decisión potencial en un instante $ t $ del intradía. Para ese instante, el *target* continuo `ret_h` indica el **retorno simple intradía futuro**, calculado sobre el precio de cierre, entre $ t $ y $ t+h $, para horizontes fijos (por ejemplo, $ h = 60 $ o $ 90 $ minutos).

La **ventana de gestión** no redefine el *target* ni introduce una nueva variable temporal, sino que delimita el conjunto de instantes $ t $ en los cuales el sistema está habilitado a evaluar señales y generar decisiones. En este trabajo, dicha ventana se define empíricamente como el período donde se observa mayor persistencia estadística de los factores, y constituye el intervalo operativo en el que el modelo “observa” el mercado y produce estimaciones de retorno futuro.

Por su parte, la **ventana de ejecución** (o expansión) no es una entidad explícita del modelo ni del cálculo del *Information Coefficient* (IC). Esta ventana surge de manera implícita, ya que corresponde al intervalo temporal en el que se materializa el resultado futuro de las decisiones tomadas durante la ventana de gestión. Para cada instante $ t $ perteneciente a la ventana de gestión, el *target* `ret_h` refleja el comportamiento del precio en $ t+h $, que naturalmente se ubica en una franja horaria posterior.

Con el objetivo de evaluar la **robustez temporal** de los indicadores técnicos y evitar conclusiones dependientes de un único tramo horario, el análisis de IC se realiza bajo tres contextos complementarios:

a) **Jornada completa**: se consideran todos los instantes intradía válidos, con el fin de identificar factores estructurales con capacidad predictiva estable a lo largo del día.

b) **Ventana de gestación (08:00-09:00)**: se restringe el análisis al período previo al inicio del movimiento operativo principal, donde suelen formarse las condiciones iniciales del desplazamiento posterior.

c) **Ventana de ejecución o expansión (09:00-10:00)**: se analiza el período inmediatamente posterior, donde los movimientos tienden a desarrollarse con mayor intensidad y direccionalidad.

Este enfoque permite distinguir entre indicadores con valor explicativo global y aquellos cuya capacidad predictiva depende del contexto horario.

Bajo este esquema, el *Information Coefficient* (IC) se calcula correlacionando, para cada instante $ t $ perteneciente al conjunto temporal considerado:

- $ X_t $: el valor del indicador técnico, calculado exclusivamente con información disponible hasta $ t $.
- $ Y_{t,h} $: el *target* continuo `ret_h`, asociado a ese mismo instante $ t $ y a un horizonte fijo $ h $.

De este modo, el IC mide directamente la capacidad del indicador, evaluado en el momento de decisión, para anticipar la **magnitud y el signo del retorno intradía futuro**. La relación se establece fila a fila, sin agregar bloques temporales ni comparar ventanas entre sí, respetando estrictamente la causalidad temporal.

En términos operativos, un IC distinto de cero indica que ciertos estados del mercado, caracterizados por los indicadores técnicos en el instante de evaluación, están sistemáticamente asociados a retornos futuros de mayor o menor magnitud y/o dirección. Esto justifica su uso como variables explicativas en el entrenamiento del modelo predictivo.


### **1.3. Alineación entre el punto 8 (stage_03b) y el cálculo del IC (stage_04)**

La construcción de las ventanas operativas desarrollada en el punto 8 del stage_03b establece una separación conceptual fundamental entre:

- una ventana de gestación (predicción), donde se origina la información anticipatoria, y

- una ventana de expansión (ejecución), donde los movimientos alcanzan magnitud económica explotable.

Esta separación no entra en conflicto con la definición de los targets ni con el cálculo del Information Coefficient (IC); por el contrario, ambos enfoques son complementarios y coherentes, siempre que se entienda correctamente el rol temporal de cada elemento.



#### **1.3.1. Nivel estadístico (dataset y targets)**

En `mnq_intraday_labeled`, el *target* (`ret_h`) está definido **fila a fila**, para cada instante $ t $, como el **retorno simple intradía futuro** observado entre $ t $ y $ t+h $, para un horizonte fijo $ h $.

Esto implica que:

- el *target* está **anclado temporalmente al instante $ t $**,
- el horizonte $ h $ determina **cuándo se materializa el resultado futuro**,
- no existen *targets* definidos “por ventana”, sino **por cada decisión potencial asociada a un minuto específico**.

Cuando el análisis se restringe a la **ventana de gestación** (por ejemplo, 08:20-08:40), lo que se realiza es una **selección del subconjunto de instantes $ t $** que, de acuerdo con el análisis empírico previo, concentran mayor información anticipatoria relevante. Esta restricción no modifica la definición del *target*, sino únicamente el conjunto temporal sobre el cual se evalúa la relación entre indicadores y retornos futuros.

En este contexto, el *Information Coefficient* (IC) mide:

- la **relación estadística** entre el estado del mercado en el instante $ t $, capturado por los indicadores técnicos calculados con información disponible hasta ese momento, y
- el **retorno simple futuro `ret_h`** asociado a ese mismo instante $ t $, cuyo efecto se materializa naturalmente en una franja horaria posterior, correspondiente a la ventana de ejecución o expansión.

De este modo, el IC evalúa de forma directa la capacidad de los indicadores técnicos, observados en el momento de decisión, para anticipar la magnitud y el signo del movimiento futuro del precio, respetando estrictamente la causalidad temporal.


#### **1.3.2. Nivel operativo (modelo y ejecución)**

Desde el punto de vista operativo, el esquema temporal se interpreta de la siguiente manera:

- **Ventana de gestación (08:20-08:40)**  
  - Se calculan los indicadores técnicos utilizando únicamente información disponible hasta cada instante \( t \).  
  - El modelo evalúa si, desde esos instantes, es esperable un **retorno simple intradía** significativo a horizontes de 60 o 90 minutos.  
  - En esta franja reside la **capacidad predictiva**, ya que se identifican contextos de mercado con potencial direccional futuro.

- **Ventana de expansión (09:10-09:40)**  
  - Corresponde al período donde, de forma empírica, los movimientos intradía tienden a manifestarse con mayor frecuencia e intensidad.  
  - Las predicciones generadas durante la ventana de gestación habilitan (o no) la toma de decisiones operativas en esta franja.  
  - El punto de entrada puede ubicarse en cualquier minuto dentro de esta ventana, sujeto a reglas operativas adicionales y a la lógica del sistema de ejecución.

- **Horizonte de resultado**  
  - El resultado de la operación se evalúa en función del **retorno simple futuro** observado a un horizonte fijo (60 o 90 minutos) contado desde el instante de entrada.  
  - El cierre de la operación se rige por reglas operativas externas al modelo predictivo (por ejemplo, gestión de riesgo, toma de ganancias o stop), manteniendo siempre la coherencia temporal con el horizonte definido.

Bajo este esquema, el modelo no predice eventos discretos ni umbrales de puntos, sino la **magnitud y el signo del retorno futuro**, mientras que las ventanas temporales organizan el proceso de decisión y ejecución sin alterar la definición del *target*.

#### **1.3.3. Rol del IC dentro de este esquema**

El *Information Coefficient* (IC) no evalúa la ejecución operativa, sino la **calidad de la información predictiva** contenida en los indicadores técnicos bajo distintos contextos temporales.

Su función es responder a la pregunta:

¿Los indicadores técnicos calculados en el instante \( t \) contienen información útil para anticipar el retorno simple futuro `ret_h` en \( t+h \)?

Por lo tanto:

- El IC Se calcula en **tres contextos complementarios**:
  - **Jornada completa** (todos los instantes intradía válidos),
  - **Ventana de gestación** (por ejemplo, 08:00–09:00),
  - **Ventana de ejecución/expansión** (por ejemplo, 09:00–10:00).
- En todos los casos, los *targets* (`ret_h`) incorporan de forma natural el desfase temporal \( h \), ya que están definidos como retorno futuro entre \( t \) y \( t+h \).
- La coherencia temporal y la ausencia de *data leakage* quedan garantizadas por construcción: $ X_t $ se calcula con información disponible hasta \( t \), y $ Y_{t,h} $ se observa en el futuro, en \( t+h \).


### **1.4. Conclusión sintética**

La lógica operativa define **cuándo se observa información** (gestación), **cuándo se ejecuta** (expansión) y **cómo se mide el resultado** (horizonte \( h \) desde el instante de entrada).

El cálculo del IC, en el `stage_04`, cuantifica qué tan informativos son los indicadores técnicos respecto del retorno futuro `ret_h` bajo tres escenarios: **global (todo el día)** y **dependientes del contexto horario** (gestación y expansión).

Ambos enfoques describen el mismo fenómeno desde niveles distintos (operativo vs estadístico) y se mantienen alineados dentro del diseño del pipeline.

# **2. Separación de dataset IS vs OOS**

En esta etapa se realiza la separación del dataset en dos subconjuntos temporales: **in-sample (IS)** y **out-of-sample (OOS)**.  
Esta división se efectúa de manera estrictamente cronológica, respetando el orden temporal de los datos y evitando cualquier tipo de filtración de información futura.

El conjunto **in-sample (IS)** se utiliza para:
- el cálculo y selección de indicadores técnicos,
- el análisis del Information Coefficient (IC),
- la evaluación de correlaciones y redundancias entre features.

El conjunto **out-of-sample (OOS)** se reserva exclusivamente para:
- validar la estabilidad temporal de las relaciones observadas,
- comprobar que la capacidad predictiva de los indicadores no es producto del sobreajuste,
- verificar que las señales seleccionadas mantienen poder explicativo en datos no vistos.

Esta separación es un paso crítico para garantizar la validez estadística del proceso de feature engineering.  
Un indicador solo se considera apto para su uso en el modelo predictivo si demuestra un comportamiento consistente entre IS y OOS, tanto en magnitud como en signo del IC.

De este modo, la selección final de features se basa en criterios de **robustez temporal**, y no únicamente en el desempeño observado dentro del período de entrenamiento.


**Criterio propuesto (ajustable)**

- IS: desde 2019-12-23 hasta 2022-12-31
- OOS: desde 2023-01-01 hasta 2025-06-13

Este split es solo para selección y validación de features, no es el split final de modelado.

In [158]:
# Trabajamos sobre el dataset existente
df = mnq_intraday_labeled.copy()

# Aseguramos tipo datetime
df["date"] = pd.to_datetime(df["date"])

# Definimos fecha de corte IS / OOS
cut_date = pd.to_datetime("2022-12-31")

# Creamos columna de split para Feature Engineering
df["split_fe"] = np.where(
    df["date"] <= cut_date,
    "IS",   # In-Sample (selección de features)
    "OOS",  # Out-Of-Sample (validación de features)
)

# Sobrescribimos el dataset con la nueva columna
mnq_intraday_labeled = df



In [159]:
# Asumimos que el índice es DatetimeIndex tz-aware
idx = mnq_intraday_labeled.index

mnq_intraday_labeled["minute_of_day"] = idx.hour * 60 + idx.minute

# **3. Indicadores Técnicos**

Los indicadores técnicos calculados en cada jornada tienen como objetivo capturar dinámicas intradía relevantes del precio y el volumen, tales como momentum, sobrecompra/sobreventa, presión institucional o posibles reversiones. Cada uno aporta información complementaria sobre el comportamiento del mercado a corto plazo. En particular:

## **3.1. Indicadores técnicos individuales**

#### 1. **RSI (Relative Strength Index)**

Mide la fuerza relativa del precio en los últimos períodos (3, 5, 7, 14), oscilando entre 0 y 100.

  - Valores altos indican posibles condiciones de sobrecompra, mientras que valores bajos sugieren sobreventa.
  
  - Calculado sobre los precios de cierre intradía, el RSI es útil para identificar puntos de reversión potenciales en el corto plazo.

In [58]:
def calcular_rsi(df=mnq_intraday_labeled, target='close' ):
  rsi_columns = ['rsi_14', 'rsi_7', 'rsi_5', 'rsi_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['rsi_14'] = ta.momentum.RSIIndicator(grupo[target], window=14).rsi()
        grupo['rsi_7'] = ta.momentum.RSIIndicator(grupo[target], window=7).rsi()
        grupo['rsi_5'] = ta.momentum.RSIIndicator(grupo[target], window=5).rsi()
        grupo['rsi_3'] = ta.momentum.RSIIndicator(grupo[target], window=3).rsi()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, rsi_columns

#### 2. **Momentum**

Mide la aceleración reciente del precio mediante la variación porcentual entre el precio actual y el de hace N minutos.

  - Un valor positivo indica una subida reciente, lo que podría sugerir una continuación alcista.

  - Un valor negativo señala presión bajista reciente, potencialmente anticipando una continuación a la baja.

In [59]:
def calcular_momentum(df=mnq_intraday_labeled, target='close' ):
  momentum_columns = ['momentum_10', 'momentum_5','momentum_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['momentum_10'] = grupo[target].pct_change(10)
        grupo['momentum_5'] = grupo[target].pct_change(5)
        grupo['momentum_3'] = grupo[target].pct_change(3)
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, momentum_columns

#### 3. **Relación de volumen actual vs. su promedio reciente**

Compara el volumen actual con su media móvil en distintas ventanas de tiempo: 15, 20 y 30 minutos.

  - Un valor mayor a 1 indica un volumen superior al promedio de la ventana correspondiente, lo que puede reflejar interés creciente o actividad institucional.

  - Un valor menor a 1 sugiere baja actividad o consolidación del precio.

Esta métrica permite detectar aumentos de volumen ("spikes") sin depender del volumen en crudo, y las diferentes ventanas permiten capturar variaciones en la dinámica de corto plazo con distinta sensibilidad.


In [60]:
def calcular_volumen_ratio(df=mnq_intraday_labeled, target='close'):
  volume_ratio_columns = ['volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['volume_ratio_15'] = grupo['volume'] / grupo['volume'].rolling(15).mean()
        grupo['volume_ratio_20'] = grupo['volume'] / grupo['volume'].rolling(20).mean()
        grupo['volume_ratio_30'] = grupo['volume'] / grupo['volume'].rolling(30).mean()
        grupo['volume_ratio_60'] = grupo['volume'] / grupo['volume'].rolling(60).mean()
        grupo['volume_ratio_90'] = grupo['volume'] / grupo['volume'].rolling(90).mean()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

  return df, volume_ratio_columns

#### 4. **MACD diferencial (señal de cruce)**

Representa la diferencia entre la línea MACD y su línea de señal (una media exponencial de sí misma).

  - Un valor positivo y creciente indica momentum alcista.

  - Un valor negativo sugiere presión bajista.
  
Es ampliamente utilizado para detectar giros de tendencia y cambios en la dinámica del mercado.


In [61]:
def calcular_macd(df=mnq_intraday_labeled, target='close'):
    macd_columns = ['macd']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['macd'] = ta.trend.MACD(grupo[target]).macd_diff()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, macd_columns

#### 5. **Distancia del precio actual a su EMA (15, 20 y 30 minutos)**

Mide el desvío porcentual del precio respecto a su media exponencial en diferentes ventanas, y actúa como indicador de sobreextensión o retorno a la media.

  - Si el precio está muy por encima de la EMA, puede anticipar una reversión bajista o una posible aceleración alcista.

  - Si está por debajo, podría indicar agotamiento o presión vendedora.<br>

Esta métrica se expresa como un porcentaje relativo, lo que facilita la comparación entre distintas ventanas temporales y condiciones de mercado.

Usar varias ventanas (15, 20 y 30 minutos) permite capturar diferentes horizontes de reacción del precio frente a su media móvil.


In [62]:
def calcular_ema(df=mnq_intraday_labeled, target='close'):
    ema_columns = ['ema_15', 'ema_20', 'ema_30',  'ema_60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['ema_15'] = grupo[target] / grupo[target].ewm(span=15).mean() - 1
        grupo['ema_20'] = grupo[target] / grupo[target].ewm(span=20).mean() - 1
        grupo['ema_30'] = grupo[target] / grupo[target].ewm(span=30).mean() - 1
        grupo['ema_60'] = grupo[target] / grupo[target].ewm(span=60).mean() - 1
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, ema_columns

#### 6. **%K Estocástico**

Mide la posición relativa del precio actual dentro del rango alto-bajo de los últimos n periodos (generalmente 14).

  - Se utiliza para identificar condiciones extremas de sobrecompra o sobreventa.

  - Un valor cercano a 100 indica que el precio está cerca del máximo reciente (potencial sobrecompra), mientras que un valor cercano a 0 indica proximidad al mínimo reciente (posible sobreventa).

Es útil para detectar momentos en los que el precio puede estar excesivamente extendido y susceptible a una reversión.


In [63]:
def calcular_stochastic(df=mnq_intraday_labeled, target='close'):
    stoch_columns = ['stoch_k_14', 'stoch_k_20', 'stoch_k_30']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        stoch_14 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=14, smooth_window=3
        )
        grupo['stoch_k_14'] = stoch_14.stoch()

        stoch_20 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=20, smooth_window=3
        )
        grupo['stoch_k_20'] = stoch_20.stoch()

        stoch_30 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=30, smooth_window=3
        )
        grupo['stoch_k_30'] = stoch_30.stoch()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, stoch_columns


#### 7.**%B de Bollinger (Bollinger Band Percent)**

Indica la posición del precio actual en relación con las bandas de Bollinger, que están construidas alrededor de una media móvil usando desviaciones estándar.

  - Un valor de %B > 1 sugiere que el precio está por encima de la banda superior, lo que podría implicar exceso de optimismo o momentum fuerte.

  - Un valor < 0 indica que está por debajo de la banda inferior, posible señal de pánico o sobreventa extrema.

Este indicador es eficaz para identificar zonas de congestión, breakout o reversiones basadas en la volatilidad reciente.


In [64]:
from ta.volatility import BollingerBands

def calcular_bollinger(df=mnq_intraday_labeled, target='close'):
    '''bollinger_columns = [
        'bb_percent_15_15', 'bb_percent_20_15', 'bb_percent_30_15',
        'bb_percent_15_20', 'bb_percent_20_20', 'bb_percent_30_20',
        'bb_percent_15_25', 'bb_percent_20_25', 'bb_percent_30_25',
    ]'''

    bollinger_columns = [
        'bb_15_15', 'bb_20_15', 'bb_30_15', 'bb_60_15',
        'bb_15_20', 'bb_20_20', 'bb_30_20', 'bb_60_20',
        'bb_15_25', 'bb_20_25', 'bb_30_25', 'bb_60_25',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 1.5
        grupo['bb_15_15'] = BollingerBands(grupo[target], window=15, window_dev=1.5).bollinger_pband()
        grupo['bb_20_15'] = BollingerBands(grupo[target], window=20, window_dev=1.5).bollinger_pband()
        grupo['bb_30_15'] = BollingerBands(grupo[target], window=30, window_dev=1.5).bollinger_pband()
        grupo['bb_60_15'] = BollingerBands(grupo[target], window=60, window_dev=1.5).bollinger_pband()

        # std: 2
        grupo['bb_15_20'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30_20'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60_20'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()

        # std: 2.5
        grupo['bb_15_25'] = BollingerBands(grupo[target], window=15, window_dev=2.5).bollinger_pband()
        grupo['bb_20_25'] = BollingerBands(grupo[target], window=20, window_dev=2.5).bollinger_pband()
        grupo['bb_30_25'] = BollingerBands(grupo[target], window=30, window_dev=2.5).bollinger_pband()
        grupo['bb_60_25'] = BollingerBands(grupo[target], window=60, window_dev=2.5).bollinger_pband()



        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns


In [65]:
def calcular_bollinger_resume(df=mnq_intraday_labeled, target='close'):

    bollinger_columns = [
        'bb_15', 'bb_20', 'bb_30', 'bb_60',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 2
        grupo['bb_15'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns

#### 8. **ATR normalizado (Average True Range / precio)**

Representa la volatilidad absoluta reciente ajustada al nivel del precio.

  - El ATR mide el rango promedio de oscilación de un activo en los últimos n periodos, capturando tanto movimientos bruscos como gaps.

  - Al normalizarlo dividiéndolo por el precio, se obtiene una medida relativa, comparable entre distintos niveles de mercado.

Este indicador es útil para detectar momentos de alta o baja volatilidad intradía, que pueden influir en la confiabilidad de otras señales técnicas.


In [66]:
def calcular_atr(df=mnq_intraday_labeled, target='close', windows=[5, 10, 14, 20, 30]):
    atr_columns = []

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        for w in windows:
            col_name = f'atr_norm_{w}'
            atr = AverageTrueRange(high=grupo['high'], low=grupo['low'], close=grupo[target], window=w)
            grupo[col_name] = atr.average_true_range() / grupo[target]
            atr_columns.append(col_name)
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, list(set(atr_columns))

In [67]:
def calcular_atr_14(df=mnq_intraday_labeled, target='close'):
    atr_columns = ['atr_norm']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        atr = AverageTrueRange(high=grupo['high'], low=grupo['low'], close=grupo[target], window=14)
        grupo['atr'] = atr.average_true_range()
        grupo['atr_norm'] = grupo['atr'] / grupo[target]
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, atr_columns

####  9. **ROC (Rate of Change)**

Calcula la tasa de cambio porcentual del precio con respecto a su valor n minutos atrás.

- Es un indicador de momentum que capta aceleraciones o desaceleraciones recientes del precio.

- Valores positivos indican presión alcista; negativos, presión bajista.

A diferencia del momentum tradicional, el ROC expresa el cambio de forma normalizada y en porcentaje, lo que facilita su interpretación comparativa entre distintos activos o marcos temporales.


In [68]:
def calcular_roc(df=mnq_intraday_labeled, target='close'):
    roc_columns = ['roc_5', 'roc_10', 'roc_20','roc_30','roc_60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['roc_5'] = ROCIndicator(close=grupo[target], window=5).roc()
        grupo['roc_10'] = ROCIndicator(close=grupo[target], window=10).roc()
        grupo['roc_20'] = ROCIndicator(close=grupo[target], window=20).roc()
        grupo['roc_30'] = ROCIndicator(close=grupo[target], window=30).roc()
        grupo['roc_60'] = ROCIndicator(close=grupo[target], window=60).roc()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, roc_columns

## **3.2. Cálculo de indicadores técnicos**

Calculamos los indicadores técnicos

In [76]:
import os
import glob
import pandas as pd

# ============================================================
# Cargar parquet existente o calcular indicadores y guardar
# ============================================================

PROCESSED_DIR = "/content/drive/MyDrive/neural_profit/data/processed"
PARQUET_NAME = "mnq_intraday_with_indicators.parquet"
PARQUET_PATH = os.path.join(PROCESSED_DIR, PARQUET_NAME)

os.makedirs(PROCESSED_DIR, exist_ok=True)

# 1) Si existe el parquet (o alguno compatible), cargarlo
if os.path.exists(PARQUET_PATH):
    mnq_intraday_with_indicators = pd.read_parquet(PARQUET_PATH)
    print(f"[OK] Cargado: {PARQUET_PATH}")
    indicator_columns = mnq_intraday_with_indicators.columns.tolist()


    # Asegurar que el índice esté en formato datetime
    mnq_intraday_with_indicators.index = pd.to_datetime(mnq_intraday_with_indicators.index)


    columns_to_remove = [
    'date', 'minute_of_day', 'open', 'high', 'low', 'close', 'volume',
    'ret_60', 'ret_90', 'split_fe'
      ]

    indicator_columns = [
        col for col in indicator_columns if col not in columns_to_remove
    ]

else:
    # Fallback: si no existe el nombre exacto, intenta encontrar algún parquet similar
    candidates = sorted(glob.glob(os.path.join(PROCESSED_DIR, "*with_indicators*.parquet")))
    if candidates:
        mnq_intraday_with_indicators = pd.read_parquet(candidates[-1])
        print(f"[OK] Cargado (fallback): {candidates[-1]}")
    else:
        # 2) Calcular indicadores
        mnq_intraday_with_indicators = mnq_intraday_labeled.copy()
        print(f"[OK] Calculando indicadores técnicos")

        mnq_intraday_with_indicators, rsi_columns = calcular_rsi(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, momentum_columns = calcular_momentum(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, volume_ratio_columns = calcular_volumen_ratio(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, macd_columns = calcular_macd(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, ema_columns = calcular_ema(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, stoch_columns = calcular_stochastic(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, bollinger_columns = calcular_bollinger(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, atr_columns = calcular_atr(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, roc_columns = calcular_roc(mnq_intraday_with_indicators)

        indicator_columns = (
          rsi_columns
          + momentum_columns
          + volume_ratio_columns
          + macd_columns
          + ema_columns
          + stoch_columns
          + bollinger_columns
          + atr_columns
          + roc_columns
          )


        # 3) Guardar parquet final
        mnq_intraday_with_indicators.to_parquet(PARQUET_PATH, index=True)
        print(f"[OK] Calculado y guardado: {PARQUET_PATH}")


[OK] Calculando indicadores técnicos
[OK] Calculado y guardado: /content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_with_indicators.parquet


Filtramos todos los NaNs del dataset

In [77]:
# Eliminación de filas con NaN
mnq_intraday_with_indicators = mnq_intraday_with_indicators.dropna()

# Verificación posterior
start_time_full_day, final_time_full_day = info_dataset(mnq_intraday_with_indicators)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 392
	Hora diaria de inicio 07:59
	Hora diaria de final 14:30
	Zona horaria: America/New_York


In [78]:
assert not mnq_intraday_with_indicators.isna().any().any(), \
    "El dataset contiene NaN"

##**4. Calculo de Information Coefficient (IC)**

## **4.1. Target `ret_h`**

En el dataset `mnq_intraday_labeled` se emplea actualmente **un único tipo de variable objetivo**, correspondiente a **targets continuos de retorno**, definidos para distintos horizontes temporales. En particular, se consideran los targets `ret_h` para **h = 60** y **h = 90 minutos**.

Dado que ambos horizontes comparten la misma definición y naturaleza estadística, el cálculo y la interpretación del **Information Coefficient (IC)** se realizan bajo un **marco metodológico único, coherente y homogéneo**.

La variable `ret_h` cuantifica el **retorno futuro del precio**, expresado en términos porcentuales, entre el instante \( t \) y el instante \( t + h \). Se trata de un target **continuo**, que encapsula simultáneamente la **dirección** y la **magnitud relativa** del movimiento futuro del mercado.

Este tipo de variable objetivo constituye el caso **más directo y conceptualmente apropiado** para el análisis mediante IC, ya que permite evaluar si un indicador técnico:

- anticipa correctamente la **dirección** de los movimientos futuros, y  
- discrimina la **intensidad relativa** de dichos movimientos.

Para este análisis se adopta como métrica el **coeficiente de correlación de Spearman**, debido a que:

- no presupone relaciones lineales,  
- presenta robustez frente a valores atípicos, y  
- captura relaciones **monotónicas**, más acordes con el comportamiento de series financieras intradía.

La interpretación del IC es inmediata:

- un **IC positivo** indica que valores más elevados del indicador se asocian, en promedio, con retornos futuros mayores,  
- un **IC negativo** indica una relación inversa entre el indicador y el retorno futuro.

En virtud de lo anterior, `ret_h` (para **h = 60** y **h = 90**) se adopta como **target principal y exclusivo** para el análisis de Information Coefficient en esta etapa del pipeline.


## **4.2. Criterio metodológico — Evaluación de IC por jornada completa y ventanas horarias**

Con el objetivo de alinear la selección de indicadores técnicos con el enfoque del libro, se adopta el siguiente criterio:

Se calculan **múltiples tablas de Information Coefficient (IC)** sobre distintos recortes temporales del día, manteniendo siempre la separación **IS / OOS**:

- **Jornada completa** (mercado intradía completo)
- **Ventana de gestación 08:00–09:00**
- **Ventana  de ejecución 09:00–10:00**

**Justificación**

Este enfoque es metodológicamente sólido porque:

- Permite identificar **factores estructurales**, es decir, indicadores que mantienen IC OOS positivo a lo largo de toda la jornada.
- Permite detectar **factores dependientes del horario**, cuya capacidad predictiva se concentra en ventanas específicas.
- Evita sesgos, siempre que la **selección inicial se base en la jornada completa** y el análisis por ventanas se utilice como refinamiento posterior.

De este modo, el análisis no optimiza prematuramente por horario, sino que primero prioriza **robustez global**.

---

**Uso de los resultados**

A partir de las distintas `ic_tables`, se pueden realizar los siguientes pasos:

1. Identificar la **intersección** de indicadores con IC OOS positivo en todas las ventanas  
   → *core de factores robustos*.

2. Analizar las **diferencias entre ventanas**  
   → detección de features condicionadas por tramo horario.

3. Decidir estrategias posteriores:
   - activación o ponderación de features según el horario, o
   - uso de un único modelo con contexto temporal explícito, o
   - modelos específicos por ventana (etapa posterior).

---

**Síntesis**

> Evaluar IC en la jornada completa y en ventanas horarias permite pasar de una  
> **selección global de factores** a un **refinamiento operativo**,  
> sin romper la lógica metodológica del libro ni introducir sobreajuste.


## **4.3. Implementación de cálculo de IC**

### **4.3.1. Código**

In [79]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# =========================
# Utilidades base
# =========================

def spearman_ic(x: pd.Series, y: pd.Series) -> float:
    """IC Spearman entre x e y, ignorando NaNs."""
    mask = x.notna() & y.notna()
    if mask.sum() < 3:
        return np.nan
    return spearmanr(x[mask], y[mask]).correlation


def filter_time_window(df: pd.DataFrame, start: str = "07:59", end: str = "14:30") -> pd.DataFrame:
    """Filtra por ventana horaria intradía. Requiere DatetimeIndex."""
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El índice del DataFrame debe ser DatetimeIndex para usar between_time().")
    return df.between_time(start, end)


def daily_ic(df: pd.DataFrame, indicator_col: str, target_col: str, date_col: str = "date") -> pd.Series:
    """IC Spearman por día (promediable). Requiere columna date_col."""
    if date_col not in df.columns:
        raise ValueError(f"Falta la columna '{date_col}' en df.")
    return df.groupby(date_col).apply(lambda g: spearman_ic(g[indicator_col], g[target_col]))


# =========================
# Tabla IC (indicadores × horizontes) con IS/OOS
# =========================

def compute_ic_table_is_oos_delta(
    df: pd.DataFrame,
    indicator_columns: list[str],
    horizons: tuple[int, ...] = (60, 90),
    window_start: str = "07:59",
    window_end: str = "14:30",
    use_daily_ic: bool = True,
    split_col: str = "split_fe",     # "IS" / "OOS"
    date_col: str = "date",
) -> pd.DataFrame:
    """
    Calcula IC(indicador, ret_h) para cada indicador y horizonte (60/90),
    separando IS y OOS según split_col.

    Targets esperados:
      - ret_60, ret_90 (formato: f"ret_{h}")

    Devuelve una tabla con:
      - indicator, horizon
      - IC_IS, IC_OOS, ret_oos_minus_is
      - n_pairs_IS, n_pairs_OOS
      - abs_IC_OOS (ranking recomendado) y notas
    """

    # 1) Filtrado horario intradía
    dfw = filter_time_window(df, window_start, window_end).copy()

    # 2) Validaciones de columnas estructurales
    required = {split_col, date_col}
    missing_req = [c for c in required if c not in dfw.columns]
    if missing_req:
        raise ValueError(f"Faltan columnas requeridas: {missing_req}")

    # 3) Subsets IS / OOS
    df_is = dfw[dfw[split_col] == "IS"]
    df_oos = dfw[dfw[split_col] == "OOS"]

    rows = []

    for ind in indicator_columns:
        for h in horizons:
            target_col = f"ret_{h}"

            # Validación de columnas
            missing = [c for c in [ind, target_col] if c not in dfw.columns]
            if missing:
                rows.append({
                    "indicator": ind,
                    "horizon": h,
                    "IC_IS": np.nan,
                    "IC_OOS": np.nan,
                    "ret_oos_minus_is": np.nan,
                    "n_pairs_IS": 0,
                    "n_pairs_OOS": 0,
                    "note": f"missing: {missing}",
                })
                continue

            # --- IS ---
            if use_daily_ic:
                ic_is_series = daily_ic(df_is, ind, target_col, date_col=date_col)
                ic_is = ic_is_series.mean()
            else:
                ic_is = spearman_ic(df_is[ind], df_is[target_col])
            n_pairs_is = int((df_is[ind].notna() & df_is[target_col].notna()).sum())

            # --- OOS ---
            if use_daily_ic:
                ic_oos_series = daily_ic(df_oos, ind, target_col, date_col=date_col)
                ic_oos = ic_oos_series.mean()
            else:
                ic_oos = spearman_ic(df_oos[ind], df_oos[target_col])
            n_pairs_oos = int((df_oos[ind].notna() & df_oos[target_col].notna()).sum())

            rows.append({
                "indicator": ind,
                "horizon": h,
                "IC_IS": ic_is,
                "IC_OOS": ic_oos,
                "ret_oos_minus_is": (ic_oos - ic_is) if (pd.notna(ic_is) and pd.notna(ic_oos)) else np.nan,
                "n_pairs_IS": n_pairs_is,
                "n_pairs_OOS": n_pairs_oos,
                "note": "",
            })

    out = pd.DataFrame(rows)

    # Ranking recomendado: por |IC_OOS| (lo que generaliza)
    out["abs_IC_OOS"] = out["IC_OOS"].abs()
    out = (
        out.sort_values(["horizon", "abs_IC_OOS"], ascending=[True, False])
           .reset_index(drop=True)
    )

    return out


### **4.3.2. Aplicación**

In [80]:
# ============================================================
# Ventanas temporales para IC (full day + ventanas por hora)
# ============================================================

# Jornada completa (reutiliza los límites generales del dataset intradía)
start_time_full_day
final_time_full_day

# Ventana de gestación (08:00–09:00)
start_time_gestation_window = "08:00"
final_time_gestation_window = "09:00"

# Ventana de ejecución/expansión (09:00–10:00)
start_time_execution_window = "09:00"
final_time_execution_window = "10:00"


In [81]:
import os
import json
import pandas as pd

# ============================================================
# Cache de IC tables (Google Drive)
# ============================================================

CACHE_DIR = "/content/drive/MyDrive/neural_profit/data/processed/ic_tables"
os.makedirs(CACHE_DIR, exist_ok=True)

def _ic_cache_paths(name: str) -> dict:
    """
    Devuelve paths de cache para una ic_table:
      - parquet: datos
      - json: metadatos (parámetros relevantes)
    """
    return {
        "data": os.path.join(CACHE_DIR, f"{name}.parquet"),
        "meta": os.path.join(CACHE_DIR, f"{name}.meta.json"),
    }

def load_or_compute_ic_table(
    *,
    name: str,
    df: pd.DataFrame,
    indicator_columns: list[str],
    horizons: tuple[int, ...],
    window_start: str,
    window_end: str,
    use_daily_ic: bool,
    split_col: str,
    date_col: str,
    force_recompute: bool = False,
) -> pd.DataFrame:
    """
    Carga ic_table desde cache si existe; si no existe, la calcula y la guarda.
    """
    paths = _ic_cache_paths(name)

    # 1) Si existe y no forzamos recálculo → cargar
    if (not force_recompute) and os.path.exists(paths["data"]):
        ic_table = pd.read_parquet(paths["data"])
        print(f"[cache] Loaded: {paths['data']}")
        return ic_table

    # 2) Calcular ic_table
    ic_table = compute_ic_table_is_oos_delta(
        df=df,
        indicator_columns=indicator_columns,
        horizons=horizons,
        window_start=window_start,
        window_end=window_end,
        use_daily_ic=use_daily_ic,
        split_col=split_col,
        date_col=date_col,
    )

    # 3) Guardar datos
    ic_table.to_parquet(paths["data"], index=False)

    # 4) Guardar metadatos mínimos
    meta = {
        "name": name,
        "horizons": list(horizons),
        "window_start": window_start,
        "window_end": window_end,
        "use_daily_ic": use_daily_ic,
        "split_col": split_col,
        "date_col": date_col,
        "n_indicators": len(indicator_columns),
    }
    with open(paths["meta"], "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print(f"[cache] Computed & saved: {paths['data']}")
    return ic_table

In [82]:
# ============================================================
# IC tables IS vs OOS por ventana (con cache en Drive)
# ============================================================

ic_table_full_day = load_or_compute_ic_table(
    name="ic_table_full_day",
    df=mnq_intraday_with_indicators,
    indicator_columns=indicator_columns,
    horizons=(60, 90),
    window_start=start_time_full_day,
    window_end=final_time_full_day,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

ic_table_gestation = load_or_compute_ic_table(
    name="ic_table_gestation",
    df=mnq_intraday_with_indicators,
    indicator_columns=indicator_columns,
    horizons=(60, 90),
    window_start=start_time_gestation_window,
    window_end=final_time_gestation_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

ic_table_execution = load_or_compute_ic_table(
    name="ic_table_execution",
    df=mnq_intraday_with_indicators,
    indicator_columns=indicator_columns,
    horizons=(60, 90),
    window_start=start_time_execution_window,
    window_end=final_time_execution_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)


[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/processed/ic_tables/ic_table_full_day.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/processed/ic_tables/ic_table_gestation.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/processed/ic_tables/ic_table_execution.parquet


### **4.3.3. Resultado**

In [83]:
ic_table_full_day.head(10)

,indicator,horizon,IC_IS,IC_OOS,ret_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,-0.193578,-0.189173,0.004405,281064,229712,,0.189173
1,roc_60,60,-0.187156,-0.177858,0.009298,281064,229712,,0.177858
2,bb_60_15,60,-0.152318,-0.150149,0.002168,281064,229712,,0.150149
3,bb_60_20,60,-0.152318,-0.150149,0.002168,281064,229712,,0.150149
4,bb_60_25,60,-0.152318,-0.150149,0.002168,281064,229712,,0.150149
5,ema_30,60,-0.141073,-0.137669,0.003404,281064,229712,,0.137669
6,roc_30,60,-0.135632,-0.135081,0.000551,281064,229712,,0.135081
7,rsi_14,60,-0.135146,-0.134811,0.000335,281064,229712,,0.134811
8,stoch_k_30,60,-0.122581,-0.127539,-0.004957,281064,229712,,0.127539
9,bb_30_15,60,-0.113618,-0.117318,-0.003699,281064,229712,,0.117318


In [84]:
ic_table_gestation.head(10)

,indicator,horizon,IC_IS,IC_OOS,ret_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,-0.354846,-0.448972,-0.094126,43737,35746,,0.448972
1,ema_30,60,-0.320869,-0.416121,-0.095253,43737,35746,,0.416121
2,roc_30,60,-0.308842,-0.395218,-0.086377,43737,35746,,0.395218
3,bb_60_15,60,-0.307776,-0.389431,-0.081656,43737,35746,,0.389431
4,bb_60_20,60,-0.307776,-0.389431,-0.081656,43737,35746,,0.389431
5,bb_60_25,60,-0.307776,-0.389431,-0.081656,43737,35746,,0.389431
6,rsi_14,60,-0.299629,-0.387406,-0.087777,43737,35746,,0.387406
7,roc_60,60,-0.319006,-0.383793,-0.064787,43737,35746,,0.383793
8,ema_20,60,-0.292367,-0.382015,-0.089648,43737,35746,,0.382015
9,roc_20,60,-0.277677,-0.373996,-0.096319,43737,35746,,0.373996


## **4.4. Separación por horizonte**

In [85]:
# ============================================================
# Separación por horizonte – jornada completa
# ============================================================

ic_60_full_day = ic_table_full_day[ic_table_full_day["horizon"] == 60].copy()
ic_90_full_day = ic_table_full_day[ic_table_full_day["horizon"] == 90].copy()

# ============================================================
# Separación por horizonte – ventana de gestación
# ============================================================

ic_60_gestation = ic_table_gestation[ic_table_gestation["horizon"] == 60].copy()
ic_90_gestation = ic_table_gestation[ic_table_gestation["horizon"] == 90].copy()

# ============================================================
# Separación por horizonte – ventana de ejecución / expansión
# ============================================================

ic_60_execution = ic_table_execution[ic_table_execution["horizon"] == 60].copy()
ic_90_execution = ic_table_execution[ic_table_execution["horizon"] == 90].copy()


## **4.5. Primer filtro: fuerza miníma de señal**

Para el análisis intradía se adopta el siguiente criterio empírico de interpretación del Information Coefficient (IC):

- |IC| < 0.02 → ruido
- 0.02 ≤ |IC| < 0.05 → débil
- 0.05 ≤ |IC| < 0.10 → moderado
- |IC| ≥ 0.10 → fuerte

Dado que el objetivo de este stage es identificar señales con capacidad predictiva real, se descartan aquellas cuyo |IC| se encuentra por debajo del umbral de relevancia. En consecuencia, se conservan únicamente los indicadores que presentan una señal fuerte, definida como:

$$|IC_Δ|≥0.10$$

Este primer filtro elimina indicadores dominados por ruido y reduce el espacio de features a un conjunto con relación estadísticamente significativa respecto a la magnitud del movimiento futuro.

In [86]:
# ============================================================
# Umbral mínimo de relevancia estadística (|IC_OOS|)
# ============================================================

IC_TH = 0.10  # umbral de señal fuerte


# ============================================================
# Jornada completa (full day)
# ============================================================

ic_60_relevant_full_day = (
    ic_60_full_day[ic_60_full_day["abs_IC_OOS"] >= IC_TH].copy()
)

ic_90_relevant_full_day = (
    ic_90_full_day[ic_90_full_day["abs_IC_OOS"] >= IC_TH].copy()
)


# ============================================================
# Ventana de gestación
# ============================================================

ic_60_relevant_gestation = (
    ic_60_gestation[ic_60_gestation["abs_IC_OOS"] >= IC_TH].copy()
)

ic_90_relevant_gestation = (
    ic_90_gestation[ic_90_gestation["abs_IC_OOS"] >= IC_TH].copy()
)


# ============================================================
# Ventana de ejecución / expansión
# ============================================================

ic_60_relevant_execution = (
    ic_60_execution[ic_60_execution["abs_IC_OOS"] >= IC_TH].copy()
)

ic_90_relevant_execution = (
    ic_90_execution[ic_90_execution["abs_IC_OOS"] >= IC_TH].copy()
)


In [87]:
# ============================================================
# Helper: extraer lista ordenada y única de indicadores
# ============================================================

def extract_indicator_list(df: pd.DataFrame) -> list[str]:
    """
    Extrae una lista única, ordenada alfabéticamente, de la columna 'indicator'.
    """
    return (
        df["indicator"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )


# ============================================================
# Jornada completa (full day)
# ============================================================

ti_ic_60_relevant_list_full_day = extract_indicator_list(
    ic_60_relevant_full_day
)

ti_ic_90_relevant_list_full_day = extract_indicator_list(
    ic_90_relevant_full_day
)

# ============================================================
# Ventana de gestación
# ============================================================

ti_ic_60_relevant_list_gestation = extract_indicator_list(
    ic_60_relevant_gestation
)

ti_ic_90_relevant_list_gestation = extract_indicator_list(
    ic_90_relevant_gestation
)


# ============================================================
# Ventana de ejecución / expansión
# ============================================================

ti_ic_60_relevant_list_execution = extract_indicator_list(
    ic_60_relevant_execution
)

ti_ic_90_relevant_list_execution = extract_indicator_list(
    ic_90_relevant_execution
)


### **4.5.1. Comentarios sobre los resultados del filtrado por Information Coefficient (IC)**

In [88]:
print("Full day H=60:", ti_ic_60_relevant_list_full_day)
print("Full day H=90:", ti_ic_90_relevant_list_full_day)
print("Gestation H=60:", ti_ic_60_relevant_list_gestation)
print("Gestation H=90:", ti_ic_90_relevant_list_gestation)
print("Execution H=60:", ti_ic_60_relevant_list_execution)
print("Execution H=90:", ti_ic_90_relevant_list_execution)

Full day H=60: ['bb_30_15', 'bb_30_20', 'bb_30_25', 'bb_60_15', 'bb_60_20', 'bb_60_25', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'roc_20', 'roc_30', 'roc_60', 'rsi_14', 'stoch_k_20', 'stoch_k_30']
Full day H=90: ['atr_norm_10', 'atr_norm_14', 'atr_norm_20', 'atr_norm_30', 'atr_norm_5', 'bb_15_15', 'bb_15_20', 'bb_15_25', 'bb_20_15', 'bb_20_20', 'bb_20_25', 'bb_30_15', 'bb_30_20', 'bb_30_25', 'bb_60_15', 'bb_60_20', 'bb_60_25', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'momentum_10', 'roc_10', 'roc_20', 'roc_30', 'roc_60', 'rsi_14', 'rsi_5', 'rsi_7', 'stoch_k_14', 'stoch_k_20', 'stoch_k_30']
Gestation H=60: ['bb_15_15', 'bb_15_20', 'bb_15_25', 'bb_20_15', 'bb_20_20', 'bb_20_25', 'bb_30_15', 'bb_30_20', 'bb_30_25', 'bb_60_15', 'bb_60_20', 'bb_60_25', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'macd', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_10', 'roc_20', 'roc_30', 'roc_5', 'roc_60', 'rsi_14', 'rsi_3', 'rsi_5', 'rsi_7', 'stoch_k_14', 'stoch_k_20', 'stoch_k_30']
Gestation H=90: ['bb_15_1

1) La selección inicial es amplia y redundante  

    El filtrado por IC con umbral |IC| ≥ 0.1 produce un conjunto numeroso de indicadores, con múltiples variantes dentro de las mismas familias (Bandas de Bollinger, EMA, ROC, RSI, Stochastic, Momentum).  
    Esto es esperable en una primera etapa basada exclusivamente en una métrica univariada y confirma la presencia de **alta redundancia informativa**, que deberá ser tratada en pasos posteriores.

2) Comportamiento del análisis sobre jornada completa (Full day)  

    Para el horizonte H = 60, el conjunto resultante es relativamente más acotado y está dominado por:
    - indicadores de tendencia (EMA),
    - indicadores de magnitud del retorno (ROC),
    - medidas de dispersión y volatilidad implícita (Bandas de Bollinger),- osciladores clásicos (RSI, Stochastic).

    Esto es consistente con un análisis global del día, donde sobreviven principalmente señales con **capacidad predictiva más estable y persistente** respecto del retorno futuro.

    Para H = 90, el conjunto se amplía e incorpora explícitamente medidas de volatilidad normalizada (`atr_norm_*`), lo cual es coherente con horizontes más largos, en los que la **escala del movimiento y el régimen de volatilidad** resultan más relevantes que el timing fino.

3) Ventanas de gestación y ejecución: señales de corto plazo  
    
    En ambas ventanas, y para ambos horizontes, aparecen de forma sistemática:
    - indicadores de momentum de corto plazo (`momentum_3`, `momentum_5`, `momentum_10`),
    - ROC de horizontes cortos,
    - MACD,
    - osciladores rápidos (RSI y Stochastic con ventanas pequeñas).

    Esto indica que, en estos tramos horarios, el mercado presenta un comportamiento más **reactivo y direccional**, y que las señales asociadas a cambios recientes de precio contienen mayor información predictiva sobre el retorno futuro.

4) Similitud entre gestation y execution

    Las listas obtenidas para las ventanas de gestación y ejecución son prácticamente idénticas, tanto para H = 60 como para H = 90.  
    Esto sugiere:
    - persistencia del régimen intradía entre ambas franjas, o
    - que el filtrado por IC, sin control de correlación, aún no discrimina adecuadamente señales redundantes dentro de un mismo contexto temporal.

    Este resultado no constituye un problema metodológico, sino una señal clara de que el análisis debe avanzar hacia criterios multivariados.

5) Interpretación correcta del rol del IC

    El IC está cumpliendo su función principal: **detectar indicadores que contienen información estadística relevante sobre el retorno simple futuro**, evaluada fila a fila y respetando la causalidad temporal.  
    No implica aún que todos los indicadores seleccionados deban ser utilizados conjuntamente, ni que aporten información complementaria.

6) Siguientes pasos metodológicos

    Este conjunto de indicadores **no debe utilizarse directamente para el entrenamiento de modelos**. El paso siguiente consiste en:
    - agrupar indicadores por familias y rol económico,
    - eliminar redundancias mediante análisis de correlación y clustering,
    - seleccionar uno o pocos representantes por cluster, priorizando robustez, estabilidad y complementariedad.

    En síntesis, el filtrado por IC constituye una etapa exploratoria efectiva para identificar señales con capacidad predictiva. El desafío siguiente es **reducir la dimensionalidad con criterio**, evitando sobre-representar un mismo fenómeno del mercado bajo múltiples parametrizaciones.


# **5. Correlación**

## **5.1. Correlación de indicadores técnicos**

Una vez identificado el conjunto de indicadores técnicos relevantes a partir del análisis de Information Coefficient (IC), se procede a estudiar la estructura de dependencia existente entre ellos.

El objetivo de este análisis es cuantificar el grado de colinealidad y redundancia informativa dentro del conjunto seleccionado, así como comprender cómo se relacionan entre sí las distintas familias de indicadores técnicos (tendencia, momentum, magnitud y volatilidad).

Para ello, se calcula la matriz de correlación de Spearman promedio por día, restringida al mismo contexto temporal utilizado en el análisis de IC. En particular, el estudio se realiza de manera diferenciada para la jornada completa y para las ventanas horarias de gestación (08:00–09:00) y de ejecución/expansión (09:00–10:00). Este enfoque permite capturar relaciones monótonas, robustas frente a outliers, y evaluar la estabilidad de dichas relaciones bajo distintos regímenes intradía.

El uso de correlaciones promedio por jornada preserva la consistencia intradía del comportamiento del mercado y evita que episodios aislados dominen la estimación de dependencia entre indicadores.

Este análisis constituye un paso clave previo a la resolución de clusters de redundancia y a la selección final de features, asegurando que el conjunto de variables utilizado en los modelos predictivos sea informativamente complementario y estadísticamente robusto.


### **5.1.1. Implementación para calculo de Correlación**

In [89]:
import numpy as np
import pandas as pd

def daily_spearman_corr_matrix(
    df: pd.DataFrame,
    indicator_columns: list[str],
    window_start: str,
    window_end: str,
    date_col: str = "date",
) -> pd.DataFrame:
    """
    Matriz Spearman PROMEDIO por día, restringida a la ventana horaria.

    Requisitos:
      - df.index: DatetimeIndex
      - df contiene columna date_col
      - indicator_columns existen en df
    """
    # 0) Validaciones rápidas
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("df.index debe ser DatetimeIndex para usar between_time().")

    if not indicator_columns:
        raise ValueError("indicator_columns está vacío. No hay indicadores para correlacionar.")

    if date_col not in df.columns:
        raise ValueError(f"Falta la columna '{date_col}' en df.")

    # 1) Filtrar ventana horaria
    df_win = df.between_time(window_start, window_end).copy()

    # 2) Validar columnas
    missing = [c for c in indicator_columns if c not in df_win.columns]
    if missing:
        raise ValueError(f"Faltan columnas en df: {missing}")

    # 3) Correlación Spearman por día (matriz)
    daily_corr = []
    for _, g in df_win.groupby(date_col):
        X = g[indicator_columns]
        corr = X.corr(method="spearman")

        # Guardar solo matrices con algo útil
        if corr.notna().values.any():
            daily_corr.append(corr)

    if not daily_corr:
        raise ValueError("No se pudieron calcular correlaciones (ventana vacía, NaNs o sin datos por día).")

    # 4) Promedio (alineado por índices/columnas)
    corr_mean = sum(daily_corr) / len(daily_corr)
    return corr_mean


def top_abs_correlations(
    corr: pd.DataFrame,
    top_n: int = 15,
    min_abs_rho: float = 0.0,
) -> pd.DataFrame:
    """
    Ranking de pares con mayor |rho| (sin duplicados i-j y sin diagonal).
    """
    c = corr.copy()

    # Enmascarar diagonal y triángulo inferior (evita duplicados)
    mask = np.tril(np.ones(c.shape, dtype=bool))
    c = c.mask(mask)

    pairs = (
        c.stack()
         .rename("rho")
         .reset_index()
         .rename(columns={"level_0": "indicator_1", "level_1": "indicator_2"})
    )

    pairs["abs_rho"] = pairs["rho"].abs()
    pairs = pairs[pairs["abs_rho"] >= min_abs_rho]

    return pairs.sort_values("abs_rho", ascending=False).head(top_n).reset_index(drop=True)



### **5.1.2. Aplicación de cálculo de Correlación**

In [90]:
# ============================================================
# Configuración de ventanas (alineado con full_day / gestation / execution)
# ============================================================

WINDOWS = {
    "full_day":   (start_time_full_day, final_time_full_day),
    "gestation":  (start_time_gestation_window, final_time_gestation_window),
    "execution":  (start_time_execution_window, final_time_execution_window),
}

# ============================================================
# Listas de indicadores relevantes por horizonte y ventana
# ============================================================

INDICATORS = {
    ("full_day", 60):   ti_ic_60_relevant_list_full_day,
    ("full_day", 90):   ti_ic_90_relevant_list_full_day,
    ("gestation", 60):  ti_ic_60_relevant_list_gestation,
    ("gestation", 90):  ti_ic_90_relevant_list_gestation,
    ("execution", 60):  ti_ic_60_relevant_list_execution,
    ("execution", 90):  ti_ic_90_relevant_list_execution,
}

# ============================================================
# Calcular matrices de correlación Spearman promedio por día
# ============================================================

corr_mean = {}

for (window_name, horizon), indicators in INDICATORS.items():
    w_start, w_end = WINDOWS[window_name]

    # Si por algún motivo la lista está vacía, evitamos errores
    if not indicators:
        print(f"[warn] Lista vacía: window={window_name}, horizon={horizon}. Se omite.")
        continue

    corr_mean[(window_name, horizon)] = daily_spearman_corr_matrix(
        df=mnq_intraday_with_indicators,
        indicator_columns=indicators,
        window_start=w_start,
        window_end=w_end,
        date_col="date",
    )

# ============================================================
# Acceso a resultados (variables explícitas, naming consistente)
# ============================================================

corr_mean_ic_60_relevant_full_day   = corr_mean[("full_day", 60)]
corr_mean_ic_90_relevant_full_day   = corr_mean[("full_day", 90)]

corr_mean_ic_60_relevant_gestation  = corr_mean[("gestation", 60)]
corr_mean_ic_90_relevant_gestation  = corr_mean[("gestation", 90)]

corr_mean_ic_60_relevant_execution  = corr_mean[("execution", 60)]
corr_mean_ic_90_relevant_execution  = corr_mean[("execution", 90)]


In [164]:
CORR_MATRIX = """
print("\ncorr_mean_ic_60_relevant_full_day \n")
display(corr_mean_ic_60_relevant_full_day )
print("\ncorr_mean_ic_60_relevant_gestation \n")
display(corr_mean_ic_60_relevant_gestation)
print("\ncorr_mean_ic_60_relevant_execution \n")
display(corr_mean_ic_60_relevant_execution)


print("\ncorr_mean_ic_90_relevant_full_day \n")
display(corr_mean_ic_60_relevant_full_day)
print("\ncorr_mean_ic_90_relevant_gestation \n")
display(corr_mean_ic_90_relevant_gestation)
print("\ncorr_mean_ic_90_relevant_execution \n")
display(corr_mean_ic_90_relevant_execution)
"""

### **5.1.3. Observaciones sobre las matrices de correlación (features relevantes por IC)**



- **Redundancia extrema en Bollinger**: en *full day* las variantes `bb_30_15/20/25` son **1.00 entre sí** (idénticas) y las `bb_60_15/20/25` también. Esto indica que esas parametrizaciones aportan prácticamente la misma señal → conviene dejar **una sola** por familia.

- **Bloques muy correlacionados (multicolinealidad alta)**: en *full day (H=60)* casi todas las correlaciones están por arriba de **0.80–0.95** entre familias (BB ↔ EMA ↔ RSI ↔ Stoch). Por ejemplo, `rsi_14` con BB y con EMA ronda **0.95–0.97**. Esto sugiere que muchos indicadores “distintos” están midiendo el mismo fenómeno (nivel/tendencia + posición relativa).

- **ROC es el más “distinto” dentro del set**: `roc_60` reduce bastante sus correlaciones (p.ej. con BB ~0.49, con Stoch ~0.41, con RSI ~0.60). Si el objetivo es **complementariedad**, `roc_60` es candidato a sobrevivir frente a ROC más correlacionados.

- **EMA es casi una sola variable**: `ema_15`–`ema_20` correlacionan ~**0.99**, y en general todas las EMA están muy altas entre sí. Conservar **una** (o como máximo dos: corto y largo) suele ser suficiente.

- **Gestation y Execution son prácticamente el mismo mapa**: los patrones de correlación y magnitudes son muy similares en ambas matrices (duplicados 1.00 en BB, alta correlación en RSI cortos, etc.). Esto sugiere que la diferencia entre ventanas no está en la redundancia interna, sino en **qué indicadores pasan el umbral de IC**.

- **Indicadores “rápidos” forman un cluster propio**: en gestation/execution, `rsi_3`, `rsi_5`, `rsi_7` están **muy correlacionados** (≈0.95–0.98). Lo mismo `stoch_k_14/20/30` (≈0.92–0.94). Aquí también conviene elegir **un representante**.

- **MACD aporta algo de independencia relativa**: presenta correlaciones moderadas (p.ej. con `roc_30` ~0.33–0.44; con BB ~0.77–0.81). No es “ortogonal”, pero suele aportar un ángulo diferente frente a RSI/Stoch.

Criterio práctico (resultado esperado tras clustering): el set final probablemente se reduzca a algo como  
**1 BB + 1 EMA + 1 ROC (típicamente el más largo) + 1 oscilador (RSI o Stoch) + (opcional) MACD** por ventana/horizonte.



##**5.2. Clusters de Correlación**


Dado el alto nivel de colinealidad observado entre numerosos indicadores (y la redundancia entre distintas parametrizaciones), se construyen clusters de indicadores para agrupar señales equivalentes y obtener un set final más eficiente. Este procedimiento permite:

- agrupar indicadores que capturan esencialmente la misma información,
- reducir dimensionalidad sin perder señal,
- seleccionar representantes por cluster (o combinar señales) para disminuir el riesgo de sobreajuste,
- mejorar la interpretabilidad y robustez del conjunto de features.

Definición conceptual (en términos de grafo):

- Cada indicador es un nodo.

- Existe una arista entre dos indicadores i,j si su correlación absoluta cumple:   $|𝜌_{ij} \ge 0.85|$

- Cada componente conexa del grafo es un cluster informativo.

En otras palabras, cada cluster representa un grupo de indicadores que comparten prácticamente la misma información y, por lo tanto, pueden tratarse como una misma “familia” operativa a efectos de selección de features.

### **5.2.1. Código para construcción de clusters**

Este código:
- toma la matriz de correlación media (corr_mean_ic_h_relevant)
- detecta clusters automáticamente
- devuelve un DataFrame limpio para inspección

In [165]:
import numpy as np
import pandas as pd
import networkx as nx


def build_correlation_clusters(
    corr: pd.DataFrame,
    threshold: float = 0.85,
    *,
    use_upper_triangle: bool = True,
) -> pd.DataFrame:
    """
    Construye clusters de indicadores basados en alta correlación (|rho| >= threshold).

    Idea:
      - Cada indicador = nodo
      - Conectamos dos nodos con una arista si |rho| >= threshold
      - Cada componente conexa del grafo = cluster de redundancia

    Parameters
    ----------
    corr : pd.DataFrame
        Matriz de correlación (Spearman promedio), index = columns = indicadores.
    threshold : float
        Umbral de |rho| para considerar redundancia.
    use_upper_triangle : bool
        Si True, recorre solo el triángulo superior (más eficiente y sin duplicar aristas).

    Returns
    -------
    clusters_df : pd.DataFrame
        Columnas:
          - cluster_id
          - indicator
          - n_in_cluster
    """

    # -------------------------
    # 0) Validaciones mínimas
    # -------------------------
    if not isinstance(corr, pd.DataFrame) or corr.empty:
        raise ValueError("corr debe ser un DataFrame no vacío.")

    if corr.shape[0] != corr.shape[1]:
        raise ValueError("corr debe ser una matriz cuadrada (NxN).")

    if list(corr.index) != list(corr.columns):
        # No es obligatorio, pero evita sorpresas
        corr = corr.copy()
        corr = corr.loc[corr.index, corr.index]

    indicators = corr.columns.tolist()

    # -------------------------
    # 1) Crear grafo
    # -------------------------
    G = nx.Graph()
    G.add_nodes_from(indicators)

    # -------------------------
    # 2) Agregar aristas (|rho| >= threshold)
    # -------------------------
    if use_upper_triangle:
        # Recorre solo i < j (sin duplicar)
        for a, i in enumerate(indicators):
            for j in indicators[a + 1:]:
                rho = corr.at[i, j]
                if pd.notna(rho) and abs(rho) >= threshold:
                    G.add_edge(i, j, weight=float(rho))
    else:
        # Recorre todo (más lento, pero explícito)
        for i in indicators:
            for j in indicators:
                if i == j:
                    continue
                rho = corr.at[i, j]
                if pd.notna(rho) and abs(rho) >= threshold:
                    G.add_edge(i, j, weight=float(rho))

    # -------------------------
    # 3) Componentes conexas = clusters
    # -------------------------
    components = list(nx.connected_components(G))

    # -------------------------
    # 4) Armar DataFrame de salida
    # -------------------------
    records = []
    for cluster_id, comp in enumerate(components):
        comp = sorted(comp)
        n = len(comp)
        for ind in comp:
            records.append({"cluster_id": cluster_id, "indicator": ind, "n_in_cluster": n})

    clusters_df = (
        pd.DataFrame(records)
          .sort_values(["n_in_cluster", "cluster_id", "indicator"], ascending=[False, True, True])
          .reset_index(drop=True)
    )

    return clusters_df


### **5.2.2. Aplicación**

- `cluster_id`: identifica cada grupo informativo
- `n_in_cluster`:
  - 1 → indicador independiente
  - $>$ 1 → redundancia fuerte

In [94]:
# ============================================================
# Cálculo de clusters (sin repetir llamadas)
# ============================================================

THRESH = 0.85

# Diccionario de matrices de correlación ya calculadas (de la etapa anterior)
CORR_MATS = {
    ("full_day", 60): corr_mean_ic_60_relevant_full_day,
    ("full_day", 90): corr_mean_ic_90_relevant_full_day,
    ("gestation", 60): corr_mean_ic_60_relevant_gestation,
    ("gestation", 90): corr_mean_ic_90_relevant_gestation,
    ("execution", 60): corr_mean_ic_60_relevant_execution,
    ("execution", 90): corr_mean_ic_90_relevant_execution,
}

# Calcula todos los clusters en un loop
clusters = {}
for (window_name, horizon), corr_mat in CORR_MATS.items():
    clusters[(window_name, horizon)] = build_correlation_clusters(
        corr=corr_mat,
        threshold=THRESH,
        use_upper_triangle=True,  # más eficiente
    )

# Acceso con nombres “tipo variable” (como usted venía usando)
clusters_ic_60_full_day  = clusters[("full_day", 60)]
clusters_ic_90_full_day  = clusters[("full_day", 90)]
clusters_ic_60_gestation = clusters[("gestation", 60)]
clusters_ic_90_gestation = clusters[("gestation", 90)]
clusters_ic_60_execution = clusters[("execution", 60)]
clusters_ic_90_execution = clusters[("execution", 90)]

In [169]:
IC_CLUSTERS = """

print('clusters_ic_60_full_day:')
display(clusters_ic_60_full_day)

print('clusters_ic_60_gestation:')
display(clusters_ic_60_gestation)

print('clusters_ic_60_execution:')
display(clusters_ic_60_execution)

print('clusters_ic_90_full_day:')
display(clusters_ic_90_full_day)

print('clusters_ic_90_gestation:')
display(clusters_ic_90_gestation)

print('clusters_ic_90_execution:')
display(clusters_ic_90_execution)

"""

### **5.2.3. Observaciones de clusters**

**Full Day**

**H = 60**
- Existe un único cluster dominante que agrupa BB, EMA, ROC cortos, RSI y Stochastic, evidenciando **redundancia informativa muy alta**.
- `roc_60` queda aislado, confirmando que captura una **dinámica distinta y de mayor escala temporal**.
- Interpretación: para H=60 es suficiente seleccionar **un representante del cluster principal** y complementar con `roc_60`.

**H = 90**
- El cluster dominante se amplía e incorpora momentum y ROC cortos.
- Los indicadores de volatilidad (`atr_norm_*`) forman un cluster separado, aportando una dimensión adicional.
- `roc_60` permanece aislado.
- Interpretación: para H=90 conviene un set mínimo compuesto por:
  - un indicador del cluster principal,
  - un indicador de volatilidad (ATR),
  - `roc_60`.

---

**Gestation**

**H = 60 y H = 90**
- Se observa un gran cluster redundante (23 indicadores) dominado por BB, EMA, RSI y Stochastic.
- Un cluster secundario (`macd`, `momentum_10`, `roc_10`) sugiere señales de **impulso temprano**.
- Indicadores como `momentum_3`, `roc_20`, `roc_30` y `roc_60` quedan aislados, aportando diversidad por escala temporal.
- Interpretación: la ventana de gestación concentra alta colinealidad, pero permite construir un set compacto combinando:
  - un indicador estructural,
  - uno de momentum,
  - `roc_60` como ancla de largo plazo.

---

**Execution**

**H = 60 y H = 90**
- Aparece el cluster dominante más grande de todo el análisis (27 indicadores), incluyendo BB, EMA, RSI, Stochastic, MACD y momentum/ROC cortos.
- Esto indica que, una vez iniciado el movimiento, la mayoría de los indicadores clásicos **describen el mismo estado del mercado**.
- `roc_60` y algunos momentum muy cortos permanecen aislados.
- Interpretación: en execution, la diversidad informativa es mínima; el set de features debe ser **muy reducido**, priorizando:
  - señales estructurales (`roc_60`, eventualmente `ema_60`),
  - uno o dos indicadores de corto plazo como complemento.

---

**Conclusión final**

Las observaciones originales **siguen siendo válidas** en su núcleo conceptual.  
La corrección principal es **reducir la carga operativa implícita del lenguaje** y reforzar que el clustering confirma:

- redundancia extrema,
- estabilidad estructural por ventana,
- y el rol central de `roc_60` como feature no redundante.

Metodológicamente, el análisis está bien planteado y listo para pasar a la **selección final de representantes por cluster**.

##**5.3. Procesamiento de clusters**


### **5.3.1. Resolución de cluster dominante**

Una vez identificados los clusters por alta correlación (redundancia), el objetivo es conservar un solo indicador por cluster, eligiendo el que tenga mayor potencia predictiva.

Para eso, dentro de cada `cluster_id` se selecciona el indicador con mayor `abs_IC_delta` (y, en caso de empate, mayor `abs(IC_trade_only)`).

### **5.3.2. Código**

In [100]:
import numpy as np
import pandas as pd

# ============================================================
# Resolver clusters por IC (adaptado a IS/OOS + ventanas)
# ============================================================
# Cambio clave respecto a su versión anterior:
# - Ya NO usamos IC_delta / abs_IC_delta (viejo esquema).
# - Ahora usamos IC_OOS (y su absoluto) como criterio principal,
#   porque queremos seleccionar indicadores que GENERALICEN fuera de muestra.
#
# Entradas esperadas (ic_relevant):
#   ['indicator','horizon','IC_IS','IC_OOS','abs_IC_OOS','n_pairs_OOS','note', ...]
# Entradas esperadas (clusters_df):
#   ['cluster_id','indicator','n_in_cluster']
#
# Nota:
# - El DataFrame de clusters NO tiene horizon (la correlación fue por lista),
#   así que el "horizon" lo aporta ic_relevant al hacer merge.
# - Para ventana "complete / gestation / expansion", llamaremos esta función
#   con el ic_relevant y clusters_df correspondientes a esa ventana.
# ============================================================

def resolve_clusters_by_ic_oos(
    ic_relevant: pd.DataFrame,
    clusters_df: pd.DataFrame,
    *,
    prefer_ic_is_tiebreak: bool = True,
) -> pd.DataFrame:
    """
    Selecciona 1 indicador por cluster, usando dominancia de IC out-of-sample (OOS).

    Selección:
      1) Mayor abs_IC_OOS
      2) (opcional) Si hay empate: mayor abs(IC_IS) (para preferir señal consistente)
      3) Si persiste: orden alfabético (determinístico)

    Devuelve tabla con:
      - window_name (si se agrega fuera), horizon, cluster_id, n_in_cluster,
        indicator, keep, reason, IC_IS, IC_OOS, abs_IC_OOS, n_pairs_OOS, note
    """

    needed_ic = {"indicator", "horizon", "IC_IS", "IC_OOS", "abs_IC_OOS", "n_pairs_OOS", "note"}
    needed_cl = {"cluster_id", "indicator", "n_in_cluster"}

    if not needed_ic.issubset(ic_relevant.columns):
        raise ValueError(f"ic_relevant debe contener: {sorted(needed_ic)}")
    if not needed_cl.issubset(clusters_df.columns):
        raise ValueError(f"clusters_df debe contener: {sorted(needed_cl)}")

    # Merge: cluster info + IC (incluye horizon)
    df = clusters_df.merge(
        ic_relevant[["indicator", "horizon", "IC_IS", "IC_OOS", "abs_IC_OOS", "n_pairs_OOS", "note"]],
        on="indicator",
        how="left"
    )

    # Validar que todos los indicadores tengan IC_OOS
    missing = df[df["abs_IC_OOS"].isna()]["indicator"].unique().tolist()
    if missing:
        raise ValueError(f"Indicadores en clusters sin IC_OOS en ic_relevant: {missing}")

    out_rows = []

    # Resolver por (horizon, cluster_id) para no mezclar criterios entre H=60 y H=90
    for (h, cid), g in df.groupby(["horizon", "cluster_id"], sort=True):
        g = g.copy()

        # 1) criterio principal: max abs_IC_OOS
        max_abs = g["abs_IC_OOS"].max()
        candidates = g[g["abs_IC_OOS"] == max_abs].copy()

        # 2) desempate: abs(IC_IS) (si se solicita)
        tiebreak_used = False
        if prefer_ic_is_tiebreak and len(candidates) > 1:
            candidates["abs_IC_IS_tb"] = candidates["IC_IS"].abs()
            candidates = candidates.sort_values(["abs_IC_IS_tb", "indicator"], ascending=[False, True])
            best = candidates.iloc[0]
            tiebreak_used = True
        else:
            # 3) determinístico: alfabético si hay varios
            candidates = candidates.sort_values("indicator", ascending=True)
            best = candidates.iloc[0]

        best_indicator = best["indicator"]

        for _, row in g.iterrows():
            keep = (row["indicator"] == best_indicator)

            if keep:
                reason = "max abs_IC_OOS"
                if len(candidates) > 1:
                    if prefer_ic_is_tiebreak and tiebreak_used:
                        reason += " (tiebreak: abs(IC_IS))"
                    else:
                        reason += " (tiebreak: indicator order)"
            else:
                reason = f"redundant (same cluster as {best_indicator})"

            out_rows.append({
                "horizon": int(row["horizon"]),
                "cluster_id": int(cid),
                "n_in_cluster": int(row["n_in_cluster"]),
                "indicator": row["indicator"],
                "keep": bool(keep),
                "reason": reason,
                "IC_IS": float(row["IC_IS"]),
                "IC_OOS": float(row["IC_OOS"]),
                "abs_IC_OOS": float(row["abs_IC_OOS"]),
                "n_pairs_OOS": int(row["n_pairs_OOS"]),
                "note": str(row["note"]) if row["note"] is not None else "",
            })

    resolved = (
        pd.DataFrame(out_rows)
        .sort_values(["horizon", "cluster_id", "keep", "abs_IC_OOS"], ascending=[True, True, False, False])
        .reset_index(drop=True)
    )

    return resolved

In [101]:
# ============================================================
# Orquestación por ventana y horizonte (complete / gestation / expansion)
# ============================================================
# Requiere que ya tenga:
# - ic tables por ventana (output de compute_ic_table_is_oos_delta):
#     ic_table_complete, ic_table_gestation, ic_table_expansion
# - y clusters por ventana/horizonte:
#     clusters[("complete",60)], etc. (como lo armamos antes)
#
# Importante:
# - "ic_table_*" contiene filas para h=60 y h=90.
# - Filtramos por horizon y por umbral |IC_OOS| (si aplica) ANTES de resolver clusters.
# ============================================================

TH_IC = 0.02  # ejemplo razonable para intradía; ajústelo a su criterio

def prepare_ic_relevant(ic_table: pd.DataFrame, horizon: int, th: float) -> pd.DataFrame:
    """
    Filtra la tabla de IC por horizonte y umbral de |IC_OOS|.
    Devuelve un ic_relevant compatible con resolve_clusters_by_ic_oos.
    """
    dfh = ic_table[ic_table["horizon"] == horizon].copy()

    # Asegurar abs_IC_OOS (por si no existiera)
    if "abs_IC_OOS" not in dfh.columns and "IC_OOS" in dfh.columns:
        dfh["abs_IC_OOS"] = dfh["IC_OOS"].abs()

    # Filtrar por umbral (señal mínima OOS)
    dfh = dfh[dfh["abs_IC_OOS"] >= th].copy()

    # Nota: rename por compatibilidad si su tabla usa n_pairs_OOS/n_pairs_IS
    # (si ya los tiene, no hace nada)
    if "n_pairs_OOS" not in dfh.columns and "n_pairs_OOS" in dfh.columns:
        pass

    return dfh


In [102]:
def resolve_window(
    window_name: str,
    ic_table_window: pd.DataFrame,
    clusters_dict: dict[tuple[str, int], pd.DataFrame],
    threshold_ic: float = TH_IC,
) -> dict[int, dict[str, object]]:
    """
    Resuelve clusters para una ventana (complete/gestation/expansion) y devuelve:
      - final_indicators por horizonte
      - resolved tables por horizonte
    """
    result = {}

    for h in (60, 90):
        # 1) ic_relevant para este horizonte
        ic_rel = prepare_ic_relevant(ic_table_window, horizon=h, th=threshold_ic)

        # Si queda vacío, no hay nada que resolver
        if ic_rel.empty:
            result[h] = {
                "final_indicators": [],
                "resolved": pd.DataFrame(),
                "note": f"Sin indicadores que superen el umbral |IC_OOS| >= {threshold_ic} en {window_name} (H={h})"
            }
            continue

        # 2) clusters de esta ventana/horizonte (construidos a partir de los relevantes)
        cl = clusters_dict[(window_name, h)].copy()

        # 3) filtrar clusters a indicadores presentes en ic_rel (safety)
        cl_rel = cl[cl["indicator"].isin(ic_rel["indicator"])].copy()

        # 4) resolver
        resolved = resolve_clusters_by_ic_oos(
            ic_relevant=ic_rel,
            clusters_df=cl_rel,
            prefer_ic_is_tiebreak=True,
        )

        # 5) lista final
        final_indicators = resolved.loc[resolved["keep"], "indicator"].tolist()

        result[h] = {
            "final_indicators": final_indicators,
            "resolved": resolved,
            "note": ""
        }

    return result

### **5.3.3. Aplicación**

In [103]:
# Suponga que:
# - ic_table_complete, ic_table_gestation, ic_table_expansion ya están calculadas
# - clusters dict ya existe: clusters[(window,h)]

full_day_res = resolve_window("full_day",  ic_table_full_day,  clusters, threshold_ic=TH_IC)
gest_res     = resolve_window("gestation", ic_table_gestation, clusters, threshold_ic=TH_IC)
exe_res      = resolve_window("execution", ic_table_execution, clusters, threshold_ic=TH_IC)

In [104]:
full_day_final_60 = full_day_res[60]["final_indicators"]
full_day_final_90 = full_day_res[90]["final_indicators"]
gestation_final_60 = gest_res[60]["final_indicators"]
gestation_final_90 = gest_res[90]["final_indicators"]
execution_final_60 = exe_res[60]["final_indicators"]
execution_final_90 = exe_res[90]["final_indicators"]

In [170]:
# Para ver qué se conserva:
print("\nfull_day_res (H=60):\n")
display(full_day_res[60]["resolved"].query("keep").sort_values("abs_IC_OOS", ascending=False))
print("\nfull_day_res (H=90):\n")
display(full_day_res[90]["resolved"].query("keep").sort_values("abs_IC_OOS", ascending=False))
print("\ngestation_res (H=60):\n")
display(gest_res[60]["resolved"].query("keep").sort_values("abs_IC_OOS", ascending=False))
print("\ngestation_res (H=90):\n")
display(gest_res[90]["resolved"].query("keep").sort_values("abs_IC_OOS", ascending=False))
print("\nexecution_res (H=60):\n")
display(exe_res[60]["resolved"].query("keep").sort_values("abs_IC_OOS", ascending=False))
print("\nexecution_res (H=60):\n")
display(exe_res[90]["resolved"].query("keep").sort_values("abs_IC_OOS", ascending=False))


full_day_res (H=60):



,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,60,0,15,ema_60,True,max abs_IC_OOS,-0.193578,-0.189173,0.189173,229712,
15,60,1,1,roc_60,True,max abs_IC_OOS,-0.187156,-0.177858,0.177858,229712,



full_day_res (H=90):



,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
5,90,1,26,ema_60,True,max abs_IC_OOS,-0.245388,-0.238799,0.238799,229712,
31,90,2,1,roc_60,True,max abs_IC_OOS,-0.234883,-0.225245,0.225245,229712,
0,90,0,5,atr_norm_20,True,max abs_IC_OOS,0.123218,0.107646,0.107646,229712,



gestation_res (H=60):



...

,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,90,0,23,ema_60,True,max abs_IC_OOS,-0.287640,-0.320655,0.320655,35746,
31,90,6,1,roc_60,True,max abs_IC_OOS,-0.255658,-0.285639,0.285639,35746,
30,90,5,1,roc_30,True,max abs_IC_OOS,-0.251744,-0.283501,0.283501,35746,
29,90,4,1,roc_20,True,max abs_IC_OOS,-0.226354,-0.262147,0.262147,35746,
23,90,1,3,momentum_10,True,max abs_IC_OOS (tiebreak: abs(IC_IS)),-0.180778,-0.211418,0.211418,35746,
27,90,3,2,momentum_5,True,max abs_IC_OOS (tiebreak: abs(IC_IS)),-0.148082,-0.163650,0.163650,35746,
26,90,2,1,momentum_3,True,max abs_IC_OOS,-0.118889,-0.130688,0.130688,35746,



execution_res (H=60):



,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,60,0,27,ema_60,True,max abs_IC_OOS,-0.594698,-0.556157,0.556157,35746,
30,60,3,1,roc_20,True,max abs_IC_OOS,-0.522272,-0.491503,0.491503,35746,
31,60,4,1,roc_60,True,max abs_IC_OOS,-0.542577,-0.484769,0.484769,35746,
28,60,2,2,momentum_5,True,max abs_IC_OOS (tiebreak: abs(IC_IS)),-0.335210,-0.309230,0.309230,35746,
27,60,1,1,momentum_3,True,max abs_IC_OOS,-0.270437,-0.252229,0.252229,35746,



execution_res (H=60):



,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,90,0,27,ema_60,True,max abs_IC_OOS,-0.621967,-0.597707,0.597707,35746,
30,90,3,1,roc_20,True,max abs_IC_OOS,-0.543496,-0.526863,0.526863,35746,
31,90,4,1,roc_60,True,max abs_IC_OOS,-0.569866,-0.519405,0.519405,35746,
28,90,2,2,momentum_5,True,max abs_IC_OOS (tiebreak: abs(IC_IS)),-0.344929,-0.331712,0.331712,35746,
27,90,1,1,momentum_3,True,max abs_IC_OOS,-0.278032,-0.268738,0.268738,35746,


### **5.3.4. Validación de conclusiones tras la resolución de clusters**


**1. Lectura global: validación estructural**

A. `ema_60` como factor dominante de tendencia

- `ema_60` es seleccionado de forma consistente en:
  - jornada completa,
  - ventana de gestación,
  - ventana de ejecución,
  - para ambos horizontes (H = 60 y H = 90).
- En todos los casos proviene de **clusters grandes**, lo que indica alta redundancia entre indicadores de tendencia.
- A pesar de ello:
  - maximiza sistemáticamente `|IC_OOS|`,
  - mantiene signo y magnitud coherentes entre IS y OOS,
  - alcanza los valores absolutos más altos del análisis (hasta ~0.60 en ejecución).

**Conclusión**  
> `ema_60` actúa como un **factor estructural de tendencia intradía**, robusto, estable fuera de muestra y no dependiente del contexto horario.

---

B. Indicadores `roc_*` como factores de magnitud del movimiento

- `roc_60`:
  - aparece en **todos los contextos** (full day, gestation y execution),
  - para ambos horizontes,
  - siempre como **cluster unitario o casi unitario**, indicando baja redundancia.
- `roc_20` y `roc_30`:
  - emergen únicamente en ventanas activas (gestación y expansión),
  - con IC OOS significativo, aunque menor que `roc_60`.

**Conclusión**  
> La magnitud del movimiento reciente es informativa en múltiples escalas, pero  
> `roc_60` representa un componente estructural, mientras que `roc_20` y `roc_30` dependen del tramo horario.

---

C. Indicadores `momentum_*` como factores de aceleración

- No aparecen como relevantes en la jornada completa.
- Surgen de forma consistente en:
  - ventana de gestación (`momentum_3`, `momentum_5`, `momentum_10`),
  - ventana de ejecución (`momentum_3`, `momentum_5`).
- Son seleccionados por maximizar `abs(IC_OOS)`, con valores relevantes (~0.17–0.33).

**Conclusión**  
> El momentum de corto plazo no es un factor estructural del día completo,  
> pero captura eficazmente la **aceleración local del precio** en ventanas activas del mercado.

---

**2. Diferencias por ventana: lectura contextual**

Jornada completa (full day)

- **H = 60**
  - `ema_60`
  - `roc_60`
- **H = 90**
  - `ema_60`
  - `roc_60`
  - `atr_norm_20`

**Interpretación**  
- Señales lentas y estables.
- Adecuadas para:
  - caracterizar el régimen intradía,
  - definir estructura de fondo,
  - servir como baseline o filtro.

---

Ventana de gestación (08:00–09:00)

- Conjunto ampliado de señales:
  - tendencia: `ema_60`,
  - magnitud: `roc_20`, `roc_30`, `roc_60`,
  - aceleración: `momentum_3`, `momentum_5`, `momentum_10`.
- IC OOS elevado (hasta ~0.45).

**Conclusión**  
> La ventana de gestación concentra información direccional temprana con alta capacidad anticipatoria.

---

Ventana de expansión / ejecución (09:00–10:00)

- IC OOS máximos del estudio (≈0.5–0.6).
- Dominio conjunto de:
  - tendencia (`ema_60`),
  - magnitud (`roc_*`),
  - aceleración (`momentum_*`).

**Matiz importante**  
Esto no implica “mejor predicción”, sino una **mayor alineación estadística con movimientos ya en desarrollo**.

---

**3. Aclaraciones metodológicas**

- La ausencia de `momentum_*` en la jornada completa **no invalida su utilidad**.
- Un IC negativo **no es un problema**: solo indica relación inversa con el target.
- La presencia de clusters grandes es normal en indicadores técnicos altamente correlacionados.
- El IC mide **informatividad estadística**, no timing ni reglas de entrada.

---

** 4. Conclusión general**

> El análisis conjunto de IC fuera de muestra y resolución de clusters revela una estructura jerárquica clara:
> - un **factor de tendencia estructural** (`ema_60`),
> - **factores de magnitud** (`roc_*`) con dependencia temporal,
> - y **factores de aceleración** (`momentum_*`) relevantes únicamente en ventanas activas.
>
> Esta evidencia justifica una selección jerárquica de features, combinando:
> - un núcleo estructural válido para toda la jornada,
> - con features específicas condicionadas por la ventana horaria.

La lógica previa no necesita corrección: **necesita consolidación**, y estos resultados lo hacen explícito y defendible.


In [108]:
print('TI full day finals H=60:', full_day_final_60)
print('TI full day finals H=90:', full_day_final_90)

print('TI gestation finals H=60:', gestation_final_60)
print('TI gestation finals H=90:', gestation_final_90)

print('TI execution finals H=60:', execution_final_60)
print('TI execution finals H=90:', execution_final_90)

TI full day finals H=60: ['ema_60', 'roc_60']
TI full day finals H=90: ['atr_norm_20', 'ema_60', 'roc_60']
TI gestation finals H=60: ['ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60']
TI gestation finals H=90: ['ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60']
TI execution finals H=60: ['ema_60', 'momentum_3', 'momentum_5', 'roc_20', 'roc_60']
TI execution finals H=90: ['ema_60', 'momentum_3', 'momentum_5', 'roc_20', 'roc_60']


### **4.4. Correlación final**

Una vez reducido el conjunto de indicadores técnicos mediante el filtrado por fuerza de señal (|IC| ≥ 0.10) y la resolución por clusters, analizaremos la matriz de correlación entre los indicadores resultantes con el objetivo de evaluar posibles redundancias adicionales.

**Criterio final para el análisis de correlación entre indicadores técnicos**

En esta etapa, el criterio de selección deja de ser puramente estadístico y pasa a ser arquitectónico.  
El análisis de Information Coefficient (IC) y los clusters de correlación ya cumplieron su función principal: identificar señal y eliminar redundancia evidente.  
La correlación final no se utiliza para “volver a filtrar” indicadores, sino para ordenar su uso dentro del modelo.

**Principio rector**

> No todos los indicadores finales deben competir entre sí:  
> deben coexistir por rol funcional y por ventana horaria.

**Definición de roles**

Antes de analizar correlaciones, los indicadores se agrupan por el fenómeno de mercado que representan:

  - Estructura / tendencia  
    `ema_60`

  - Magnitud del movimiento  
    `roc_60`, `roc_20`, `roc_30`

  - Aceleración / impulso  
    `momentum_3`, `momentum_5`, `momentum_10`

  - Riesgo / volatilidad  
    `atr_norm_20`

La correlación solo debe penalizar redundancia dentro de un mismo rol, no entre roles distintos.

**Criterio de uso de la correlación**

A. Dentro de un mismo rol  

  La correlación se utiliza para reducir redundancia:

  - |ρ| ≥ 0.85 se considera redundancia fuerte.
  - Se conserva:
    - el indicador con mayor |IC_OOS|, o
    - el más estable entre ventanas, si los IC son similares.

  Ejemplo:
  - momentum_3, momentum_5 y momentum_10 suelen estar altamente correlacionados.
  - Se conserva uno solo por ventana.

B. Entre roles distintos

  La correlación no se utiliza como criterio de descarte, salvo casos extremos.

  Ejemplos:
  - ema_60 vs roc_60  
  - roc_60 vs momentum_5  

  Aunque puedan mostrar correlación, capturan fenómenos distintos (tendencia, magnitud, aceleración) y pueden ser explotados conjuntamente por modelos no lineales.

**Aplicación directa a los indicadores finales**

- Core estructural (invariable)

  Estos indicadores se conservan siempre:

  - `ema_60`
  - `roc_60`

  Son estables, robustos fuera de muestra, válidos en múltiples horizontes y ventanas.

- Jornada completa

  - H = 60: `ema_60`, `roc_60`
  - H = 90: `atr_norm_20`, `ema_60`, `roc_60`

  El indicador de volatilidad (`atr_norm_20`) cumple un rol distinto y no compite con EMA o ROC.

- Ventanas de gestación y expansión

  En estas ventanas se aplica correlación intra-rol:

  - Momentum:
    - conservar un único momentum corto (por ejemplo, momentum_5 o momentum_3).
  - ROC:
    - conservar roc_60 como base y, opcionalmente, un ROC de escala más corta.

  Ejemplo razonable por ventana:
  - ema_60
  - roc_60
  - roc_20 o roc_30
  - momentum_5

**Regla operativa final**

> La correlación final no se utiliza para eliminar señales válidas,  
> sino para evitar redundancia dentro del mismo rol funcional.  
> Los indicadores estructurales (ema_60, roc_60) se conservan siempre,  
> mientras que los indicadores dependientes de ventana se seleccionan maximizando diversidad funcional: tendencia, magnitud, aceleración y riesgo.

**Síntesis práctica**

- Core fijo: ema_60, roc_60  
- Por ventana:
  - un ROC corto,
  - un momentum corto,
  - opcionalmente un indicador de volatilidad.

A partir de aquí, la responsabilidad de combinar y ponderar estas señales se delega al modelo.


### **4.5. Tratamiento de familias en ventanas de gestación y expansión**

In [109]:
import pandas as pd

def pick_one_by_family(
    *,
    window_name: str,
    horizon: int,
    finals_list: list[str],
    ic_table_window: pd.DataFrame,     # ic_table_full_day / ic_table_gestation / ic_table_execution
    corr_mean_window: pd.DataFrame,    # corr_mean correspondiente (misma ventana y horizonte)
    families: dict[str, list[str]],    # grupos candidatos redundantes (momentum, roc_short, etc.)
    corr_threshold: float = 0.85,
    score_col: str = "abs_IC_OOS",     # criterio principal: señal OOS
) -> dict:
    """
    Para una ventana/horizonte:
      - Mantiene el core que no está en families (o si lo incluye, lo respeta según usted defina)
      - Dentro de cada familia:
          - si los candidatos están altamente correlacionados (|rho|>=threshold),
            se queda con 1: el de mayor score_col (abs_IC_OOS)
          - si NO están altamente correlacionados entre sí, los deja (no los fuerza a 1)
    Devuelve:
      - selected: lista final
      - decisions: tabla con decisiones por familia
    """

    # -------------------------
    # 1) Tomar IC solo de este horizonte
    # -------------------------
    ic_h = ic_table_window[ic_table_window["horizon"] == horizon].copy()

    # Si no existe abs_IC_OOS, lo creamos
    if score_col not in ic_h.columns and "IC_OOS" in ic_h.columns:
        ic_h[score_col] = ic_h["IC_OOS"].abs()

    ic_h = ic_h.set_index("indicator")

    # -------------------------
    # 2) Sanity: aseguramos que corr tenga índices/cols compatibles
    # -------------------------
    corr = corr_mean_window.copy()
    corr = corr.loc[corr.index, corr.index]  # alinea si hace falta

    finals_set = set(finals_list)

    selected = set(finals_list)  # empezamos manteniendo todo
    decision_rows = []

    # -------------------------
    # 3) Resolver familia por familia
    # -------------------------
    for fam_name, fam_members in families.items():

        # Solo considerar los miembros que realmente están en esta lista final
        cand = [x for x in fam_members if x in finals_set]

        # Si hay 0 o 1, no hay nada que resolver
        if len(cand) <= 1:
            if len(cand) == 1:
                decision_rows.append({
                    "window": window_name, "horizon": horizon, "family": fam_name,
                    "candidates": cand, "action": "keep (single)", "kept": cand[0],
                    "reason": "only one candidate present"
                })
            continue

        # -------------------------
        # 3a) Ver si son redundantes (alta correlación)
        #     Criterio simple: si el máximo |rho| entre pares >= threshold, consideramos redundancia
        # -------------------------
        max_abs_rho = 0.0
        for i in range(len(cand)):
            for j in range(i + 1, len(cand)):
                if cand[i] in corr.index and cand[j] in corr.columns:
                    rho = corr.at[cand[i], cand[j]]
                    if pd.notna(rho):
                        max_abs_rho = max(max_abs_rho, abs(rho))

        # Si NO hay redundancia fuerte, los dejamos todos
        if max_abs_rho < corr_threshold:
            decision_rows.append({
                "window": window_name, "horizon": horizon, "family": fam_name,
                "candidates": cand, "action": "keep (not redundant)", "kept": cand,
                "reason": f"max |rho|={max_abs_rho:.3f} < {corr_threshold}"
            })
            continue

        # -------------------------
        # 3b) Redundantes: elegir el de mayor señal OOS (abs_IC_OOS)
        # -------------------------
        # Tomamos el score de cada candidato; si falta, lo dejamos en -inf para no elegirlo
        scored = []
        for ind in cand:
            score = ic_h.at[ind, score_col] if ind in ic_h.index and pd.notna(ic_h.at[ind, score_col]) else float("-inf")
            scored.append((ind, score))

        scored.sort(key=lambda t: (-t[1], t[0]))  # mayor score, desempate alfabético
        kept, kept_score = scored[0]

        # Eliminamos los demás de la selección
        for ind, _ in scored[1:]:
            if ind in selected:
                selected.remove(ind)

        decision_rows.append({
            "window": window_name, "horizon": horizon, "family": fam_name,
            "candidates": cand, "action": "select_one", "kept": kept,
            "reason": f"redundant (max |rho|={max_abs_rho:.3f} >= {corr_threshold}); kept by max {score_col}={kept_score:.6f}"
        })

    decisions = pd.DataFrame(decision_rows)
    return {"selected": sorted(selected), "decisions": decisions}


In [110]:
# Familias redundantes que queremos resolver dentro de cada ventana
FAMILIES = {
    "momentum":   ["momentum_3", "momentum_5", "momentum_10"],
    "roc_short":  ["roc_20", "roc_30"],  # roc_60 es core, no lo metemos aquí
}

# ---- Gestation ----
gest60 = pick_one_by_family(
    window_name="gestation",
    horizon=60,
    finals_list=['ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60'],
    ic_table_window=ic_table_gestation,
    corr_mean_window=corr_mean_ic_60_relevant_gestation,
    families=FAMILIES,
    corr_threshold=0.85,
)

gest90 = pick_one_by_family(
    window_name="gestation",
    horizon=90,
    finals_list=['ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60'],
    ic_table_window=ic_table_gestation,
    corr_mean_window=corr_mean_ic_90_relevant_gestation,
    families=FAMILIES,
    corr_threshold=0.85,
)

print("Gestation H=60 selected:", gest60["selected"])
display(gest60["decisions"])

print("Gestation H=90 selected:", gest90["selected"])
display(gest90["decisions"])


# ---- Expansion ----
exp60 = pick_one_by_family(
    window_name="execution",
    horizon=60,
    finals_list=['ema_60', 'momentum_3', 'momentum_5', 'roc_20', 'roc_60'],
    ic_table_window=ic_table_execution,
    corr_mean_window=corr_mean_ic_60_relevant_execution,
    families=FAMILIES,
    corr_threshold=0.85,
)

exp90 = pick_one_by_family(
    window_name="execution",
    horizon=90,
    finals_list=['ema_60', 'momentum_3', 'momentum_5', 'roc_20', 'roc_60'],
    ic_table_window=ic_table_execution,
    corr_mean_window=corr_mean_ic_90_relevant_execution,
    families=FAMILIES,
    corr_threshold=0.85,
)

print("Execution H=60 selected:", exp60["selected"])
display(exp60["decisions"])

print("Execution H=90 selected:", exp90["selected"])
display(exp90["decisions"])



Gestation H=60 selected: ['ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60']


,window,horizon,family,candidates,action,kept,reason
0,gestation,60,momentum,"[momentum_3, momentum_5, momentum_10]",keep (not redundant),"[momentum_3, momentum_5, momentum_10]",max |rho|=0.712 < 0.85
1,gestation,60,roc_short,"[roc_20, roc_30]",keep (not redundant),"[roc_20, roc_30]",max |rho|=0.687 < 0.85


Gestation H=90 selected: ['ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60']


,window,horizon,family,candidates,action,kept,reason
0,gestation,90,momentum,"[momentum_3, momentum_5, momentum_10]",keep (not redundant),"[momentum_3, momentum_5, momentum_10]",max |rho|=0.712 < 0.85
1,gestation,90,roc_short,"[roc_20, roc_30]",keep (not redundant),"[roc_20, roc_30]",max |rho|=0.687 < 0.85


Execution H=60 selected: ['ema_60', 'momentum_3', 'momentum_5', 'roc_20', 'roc_60']


,window,horizon,family,candidates,action,kept,reason
0,execution,60,momentum,"[momentum_3, momentum_5]",keep (not redundant),"[momentum_3, momentum_5]",max |rho|=0.723 < 0.85
1,execution,60,roc_short,[roc_20],keep (single),roc_20,only one candidate present


Execution H=90 selected: ['ema_60', 'momentum_3', 'momentum_5', 'roc_20', 'roc_60']


,window,horizon,family,candidates,action,kept,reason
0,execution,90,momentum,"[momentum_3, momentum_5]",keep (not redundant),"[momentum_3, momentum_5]",max |rho|=0.723 < 0.85
1,execution,90,roc_short,[roc_20],keep (single),roc_20,only one candidate present


**Gestation (H=60 y H=90):**

- Las familias momentum y roc_short no presentan redundancia fuerte (ρ < 0.85).
- Mantener múltiples escalas de momentum y ROC es consistente: cada una aporta información complementaria.
- El set final es estable entre horizontes → robustez temporal en la ventana de gestación.

**Execution (H=60 y H=90):**

- momentum_3 y momentum_5 no son redundantes, por lo que conviven correctamente.
- En roc_short solo queda roc_20 (caso trivial), sin conflicto de selección.
- El set final es idéntico para H=60 y H=90 → comportamiento consistente en ejecución.

**Lectura global:**

- La resolución por familias confirma que no se está forzando una reducción artificial.
- Las decisiones están guiadas por datos (correlación + IC), no por heurística.
- El pipeline preserva diversidad informativa donde realmente existe.

### **4.4. Decisión de consolidación final**

A) Estrategia general de entrenamiento

A partir del análisis de Information Coefficient (IC), correlación y clusters de redundancia, se concluye que **no es necesario entrenar modelos distintos por ventana horaria**.  
Es metodológicamente más sólido entrenar **un único modelo** que incorpore:

a. Un **core de features estructurales**, activas durante toda la jornada.  
b. Un conjunto de **features adicionales activadas de forma condicional** durante las ventanas de gestación y expansión.

Este enfoque permite:
- utilizar todo el volumen de datos disponible,
- evitar la fragmentación del dataset,
- reducir el riesgo de sobreajuste,
- respetar la evidencia empírica que muestra que ciertas señales solo son informativas en ventanas específicas.

El modelo aprende de forma implícita **cuándo** una feature aporta señal y cuándo no, sin forzar reglas externas ni entrenamientos separados.

---

B) Consolidación del listado final de features

Bajo este criterio, el conjunto final de features queda definido de la siguiente manera.

B.1) Target delta_60

Todo el día:
- ema_60  
- roc_60  

Ventana de gestación:
- ema_60  
- roc_60  
- momentum_10  
- momentum_3  
- momentum_5  
- roc_20  
- roc_30  

Ventana de expansión:
- ema_60  
- roc_60  
- momentum_3  
- momentum_5  
- roc_20  

---

B.2) Target delta_90

Todo el día:
- ema_60  
- roc_60  
- atr_norm_20  

Ventana de gestación:
- ema_60  
- roc_60  
- momentum_10  
- momentum_3  
- momentum_5  
- roc_20  
- roc_30  

Ventana de expansión:
- ema_60  
- roc_60  
- momentum_3  
- momentum_5  
- roc_20  

---

C) Evaluación del criterio adoptado

Este esquema es coherente con todos los resultados obtenidos:

a. ema_60 y roc_60 actúan como **factores estructurales**, robustos a lo largo de toda la jornada.  
b. atr_norm_20 aporta información de **riesgo y volatilidad**, relevante principalmente para horizontes más largos (delta_90).  
c. Los indicadores momentum_* y roc_* de corto plazo muestran **capacidad predictiva dependiente del horario**, especialmente en ventanas activas.  
d. No se observa redundancia fuerte que justifique eliminar estas señales dentro de las ventanas definidas.

---

D) Conclusión final

Es adecuado entrenar un único modelo con un core estructural activo durante todo el día y features adicionales activadas de forma condicional por ventana horaria.  
El listado final de features propuesto es consistente con el análisis empírico realizado y representa un equilibrio correcto entre robustez global y sensibilidad temporal.


In [111]:
# ============================================================
# Consolidación final de indicadores técnicos
# ============================================================

# Se combinan todos los indicadores seleccionados en:
# - jornada completa
# - ventana de gestación
# - ventana de expansión
# - para ambos horizontes (H=60 y H=90)

tech_indicators_finals = sorted(
    set(
        full_day_final_60
        + full_day_final_90
        + gestation_final_60
        + gestation_final_90
        + execution_final_60
        + execution_final_90
    )
)

# Mostrar resultado final
print("tech_indicators_finals:")
print(tech_indicators_finals)

tech_indicators_finals:
['atr_norm_20', 'ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60']


## **6. Análisis de coherencia direccional IS-OOS**

Una vez consolidado el conjunto final de indicadores técnicos:

```
tech_indicators_finals = [
  'atr_norm_20',
  'ema_60',
  'momentum_10',
  'momentum_3',
  'momentum_5',
  'roc_20',
  'roc_30',
  'roc_60']
```

Y priorizada su selección en función del desempeño fuera de muestra (OOS), se introduce un análisis adicional de carácter cualitativo: la coherencia direccional entre los períodos **in-sample (IS)** y **out-of-sample (OOS)**.

El objetivo de este análisis no es seleccionar indicadores, sino **validar su estabilidad conceptual**, verificando si cada indicador mantiene la misma dirección de relación con el target al pasar de IS a OOS. De este modo, se evalúa si la señal aprendida durante el entrenamiento conserva coherencia cuando se expone a datos no vistos.

Este enfoque permite detectar indicadores cuya señal OOS, aun siendo significativa en magnitud, presenta una **inversión de signo** respecto a IS, lo cual constituye una alerta de posible sobreajuste o inestabilidad estructural.

El criterio de coherencia se basa exclusivamente en la comparación del signo de `IC_IS` y `IC_OOS`, utilizando métricas OOS como referencia principal y dejando la diferencia `delta_oos_minus_is` como herramienta de diagnóstico complementaria.


### **5.1. Código para análisis de coherencia direccional**

In [112]:
ic_table_full_day

,indicator,horizon,IC_IS,IC_OOS,ret_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,-0.193578,-0.189173,0.004405,281064,229712,,0.189173
1,roc_60,60,-0.187156,-0.177858,0.009298,281064,229712,,0.177858
2,bb_60_15,60,-0.152318,-0.150149,0.002168,281064,229712,,0.150149
3,bb_60_20,60,-0.152318,-0.150149,0.002168,281064,229712,,0.150149
4,bb_60_25,60,-0.152318,-0.150149,0.002168,281064,229712,,0.150149
...,...,...,...,...,...,...,...,...,...
79,volume_ratio_90,90,0.032891,0.016484,-0.016406,281064,229712,,0.016484
80,volume_ratio_60,90,0.023643,0.011424,-0.012218,281064,229712,,0.011424
81,volume_ratio_30,90,0.015198,0.006253,-0.008945,281064,229712,,0.006253
82,volume_ratio_20,90,0.011545,0.005144,-0.006401,281064,229712,,0.005144


In [113]:
import numpy as np
import pandas as pd


def check_directional_coherence(
    df_selected: pd.DataFrame,
    *,
    is_col: str = "IC_IS",
    oos_col: str = "IC_OOS",
    abs_oos_col: str = "abs_IC_OOS",
) -> pd.DataFrame:
    """
    Verifica coherencia direccional entre IC_IS e IC_OOS usando ic_table_*.

    Criterio de coherencia:
      - coherent = True si sign(IC_IS) == sign(IC_OOS)
      - y ambos son distintos de 0
      - y ninguno es NaN

    Entradas esperadas (ic_table_*):
      ['indicator','horizon', IC_IS, IC_OOS, 'delta_oos_minus_is',
       'n_pairs_IS','n_pairs_OOS','note', 'abs_IC_OOS']

    Notas:
      - La columna 'family' es opcional; si existe, se incluye en el output.
      - Si falta abs_IC_OOS, se calcula como abs(IC_OOS).
      - Se agrega diagnóstico de estabilidad (gap) entre OOS e IS en magnitud y signo.
    """
    out = df_selected.copy()

    # -------------------------
    # Validaciones mínimas
    # -------------------------
    needed = {"indicator", "horizon", is_col, oos_col}
    missing = [c for c in needed if c not in out.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas en df_selected: {missing}")

    # -------------------------
    # abs_IC_OOS (si no existe)
    # -------------------------
    if abs_oos_col not in out.columns:
        out[abs_oos_col] = out[oos_col].abs()

    # -------------------------
    # Signos
    # -------------------------
    out["sign_IC_IS"] = np.sign(out[is_col])
    out["sign_IC_OOS"] = np.sign(out[oos_col])

    # -------------------------
    # Coherencia direccional IS vs OOS
    # -------------------------
    out["coherent_is_oos"] = (
        out[is_col].notna()
        & out[oos_col].notna()
        & (out["sign_IC_IS"] != 0)
        & (out["sign_IC_OOS"] != 0)
        & (out["sign_IC_IS"] == out["sign_IC_OOS"])
    )

    # -------------------------
    # Diagnósticos complementarios
    # -------------------------
    out["abs_IC_IS"] = out[is_col].abs()
    out["abs_IC_OOS"] = out[oos_col].abs()

    # Diferencia de magnitudes (OOS - IS): >0 implica que OOS es más fuerte en magnitud
    out["abs_gap_oos_minus_is"] = out["abs_IC_OOS"] - out["abs_IC_IS"]

    # Flag simple de "flip" de signo (si ambos no son 0/NaN y el signo difiere)
    out["sign_flip_is_oos"] = (
        out[is_col].notna()
        & out[oos_col].notna()
        & (out["sign_IC_IS"] != 0)
        & (out["sign_IC_OOS"] != 0)
        & (out["sign_IC_IS"] != out["sign_IC_OOS"])
    )

    # Si existe delta_oos_minus_is en el input, lo respetamos; si no, lo calculamos
    if "delta_oos_minus_is" not in out.columns:
        out["delta_oos_minus_is"] = out[oos_col] - out[is_col]

    # -------------------------
    # Orden recomendado:
    # 1) primero los incoherentes (para inspección)
    # 2) luego por fuerza OOS (abs_IC_OOS desc)
    # -------------------------
    out = out.sort_values(
        ["coherent_is_oos", abs_oos_col],
        ascending=[True, False]
    )

    # -------------------------
    # Columnas a devolver
    # -------------------------
    cols = [
        "indicator", "horizon",
        is_col, oos_col, "delta_oos_minus_is",
        "sign_IC_IS", "sign_IC_OOS",
        "coherent_is_oos", "sign_flip_is_oos",
        "abs_IC_IS", "abs_IC_OOS", "abs_gap_oos_minus_is",
        "n_pairs_IS", "n_pairs_OOS", "note",
    ]

    # Mantener solo columnas que existan (por robustez)
    cols = [c for c in cols if c in out.columns]

    # Si existe 'family', incluirla al inicio
    if "family" in out.columns:
        cols = ["family"] + cols

    return out[cols]


### **5.2. Aplicación**

In [114]:
ic_selected_60_full_day = ic_table_full_day[
    (ic_table_full_day["horizon"] == 60) &
    (ic_table_full_day["indicator"].isin(tech_indicators_finals))
].copy()

coherence_60_full_day = check_directional_coherence(ic_selected_60_full_day)

#################

ic_selected_90_full_day = ic_table_full_day[
    (ic_table_full_day["horizon"] == 90) &
    (ic_table_full_day["indicator"].isin(tech_indicators_finals))
].copy()

coherence_90_full_day = check_directional_coherence(ic_selected_90_full_day)


In [115]:
ic_selected_60_gestation = ic_table_gestation[
    (ic_table_gestation["horizon"] == 60) &
    (ic_table_gestation["indicator"].isin(tech_indicators_finals))
].copy()

coherence_60_gestation = check_directional_coherence(ic_selected_60_gestation)

##################

ic_selected_90_gestation = ic_table_gestation[
    (ic_table_gestation["horizon"] == 90) &
    (ic_table_gestation["indicator"].isin(tech_indicators_finals))
].copy()

coherence_90_gestation = check_directional_coherence(ic_selected_90_gestation)

In [116]:
ic_selected_60_execution = ic_table_execution[
    (ic_table_execution["horizon"] == 60) &
    (ic_table_execution["indicator"].isin(tech_indicators_finals))
].copy()

coherence_60_execution = check_directional_coherence(ic_selected_60_execution)

##################

ic_selected_90_execution = ic_table_execution[
    (ic_table_execution["horizon"] == 90) &
    (ic_table_execution["indicator"].isin(tech_indicators_finals))
].copy()

coherence_90_execution = check_directional_coherence(ic_selected_90_execution)

### **5.3. Resultados y análisis**

In [117]:
print('\ncoherence_60_full_day:\n')
display(coherence_60_full_day)

print('\ncoherence_90_full_day:\n')
display(coherence_90_full_day)


coherence_60_full_day:



,indicator,horizon,IC_IS,IC_OOS,delta_oos_minus_is,sign_IC_IS,sign_IC_OOS,coherent_is_oos,sign_flip_is_oos,abs_IC_IS,abs_IC_OOS,abs_gap_oos_minus_is,n_pairs_IS,n_pairs_OOS,note
0,ema_60,60,-0.193578,-0.189173,0.004405,-1.0,-1.0,True,False,0.193578,0.189173,-0.004405,281064,229712,
1,roc_60,60,-0.187156,-0.177858,0.009298,-1.0,-1.0,True,False,0.187156,0.177858,-0.009298,281064,229712,
6,roc_30,60,-0.135632,-0.135081,0.000551,-1.0,-1.0,True,False,0.135632,0.135081,-0.000551,281064,229712,
13,roc_20,60,-0.113311,-0.114449,-0.001138,-1.0,-1.0,True,False,0.113311,0.114449,0.001138,281064,229712,
21,atr_norm_20,60,0.121862,0.093256,-0.028606,1.0,1.0,True,False,0.121862,0.093256,-0.028606,281064,229712,
30,momentum_10,60,-0.085062,-0.084314,0.000748,-1.0,-1.0,True,False,0.085062,0.084314,-0.000748,281064,229712,
33,momentum_5,60,-0.063818,-0.062301,0.001517,-1.0,-1.0,True,False,0.063818,0.062301,-0.001517,281064,229712,
35,momentum_3,60,-0.052004,-0.049624,0.002380,-1.0,-1.0,True,False,0.052004,0.049624,-0.002380,281064,229712,



coherence_90_full_day:



,indicator,horizon,IC_IS,IC_OOS,delta_oos_minus_is,sign_IC_IS,sign_IC_OOS,coherent_is_oos,sign_flip_is_oos,abs_IC_IS,abs_IC_OOS,abs_gap_oos_minus_is,n_pairs_IS,n_pairs_OOS,note
42,ema_60,90,-0.245388,-0.238799,0.006590,-1.0,-1.0,True,False,0.245388,0.238799,-0.006590,281064,229712,
43,roc_60,90,-0.234883,-0.225245,0.009639,-1.0,-1.0,True,False,0.234883,0.225245,-0.009639,281064,229712,
48,roc_30,90,-0.175148,-0.170724,0.004424,-1.0,-1.0,True,False,0.175148,0.170724,-0.004424,281064,229712,
52,roc_20,90,-0.148450,-0.141795,0.006655,-1.0,-1.0,True,False,0.148450,0.141795,-0.006655,281064,229712,
63,atr_norm_20,90,0.123218,0.107646,-0.015572,1.0,1.0,True,False,0.123218,0.107646,-0.015572,281064,229712,
67,momentum_10,90,-0.110290,-0.105751,0.004540,-1.0,-1.0,True,False,0.110290,0.105751,-0.004540,281064,229712,
75,momentum_5,90,-0.081325,-0.079381,0.001944,-1.0,-1.0,True,False,0.081325,0.079381,-0.001944,281064,229712,
77,momentum_3,90,-0.065660,-0.064173,0.001487,-1.0,-1.0,True,False,0.065660,0.064173,-0.001487,281064,229712,


**Full day – H = 60**

- **100% coherencia direccional IS vs OOS**: ningún indicador cambia de signo.
- **Señal estable**: las diferencias entre IC_IS e IC_OOS son mínimas.
- **EMA, ROC y momentum** mantienen un comportamiento consistente fuera de muestra.
- **ATR_norm_20** conserva signo positivo, con una leve reducción de magnitud en OOS.

**Full day – H = 90**

- **Mismo patrón observado en H = 60**, con ICs de mayor magnitud.
- **Coherencia direccional perfecta IS–OOS** en todos los indicadores.
- La **degradación en OOS es leve y controlada**, sin inversión de señal.
- **Momentum y ROC** permanecen consistentes a un horizonte mayor.

**Conclusión Full day**

- Los indicadores analizados son **estructuralmente coherentes entre IS y OOS**.
- **No se observan señales espurias ni sobreajuste direccional** en la ventana full_day.
- Estos resultados **validan el uso de `abs_IC_OOS` como criterio principal**, utilizando la coherencia direccional como chequeo adicional de robustez.


In [118]:
print('\ncoherence_60_gestation:\n')
display(coherence_60_gestation)

print('\ncoherence_90_gestation:\n')
display(coherence_90_gestation)


coherence_60_gestation:



,indicator,horizon,IC_IS,IC_OOS,delta_oos_minus_is,sign_IC_IS,sign_IC_OOS,coherent_is_oos,sign_flip_is_oos,abs_IC_IS,abs_IC_OOS,abs_gap_oos_minus_is,n_pairs_IS,n_pairs_OOS,note
0,ema_60,60,-0.354846,-0.448972,-0.094126,-1.0,-1.0,True,False,0.354846,0.448972,0.094126,43737,35746,
2,roc_30,60,-0.308842,-0.395218,-0.086377,-1.0,-1.0,True,False,0.308842,0.395218,0.086377,43737,35746,
7,roc_60,60,-0.319006,-0.383793,-0.064787,-1.0,-1.0,True,False,0.319006,0.383793,0.064787,43737,35746,
9,roc_20,60,-0.277677,-0.373996,-0.096319,-1.0,-1.0,True,False,0.277677,0.373996,0.096319,43737,35746,
20,momentum_10,60,-0.221873,-0.294612,-0.072739,-1.0,-1.0,True,False,0.221873,0.294612,0.072739,43737,35746,
27,momentum_5,60,-0.174053,-0.218417,-0.044364,-1.0,-1.0,True,False,0.174053,0.218417,0.044364,43737,35746,
31,momentum_3,60,-0.140842,-0.168313,-0.027471,-1.0,-1.0,True,False,0.140842,0.168313,0.027471,43737,35746,
35,atr_norm_20,60,0.046362,0.025077,-0.021285,1.0,1.0,True,False,0.046362,0.025077,-0.021285,43737,35746,



coherence_90_gestation:



,indicator,horizon,IC_IS,IC_OOS,delta_oos_minus_is,sign_IC_IS,sign_IC_OOS,coherent_is_oos,sign_flip_is_oos,abs_IC_IS,abs_IC_OOS,abs_gap_oos_minus_is,n_pairs_IS,n_pairs_OOS,note
42,ema_60,90,-0.287640,-0.320655,-0.033015,-1.0,-1.0,True,False,0.287640,0.320655,0.033015,43737,35746,
44,roc_60,90,-0.255658,-0.285639,-0.029981,-1.0,-1.0,True,False,0.255658,0.285639,0.029981,43737,35746,
45,roc_30,90,-0.251744,-0.283501,-0.031757,-1.0,-1.0,True,False,0.251744,0.283501,0.031757,43737,35746,
51,roc_20,90,-0.226354,-0.262147,-0.035793,-1.0,-1.0,True,False,0.226354,0.262147,0.035793,43737,35746,
59,momentum_10,90,-0.180778,-0.211418,-0.030640,-1.0,-1.0,True,False,0.180778,0.211418,0.030640,43737,35746,
69,momentum_5,90,-0.148082,-0.163650,-0.015568,-1.0,-1.0,True,False,0.148082,0.163650,0.015568,43737,35746,
73,momentum_3,90,-0.118889,-0.130688,-0.011799,-1.0,-1.0,True,False,0.118889,0.130688,0.011799,43737,35746,
77,atr_norm_20,90,0.048632,0.023751,-0.024881,1.0,1.0,True,False,0.048632,0.023751,-0.024881,43737,35746,


**Ventana de gestación - H = 60**

- **Coherencia direccional total IS vs OOS**: ningún indicador presenta cambio de signo.
- **Refuerzo de señal en OOS** para EMA, ROC y momentum: las magnitudes absolutas aumentan fuera de muestra.
- **EMA_60 y ROC (20, 30, 60)** muestran las señales más fuertes y estables.
- **Momentum** mantiene coherencia y gana intensidad en OOS, especialmente `momentum_10`.
- **ATR_norm_20** conserva signo positivo, pero con **señal débil y decreciente**.

**Ventana de gestación - H = 90**

- **Patrón consistente con H = 60**, sin flips direccionales.
- **IC_OOS > IC_IS** en EMA, ROC y momentum, confirmando robustez fuera de muestra.
- Las diferencias IS-OOS son **moderadas y sistemáticas**, no erráticas.
- **Momentum y ROC** siguen siendo informativos incluso a mayor horizonte.
- **ATR_norm_20** vuelve a mostrar señal positiva pero marginal.

**Conclusión ventana de gestación**

- La ventana de gestación presenta **alta estabilidad direccional y fortalecimiento OOS**.
- No hay evidencia de sobreajuste; por el contrario, la señal **mejora fuera de muestra**.
- Esto valida que los indicadores capturan **estructura temprana del movimiento**, alineada con su rol conceptual.
- El análisis IS vs OOS resulta **especialmente valioso en esta ventana**, más informativo que un chequeo direccional clásico aislado.


In [119]:
print('\ncoherence_60_execution:\n')
display(coherence_60_execution)

print('\ncoherence_90_execution:\n')
display(coherence_90_execution)


coherence_60_execution:



,indicator,horizon,IC_IS,IC_OOS,delta_oos_minus_is,sign_IC_IS,sign_IC_OOS,coherent_is_oos,sign_flip_is_oos,abs_IC_IS,abs_IC_OOS,abs_gap_oos_minus_is,n_pairs_IS,n_pairs_OOS,note
0,ema_60,60,-0.594698,-0.556157,0.038542,-1.0,-1.0,True,False,0.594698,0.556157,-0.038542,43737,35746,
2,roc_30,60,-0.555345,-0.516905,0.038440,-1.0,-1.0,True,False,0.555345,0.516905,-0.038440,43737,35746,
7,roc_20,60,-0.522272,-0.491503,0.030770,-1.0,-1.0,True,False,0.522272,0.491503,-0.030770,43737,35746,
8,roc_60,60,-0.542577,-0.484769,0.057807,-1.0,-1.0,True,False,0.542577,0.484769,-0.057807,43737,35746,
16,momentum_10,60,-0.435144,-0.399036,0.036108,-1.0,-1.0,True,False,0.435144,0.399036,-0.036108,43737,35746,
28,momentum_5,60,-0.335210,-0.309230,0.025980,-1.0,-1.0,True,False,0.335210,0.309230,-0.025980,43737,35746,
31,momentum_3,60,-0.270437,-0.252229,0.018208,-1.0,-1.0,True,False,0.270437,0.252229,-0.018208,43737,35746,
40,atr_norm_20,60,-0.000562,-0.005303,-0.004741,-1.0,-1.0,True,False,0.000562,0.005303,0.004741,43737,35746,



coherence_90_execution:



,indicator,horizon,IC_IS,IC_OOS,delta_oos_minus_is,sign_IC_IS,sign_IC_OOS,coherent_is_oos,sign_flip_is_oos,abs_IC_IS,abs_IC_OOS,abs_gap_oos_minus_is,n_pairs_IS,n_pairs_OOS,note
42,ema_60,90,-0.621967,-0.597707,0.024260,-1.0,-1.0,True,False,0.621967,0.597707,-0.024260,43737,35746,
44,roc_30,90,-0.583346,-0.559240,0.024106,-1.0,-1.0,True,False,0.583346,0.559240,-0.024106,43737,35746,
49,roc_20,90,-0.543496,-0.526863,0.016633,-1.0,-1.0,True,False,0.543496,0.526863,-0.016633,43737,35746,
50,roc_60,90,-0.569866,-0.519405,0.050461,-1.0,-1.0,True,False,0.569866,0.519405,-0.050461,43737,35746,
57,momentum_10,90,-0.449058,-0.429184,0.019874,-1.0,-1.0,True,False,0.449058,0.429184,-0.019874,43737,35746,
70,momentum_5,90,-0.344929,-0.331712,0.013217,-1.0,-1.0,True,False,0.344929,0.331712,-0.013217,43737,35746,
73,momentum_3,90,-0.278032,-0.268738,0.009294,-1.0,-1.0,True,False,0.278032,0.268738,-0.009294,43737,35746,
83,atr_norm_20,90,0.014321,0.000785,-0.013536,1.0,1.0,True,False,0.014321,0.000785,-0.013536,43737,35746,


**Ventana de ejecución - H = 60**

- **Alta coherencia direccional** en casi todos los indicadores (EMA, ROC, Momentum).
- **EMA_60, ROC y Momentum** mantienen señal fuerte, aunque **ligeramente menor en OOS** → caída esperable por mayor dificultad del tramo.
- **No hay sobreajuste**: la reducción IS → OOS es gradual y consistente.
- **ATR_norm_20** presenta **flip de signo** y señal prácticamente nula → indicador **no fiable** en ejecución.

**Ventana de ejecución - H = 90**

- **Coherencia direccional total**: ningún indicador principal cambia de signo.
- **EMA_60 y ROC_60** siguen siendo las señales más robustas.
- **Momentum** conserva direccionalidad, con pérdida moderada de magnitud en OOS.
- **ATR_norm_20** mantiene signo, pero con **IC muy bajo** → aporte marginal.

**Conclusión general (ejecución)**

- La ventana de ejecución muestra **señales fuertes pero naturalmente más débiles en OOS**.
- El patrón IS vs OOS es **estable y controlado**, típico de un régimen más ruidoso.
- **EMA_60, ROC y Momentum** son consistentes y confiables para ejecución.
- **ATR_norm** no aporta valor operativo claro en esta ventana.
- El análisis IS vs OOS confirma **robustez sin señales de leakage ni inestabilidad**.




### **5.4. Conclusión de coherencia direccional – Full Day, Gestation y Execution**


- Coherencia direccional consistente: en las tres ventanas, los indicadores estructurales (EMA, ROC y Momentum) mantienen el mismo signo entre IS y OOS, lo que valida estabilidad y consistencia fuera de muestra.

- Degradación esperable IS → OOS: la magnitud del IC disminuye levemente en OOS, sin flips sistemáticos, indicando ausencia de sobreajuste y buen poder de generalización.

- Gestation = señal más limpia: es la ventana con mayor IC absoluto y mejor preservación OOS, lo que la posiciona como la ventana principal para generación de señal.

- Execution = señal más ruidosa pero válida: los IC son más bajos, pero direccionalmente coherentes, adecuados para timing y confirmación operativa.

- Full Day = comportamiento intermedio: combina estructura y ruido; es útil como referencia global, pero no como ventana operativa principal.

- ATR_norm_20 no es robusto: presenta señal débil, degradación OOS y, en ejecución (H=60), flip direccional IS → OOS, lo que lo vuelve no confiable desde el punto de vista operativo.


### **5.5. Decisiones de selección y descarte**


- ATR_norm_20 se descarta del set final de features por falta de estabilidad y coherencia entre ventanas.

- No se descarta ningún otro feature en esta etapa:

  - EMA, ROC y Momentum muestran consistencia direccional, magnitudes relevantes y estabilidad IS/OOS.

  - La resolución por familias y el análisis de correlación ya controlaron redundancia sin forzar eliminaciones innecesarias.

### **5.6. Síntesis operativa**


- EMA_60 + ROC + Momentum conforman un núcleo robusto, estable y coherente en todas las ventanas.

- El enfoque IS vs OOS + coherencia direccional valida que el set final es interpretable, generalizable y alineado con la dinámica del mercado.

- El pipeline queda correctamente preparado para modelado predictivo sin leakage y con control explícito de generalización.

In [120]:
tech_indicators_finals = [
    'ema_60',
    'momentum_10',
    'momentum_3',
    'momentum_5',
    'roc_20',
    'roc_30',
    'roc_60'
    ]

In [122]:
targets = ['ret_60', 'ret_90']


## **7. Análisis de OHLCV**

Las variables OHLCV (`open`, `high`, `low`, `close`, `volume`) representan el estado instantáneo del mercado y constituyen la materia prima a partir de la cual se construyen los indicadores técnicos.

Aunque no están diseñadas explícitamente como señales anticipatorias, su inclusión directa en el modelo puede:

- aportar información contextual relevante,
- introducir redundancia respecto a los indicadores derivados,
- o, en algunos casos, mostrar capacidad predictiva directa.

Por este motivo, se realiza un análisis específico de Information Coefficient (IC) y correlación para las variables OHLCV, con el objetivo de evaluar su contribución real y justificar su inclusión en el modelo.

### **6.1. IC de variables OHLCV vs targets**


Se evalúa la relación entre OHLCV y los targets definidos (`delta_60` y `delta_90`), utilizando el mismo criterio metodológico aplicado a los indicadores técnicos.


In [123]:
mnq_intraday_with_indicators.head()

,date,open,high,low,close,volume,ret_60,ret_90,split_fe,minute_of_day,...,atr_norm_5,atr_norm_10,atr_norm_14,atr_norm_20,atr_norm_30,roc_5,roc_10,roc_20,roc_30,roc_60
datetime,,,,,,,,,,,,,,,,,,,,,
2019-12-23 07:59:00-05:00,2019-12-23,8734.00,8734.25,8734.00,8734.00,47,0.000487,0.000029,IS,479,...,0.000093,0.000095,0.000097,0.000101,0.000106,-0.008586,0.011451,0.000000,-0.022894,0.077344
2019-12-23 08:00:00-05:00,2019-12-23,8734.00,8734.25,8733.75,8733.75,31,0.000401,-0.000057,IS,480,...,0.000086,0.000091,0.000094,0.000099,0.000105,-0.017172,0.011451,0.000000,-0.034338,0.065880
2019-12-23 08:01:00-05:00,2019-12-23,8734.00,8734.25,8733.75,8734.00,16,0.000458,-0.000515,IS,481,...,0.000080,0.000087,0.000091,0.000097,0.000103,-0.011448,0.011451,0.005725,-0.020033,0.080211
2019-12-23 08:02:00-05:00,2019-12-23,8734.00,8734.00,8733.25,8733.25,23,0.000544,-0.000830,IS,482,...,0.000081,0.000087,0.000091,0.000096,0.000103,-0.020034,-0.002863,-0.002863,-0.020034,0.060151
2019-12-23 08:03:00-05:00,2019-12-23,8734.25,8734.50,8734.00,8734.00,23,0.000429,-0.000630,IS,483,...,0.000094,0.000093,0.000095,0.000099,0.000104,0.000000,-0.017171,0.014314,-0.005724,0.065878


In [124]:
"""
    Calcula una tabla resumen de Information Coefficient (IC) para cada
    indicador técnico y cada horizonte temporal (60 / 90).

    Para cada indicador y horizonte se evalúa su relación con:
    - ret_h        → magnitud del movimiento futuro (continuo)
    """


'\n    Calcula una tabla resumen de Information Coefficient (IC) para cada\n    indicador técnico y cada horizonte temporal (60 / 90).\n\n    Para cada indicador y horizonte se evalúa su relación con:\n    - delta_h        → magnitud del movimiento futuro (continuo)\n    '

In [125]:
ohlcv_cols = ["open", "high", "low", "close", "volume"]


In [126]:
# ============================================================
# IC tables IS vs OOS para OHLCV por ventana (full day / gestation / execution)
# ============================================================

# --- Full day ---
ic_table_ohlcv_full_day = compute_ic_table_is_oos_delta(
    df=mnq_intraday_with_indicators,
    indicator_columns=ohlcv_cols,
    horizons=(60, 90),
    window_start=start_time_full_day,
    window_end=final_time_full_day,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

# --- Gestation window ---
ic_table_ohlcv_gestation = compute_ic_table_is_oos_delta(
    df=mnq_intraday_with_indicators,
    indicator_columns=ohlcv_cols,
    horizons=(60, 90),
    window_start=start_time_gestation_window,
    window_end=final_time_gestation_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

# --- Execution window ---
ic_table_ohlcv_execution = compute_ic_table_is_oos_delta(
    df=mnq_intraday_with_indicators,
    indicator_columns=ohlcv_cols,
    horizons=(60, 90),
    window_start=start_time_execution_window,
    window_end=final_time_execution_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

print("OK -> ic_table_ohlcv_full_day:", ic_table_ohlcv_full_day.shape)
print("OK -> ic_table_ohlcv_gestation:", ic_table_ohlcv_gestation.shape)
print("OK -> ic_table_ohlcv_execution:", ic_table_ohlcv_execution.shape)
#70seg

OK -> ic_table_ohlcv_full_day: (10, 9)
OK -> ic_table_ohlcv_gestation: (10, 9)
OK -> ic_table_ohlcv_execution: (10, 9)


In [127]:
print("\nic_table_ohlcv_full_day:\n")
display(ic_table_ohlcv_full_day)

print("\nic_table_ohlcv_gestation:\n")
display(ic_table_ohlcv_gestation)

print("\nic_table_ohlcv_execution:\n")
display(ic_table_ohlcv_execution)


ic_table_ohlcv_full_day:



,indicator,horizon,IC_IS,IC_OOS,ret_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,close,60,-0.518605,-0.509525,0.009080,281064,229712,,0.509525
1,high,60,-0.514695,-0.506304,0.008391,281064,229712,,0.506304
2,low,60,-0.514315,-0.504428,0.009886,281064,229712,,0.504428
3,open,60,-0.509250,-0.499916,0.009334,281064,229712,,0.499916
4,volume,60,0.071537,0.039715,-0.031822,281064,229712,,0.039715
5,close,90,-0.596851,-0.600508,-0.003657,281064,229712,,0.600508
6,high,90,-0.592396,-0.596735,-0.004339,281064,229712,,0.596735
7,low,90,-0.592414,-0.594956,-0.002542,281064,229712,,0.594956
8,open,90,-0.586712,-0.589899,-0.003187,281064,229712,,0.589899
9,volume,90,0.071848,0.045548,-0.026300,281064,229712,,0.045548



ic_table_ohlcv_gestation:



,indicator,horizon,IC_IS,IC_OOS,ret_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,close,60,-0.397561,-0.471432,-0.073871,43737,35746,,0.471432
1,high,60,-0.381440,-0.449948,-0.068507,43737,35746,,0.449948
2,low,60,-0.380435,-0.449394,-0.068959,43737,35746,,0.449394
3,open,60,-0.358193,-0.425301,-0.067108,43737,35746,,0.425301
4,volume,60,0.015643,0.012472,-0.003170,43737,35746,,0.012472
5,close,90,-0.331484,-0.343706,-0.012221,43737,35746,,0.343706
6,high,90,-0.315930,-0.327802,-0.011872,43737,35746,,0.327802
7,low,90,-0.317793,-0.326779,-0.008985,43737,35746,,0.326779
8,open,90,-0.298809,-0.308432,-0.009623,43737,35746,,0.308432
9,volume,90,0.021238,0.003749,-0.017489,43737,35746,,0.003749



ic_table_ohlcv_execution:



,indicator,horizon,IC_IS,IC_OOS,ret_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,close,60,-0.623896,-0.592212,0.031684,43737,35746,,0.592212
1,low,60,-0.598286,-0.569275,0.029011,43737,35746,,0.569275
2,high,60,-0.590522,-0.559385,0.031138,43737,35746,,0.559385
3,open,60,-0.564057,-0.533320,0.030738,43737,35746,,0.533320
4,volume,60,0.011643,0.015527,0.003884,43737,35746,,0.015527
5,close,90,-0.669524,-0.657220,0.012304,43737,35746,,0.657220
6,low,90,-0.642960,-0.631506,0.011454,43737,35746,,0.631506
7,high,90,-0.635277,-0.623447,0.011830,43737,35746,,0.623447
8,open,90,-0.607650,-0.594453,0.013197,43737,35746,,0.594453
9,volume,90,0.012111,0.004820,-0.007291,43737,35746,,0.004820


#### **6.1.1. Conclusiones – Análisis OHLCV (IS vs OOS)**

1) Precio domina claramente  
    - **Open, High, Low y Close** presentan **IC altos y estables** en las tres ventanas y en ambos horizontes.  
    - La **dirección es consistente (negativa)** y se mantiene de IS a OOS, sin flips.

2) Execution es la ventana más informativa  
    - En **Execution**, los IC absolutos de precio son los **más altos** (≈0.53–0.66 en OOS).  
    - Indica que el precio en esta ventana tiene **máximo poder explicativo directo** sobre el retorno futuro.

3) Gestation aporta señal, pero más débil  
    - En **Gestation**, el IC de precio sigue siendo significativo, pero **menor que en Execution**.  
    - Confirma que la señal se fortalece al acercarse la ventana operativa.

4) Full Day refleja comportamiento promedio  
    - En **Full Day**, los IC son altos y muy estables, pero **mezclan estructura y ruido**.  
    - Útil como referencia global, no como ventana operativa principal.

5) Volume no aporta señal predictiva  
    - **Volume muestra IC muy bajos** en todas las ventanas y horizontes.  
    - La señal es débil, inestable y cercana a cero en OOS.

**Síntesis final**

- **OHLC (precio)** es un bloque **robusto, coherente y estable** en IS y OOS.  
- **Volume no justifica su inclusión** como feature predictiva principal.  
- El resultado valida que el pipeline captura **estructura real del mercado**, no artefactos de entrenamiento.


#### **6.1.2. Interpretación metodológica**

Aunque las variables OHLC presentan valores de IC elevados, este resultado
**no invalida** el rol central de los indicadores técnicos. Por el contrario,
confirma que:

- Los indicadores técnicos capturan y **reexpresan información ya contenida
  en el precio**, de forma más estructurada y filtrada.
- Las variables OHLC actúan como **variables de estado del mercado**, no como
  señales diseñadas explícitamente para anticipar movimientos.

En este contexto, el IC elevado de OHLC es esperable y no implica que deban
sustituir a los indicadores técnicos, sino que:

- **Refuerzan la consistencia del análisis**, al confirmar el régimen dominante.
- Aportan **contexto estructural** útil para el modelo.

#### **6.1.3. Decisión de diseño del modelo**


En base a este análisis, se adopta el siguiente criterio:

- Las variables **OHLC** se mantienen como **features base**, normalizadas o
  transformadas según corresponda.
- Los **indicadores técnicos seleccionados** constituyen el **núcleo predictivo**
  del modelo.
- La variable **`volume` no se utiliza como feature principal** en esta etapa,
  pudiendo reservarse para análisis contextuales futuros.

Esta decisión permite equilibrar **información estructural**, **capacidad
predictiva real** e **interpretabilidad económica**, evitando tanto redundancia
innecesaria como exclusiones arbitrarias.

In [129]:
ohlc_cols = ['open', 'high', 'low', 'close']

### **6.2. Correlación entre OHLC**


Antes de profundizar en el análisis predictivo, se realiza un chequeo estructural de redundancia entre las variables de precio básicas (Open, High, Low, Close). Este análisis no tiene como objetivo evaluar capacidad predictiva, ni reemplazar el estudio de Information Coefficient (IS vs OOS), sino verificar el grado de colinealidad intrínseca entre las componentes OHLC, que representan distintas observaciones de una misma variable latente: el precio.

Dado que el pipeline ya define la relevancia de las variables mediante IC IS/OOS y coherencia direccional, el análisis de correlación OHLC se utiliza únicamente como sanity check, con el fin de:

- confirmar la redundancia extrema entre las variables de precio,
- evitar la duplicación innecesaria de información,
- y justificar decisiones de representación posteriores (por ejemplo, uso exclusivo de close, uso de agregados como ohlc4, o mantenimiento explícito de OHLC por motivos de estabilidad numérica o interpretabilidad).

En este marco, la elección de trabajar únicamente con close constituye una decisión de representación y parsimonia, basada en redundancia estadística, no una selección de variables por desempeño predictivo. La correlación OHLC, por lo tanto, complementa el análisis, pero no interviene en el ranking ni en la selección predictiva de features.

In [130]:
corr_ohlc = (
    mnq_intraday_with_indicators[
        ohlc_cols
    ]
    .corr(method="spearman")
    .loc[ohlc_cols]
)

corr_ohlc

,open,high,low,close
open,1.000000,0.999998,0.999998,0.999996
high,0.999998,1.000000,0.999996,0.999998
low,0.999998,0.999996,1.000000,0.999998
close,0.999996,0.999998,0.999998,1.000000


- Redundancia extrema OHLC: el análisis de correlación muestra que open, high, low y close están prácticamente perfectamente correlacionadas (ρ ≈ 0.999). Mantenerlas simultáneamente no agrega información nueva.

- Señal ya validada por IC IS/OOS: `close` presenta IC alto, estable y coherente entre IS y OOS en las tres ventanas (Full Day, Gestation y Execution). No existe evidencia de que open, high o low aporten poder predictivo incremental.

- Principio de parsimonia: eliminar variables redundantes reduce dimensionalidad, colinealidad y complejidad del modelo, mejorando estabilidad y generalización.

- Consistencia conceptual: `close` representa el precio de consenso del mercado al final de cada barra y es la base de la mayoría de indicadores técnicos utilizados (EMA, ROC, Momentum), manteniendo coherencia semántica en todo el pipeline.

- No es una selección “por descarte arbitrario”: la decisión se apoya en evidencia empírica (IC, coherencia direccional, correlación) y no compromete la información relevante.

**Decisión final:**

- Se utiliza close como única variable de precio, descartando open, high y low.
- Esta decisión responde a redundancia estadística, estabilidad IS/OOS y parsimonia, no a limitaciones del modelo

### **6.3. Correlación entre `close` e indicadores técnicos seleccionados**


El objetivo de este paso es evaluar redundancia informativa, es decir, determinar si las variables OHLCV ya están implícitamente capturadas por los indicadores técnicos seleccionados.

In [131]:
corr_ohlc_vs_indicators = (
    mnq_intraday_with_indicators[
        ["close"] + tech_indicators_finals
    ]
    .corr(method="spearman")
    .loc[["close"], tech_indicators_finals]
)

corr_ohlc_vs_indicators

,ema_60,momentum_10,momentum_3,momentum_5,roc_20,roc_30,roc_60
close,-0.010662,-0.004462,-0.00181,-0.002915,-0.006856,-0.007442,-0.007388


#### **6.2.1. Análisis de Correlación entre features `close` e indicadores técnicos seleccionados**

- En concreto, la correlación es prácticamente nula entre `close` y los indicadores técnicos (|ρ| < 0.01 en todos los casos).
- Esto indica que los indicadores no son transformaciones lineales triviales del precio, sino que capturan dinámica, cambios relativos y estructura temporal.
- No hay colinealidad problemática entre close y EMA / ROC / Momentum.
- Se Refuerza que close puede convivir con los indicadores sin redundancia informativa.

**Conclusión breve**

`close` aporta el nivel de precio, mientras que los indicadores aportan estructura y dinámica. La combinación es sana, estable y bien condicionada para el modelado.

#### **6.2.2. Conclusión metodológica**


El conjunto final de variables queda compuesto por la variable de precio `close` y un set reducido de indicadores técnicos derivados: `ema_60`, `momentum_10`, `momentum_5`, `momentum_3`, `roc_20`, `roc_30` y `roc_60`

Esta selección es el resultado de un proceso sistemático que combinó análisis de Information Coefficient (IS vs OOS), coherencia direccional, colinealidad y redundancia. El set final preserva información complementaria —nivel de precio y dinámica temporal— evitando redundancias estructurales y asegurando estabilidad fuera de muestra.

## **8. Machine Learning for Algorithmic Trading**

### **8.1. Convertir indicadores en features estadísticamente estables**

Hasta este punto, los indicadores técnicos han sido utilizados como series crudas, es decir, en su escala original y sin normalización contextual.
El siguiente nivel del pipeline consiste en transformarlos en features estadísticamente estables, incorporando el contexto intradía del mercado.

El objetivo no es crear nuevos indicadores, sino reexpresar los existentes de forma que su significado sea comparable entre jornadas y más robusto fuera de muestra.

---

**Normalización en contexto intradía**

Las transformaciones consideradas incluyen:

- Z-score rolling por día
  
  Normaliza cada indicador respecto a su media y desviación estándar intradía.

- Percentil intradía

  Expresa la posición relativa del indicador dentro de su distribución diaria.

- Distancia al régimen típico del día

  Mide cuán alejado está el valor actual del comportamiento intradía más frecuente.

---

**Intuición clave**

El valor informativo de un indicador no reside en su magnitud absoluta, sino en su rareza relativa dentro del día.

Ejemplo conceptual:

- `roc_30` no aporta señal por su valor bruto.
- Aporta señal por qué tan extremo es respecto a su distribución intradía.

Este enfoque permite:

- Reducir sensibilidad a escalas y cambios de volatilidad.
- Mejorar la estabilidad del Information Coefficient out-of-sample.
- Mantener intacta la lógica económica de los indicadores originales.

In [134]:
base_cols = ['date',	'minute_of_day', 'split_fe']
price_features = ['close']
technical_indicators_features = tech_indicators_finals.copy()
targets

print(f'base_cols: {base_cols}')
print(f'price_features: {price_features}')
print(f'technical_indicators_features: {technical_indicators_features}')
print(f'targets: {targets}')


base_cols: ['date', 'minute_of_day', 'split_fe']
price_features: ['close']
technical_indicators_features: ['ema_60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60']
targets: ['ret_60', 'ret_90']


In [135]:
import numpy as np
import pandas as pd

# ============================================================
# Configuración (usa tus listas ya definidas)
# ============================================================
# price_features = ['close']
# technical_indicators_features = ['ema_60',  'momentum_10',  'momentum_3',  'momentum_5',  'roc_20', 'roc_30',  'roc_60']
# targets = ['delta_60', 'delta_90']

EPS = 1e-12  # Para evitar división por cero en la std


# ============================================================
# Función: Z-score "point-in-time" por día (sin fuga)
# ============================================================
def add_expanding_zscore_by_day(
    df: pd.DataFrame,
    cols: list[str],
    date_col: str = "date",
    min_periods: int = 10,
) -> pd.DataFrame:
    """
    Crea features normalizadas por día usando estadísticos "hasta el momento".

    Para cada día (date_col), y para cada minuto t dentro de ese día:
        z(t) = (x(t) - mean(x[<=t])) / std(x[<=t])

    Ventajas:
    - No mezcla días (cada día se normaliza por separado)
    - No usa información del futuro (solo datos hasta t)
    - Suele mejorar estabilidad OOS cuando hay cambios de régimen/volatilidad

    Parámetros:
    - cols: columnas a normalizar
    - date_col: columna que identifica el día (por ejemplo, 'date')
    - min_periods: mínimos puntos del día para empezar a calcular mean/std
                  (antes de eso devuelve NaN en el z-score)
    """
    out = df.copy()

    # Validaciones mínimas
    if not isinstance(out.index, pd.DatetimeIndex):
        raise TypeError("El índice debe ser DatetimeIndex (datetime).")
    if date_col not in out.columns:
        raise KeyError(f"Falta la columna '{date_col}' para agrupar por día.")

    # Asegura orden temporal global (por seguridad)
    out = out.sort_index()

    # Agrupa por día (no reordena los grupos)
    g = out.groupby(date_col, sort=False)

    # Calcula z-score expanding por día para cada feature
    for c in cols:
        exp_mean = (
            g[c]
            .expanding(min_periods=min_periods)
            .mean()
            .reset_index(level=0, drop=True)
        )
        exp_std = (
            g[c]
            .expanding(min_periods=min_periods)
            .std(ddof=0)
            .reset_index(level=0, drop=True)
        )

        # Feature normalizada
        out[f"{c}_z_exp"] = (out[c] - exp_mean) / (exp_std + EPS)

    return out



In [136]:
final_columns = (
    base_cols
    + price_features
    + technical_indicators_features
    + targets
)

# ============================================================
# Sanity check de columnas
# ============================================================
missing_cols = [c for c in final_columns if c not in mnq_intraday_with_indicators.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas requeridas en el dataset original: {missing_cols}")

# ============================================================
# Construcción del dataset final
# ============================================================
mnq_features_targets = (
    mnq_intraday_with_indicators[final_columns]
    .copy()
)

# Opcional: orden explícito
mnq_features_targets = mnq_features_targets[final_columns]

In [137]:
mnq_features_targets.head()

,date,minute_of_day,split_fe,close,ema_60,momentum_10,momentum_3,momentum_5,roc_20,roc_30,roc_60,ret_60,ret_90
datetime,,,,,,,,,,,,,
2019-12-23 07:59:00-05:00,2019-12-23,479,IS,8734.00,0.000076,0.000115,-0.000114,-0.000086,0.000000,-0.022894,0.077344,0.000487,0.000029
2019-12-23 08:00:00-05:00,2019-12-23,480,IS,8733.75,0.000046,0.000115,-0.000143,-0.000172,0.000000,-0.034338,0.065880,0.000401,-0.000057
2019-12-23 08:01:00-05:00,2019-12-23,481,IS,8734.00,0.000072,0.000115,0.000000,-0.000114,0.005725,-0.020033,0.080211,0.000458,-0.000515
2019-12-23 08:02:00-05:00,2019-12-23,482,IS,8733.25,-0.000013,-0.000029,-0.000086,-0.000200,-0.002863,-0.020034,0.060151,0.000544,-0.000830
2019-12-23 08:03:00-05:00,2019-12-23,483,IS,8734.00,0.000070,-0.000172,0.000029,0.000000,0.014314,-0.005724,0.065878,0.000429,-0.000630


In [138]:
# ============================================================
# Pipeline: crea raw + z_exp para tus indicadores finales
# ============================================================
df = mnq_intraday_with_indicators.copy()

# 1) Nos quedamos con las columnas necesarias (features + targets + date)
#    (esto evita arrastrar columnas que no usaremos)
keep_cols = base_cols + price_features + technical_indicators_features + targets
df = df.loc[:, keep_cols].copy()
features_cols = price_features + technical_indicators_features

# 2) Creamos columnas *_raw explícitas (para comparar raw vs normalizado)
for c in features_cols:
    df[f"{c}_raw"] = df[c]

# 3) Creamos columnas normalizadas *_z_exp (expanding z-score por día)
df = add_expanding_zscore_by_day(
    df,
    cols=features_cols,
    date_col="date",
    min_periods=10,  # Ajustable: 5–15 suele ser razonable intradía
)

# 4) Armamos dataset final (raw + z_exp + targets)
final_features = (
    [f"{c}_raw" for c in features_cols] +
    [f"{c}_z_exp" for c in features_cols]
)

final_cols = base_cols + final_features + [c for c in targets if c in df.columns]

# 5) Eliminamos filas donde aún no hay z-score (primeros min_periods-1 minutos de cada día)
mnq_features_targets_norm = df.loc[:, final_cols].dropna()

#print(mnq_features_targets_norm.head(3))


In [139]:
import numpy as np
import pandas as pd

def compute_ic_is_oos_by_split(
    df: pd.DataFrame,
    features: list[str],
    target: str,
    *,
    split_col: str = "split_fe",
    is_label: str = "is",
    oos_label: str = "oos",
    method: str = "spearman",
    min_obs: int = 200,
) -> pd.DataFrame:
    """
    Calcula IC (feature vs target) por separado en IS y OOS usando split_col.

    Retorna un DataFrame con:
      feature, n_is, ic_is, n_oos, ic_oos, delta_oos_minus_is
    """

    data = df.copy()

    # Checks básicos
    needed_cols = [split_col, target] + features
    missing = [c for c in needed_cols if c not in data.columns]
    if missing:
        raise KeyError(f"Faltan columnas requeridas: {missing}")

    # Normalizamos split a string lowercase para robustez
    split = data[split_col].astype(str).str.lower()

    mask_is = split == str(is_label).lower()
    mask_oos = split == str(oos_label).lower()

    df_is = data.loc[mask_is, :]
    df_oos = data.loc[mask_oos, :]

    rows = []
    for f in features:
        # --- IS ---
        tmp_is = df_is[[f, target]].dropna()
        n_is = len(tmp_is)
        ic_is = np.nan
        if n_is >= min_obs:
            ic_is = tmp_is[f].corr(tmp_is[target], method=method)

        # --- OOS ---
        tmp_oos = df_oos[[f, target]].dropna()
        n_oos = len(tmp_oos)
        ic_oos = np.nan
        if n_oos >= min_obs:
            ic_oos = tmp_oos[f].corr(tmp_oos[target], method=method)

        rows.append({
            "feature": f,
            "n_is": n_is,
            "ic_is": ic_is,
            "n_oos": n_oos,
            "ic_oos": ic_oos,
            "delta_oos_minus_is": (ic_oos - ic_is) if (pd.notna(ic_oos) and pd.notna(ic_is)) else np.nan,
        })

    out = (
        pd.DataFrame(rows)
        .sort_values(by="ic_oos", ascending=False)
        .reset_index(drop=True)
    )
    return out


In [140]:
features_raw = [
    "close_raw", "ema_60_raw",
    "momentum_10_raw", "momentum_3_raw", "momentum_5_raw",
    "roc_20_raw", "roc_30_raw", "roc_60_raw",
]

features_z = [
    "close_z_exp", "ema_60_z_exp",
    "momentum_10_z_exp", "momentum_3_z_exp", "momentum_5_z_exp",
    "roc_20_z_exp", "roc_30_z_exp", "roc_60_z_exp",
]



In [142]:
ic_60_raw = compute_ic_is_oos_by_split(
    df=mnq_features_targets_norm,
    features=features_raw,
    target="ret_60",
    split_col="split_fe",
    is_label="is",
    oos_label="oos",
)

ic_60_z = compute_ic_is_oos_by_split(
    df=mnq_features_targets_norm,
    features=features_z,
    target="ret_60",
    split_col="split_fe",
    is_label="is",
    oos_label="oos",
)

In [143]:
ic_90_raw = compute_ic_is_oos_by_split(
    df=mnq_features_targets_norm,
    features=features_raw,
    target="ret_90",
    split_col="split_fe",
    is_label="is",
    oos_label="oos",
)

ic_90_z = compute_ic_is_oos_by_split(
    df=mnq_features_targets_norm,
    features=features_z,
    target="ret_90",
    split_col="split_fe",
    is_label="is",
    oos_label="oos",
)

In [144]:
print('\nic_60_raw:\n')
display(ic_60_raw)

print('\nic_60_z:\n')
display(ic_60_z)

print('\nic_90_raw:\n')
display(ic_90_raw)

print('\nic_90_z:\n')
display(ic_90_z)




ic_60_raw:



,feature,n_is,ic_is,n_oos,ic_oos,delta_oos_minus_is
0,roc_60_raw,274611,0.006143,224438,0.032794,0.026651
1,ema_60_raw,274611,0.011230,224438,0.030592,0.019362
2,roc_30_raw,274611,0.008698,224438,0.020319,0.011621
3,roc_20_raw,274611,0.010358,224438,0.016830,0.006472
4,momentum_10_raw,274611,0.008395,224438,0.015515,0.007119
5,momentum_5_raw,274611,0.005174,224438,0.011353,0.006178
6,momentum_3_raw,274611,0.002538,224438,0.008055,0.005517
7,close_raw,274611,-0.042849,224438,-0.044036,-0.001187



ic_60_z:



,feature,n_is,ic_is,n_oos,ic_oos,delta_oos_minus_is
0,close_z_exp,274611,0.024172,224438,0.019992,-0.004180
1,roc_60_z_exp,274611,0.003252,224438,0.013661,0.010408
2,ema_60_z_exp,274611,0.005903,224438,0.010075,0.004172
3,momentum_10_z_exp,274611,0.001725,224438,0.004816,0.003091
4,roc_30_z_exp,274611,0.004761,224438,0.004212,-0.000549
5,momentum_5_z_exp,274611,0.000339,224438,0.003000,0.002661
6,roc_20_z_exp,274611,0.004045,224438,0.002263,-0.001782
7,momentum_3_z_exp,274611,-0.000678,224438,0.001996,0.002674



ic_90_raw:



,feature,n_is,ic_is,n_oos,ic_oos,delta_oos_minus_is
0,roc_60_raw,274611,0.010268,224438,0.031032,0.020764
1,ema_60_raw,274611,0.012279,224438,0.030305,0.018025
2,roc_30_raw,274611,0.010572,224438,0.023590,0.013018
3,roc_20_raw,274611,0.007880,224438,0.020011,0.012131
4,momentum_10_raw,274611,0.006854,224438,0.014364,0.007510
5,momentum_5_raw,274611,0.003967,224438,0.009184,0.005218
6,momentum_3_raw,274611,0.001734,224438,0.006043,0.004309
7,close_raw,274611,-0.051041,224438,-0.053863,-0.002822



ic_90_z:



,feature,n_is,ic_is,n_oos,ic_oos,delta_oos_minus_is
0,close_z_exp,274611,0.027693,224438,0.025650,-0.002043
1,roc_60_z_exp,274611,0.002519,224438,0.015150,0.012630
2,ema_60_z_exp,274611,0.002709,224438,0.010296,0.007586
3,roc_30_z_exp,274611,0.003060,224438,0.007037,0.003977
4,roc_20_z_exp,274611,0.001276,224438,0.005077,0.003801
5,momentum_10_z_exp,274611,-0.000450,224438,0.004310,0.004761
6,momentum_5_z_exp,274611,-0.001647,224438,0.000813,0.002460
7,momentum_3_z_exp,274611,-0.002384,224438,-0.000791,0.001594


#### **8.1.1. Conclusiones - IC IS vs OOS (raw vs z-exp)**

**1. Los indicadores **raw** dominan en señal predictiva**

- Para **H = 60 y H = 90**, las versiones **raw** (ROC, EMA, Momentum) presentan:
  - **IC OOS más altos**
  - **Δ(OOS–IS) positivo y consistente**
- Esto indica que **la señal principal reside en el valor crudo**, no en la estandarización actual.

---

**2. La normalización `z_exp` no mejora el IC OOS**

- En la mayoría de los casos:
  - El IC OOS **disminuye** o queda cercano a cero.
  - Algunos features **pierden señal** (Δ negativo).
- Conclusión clara:  
  **la normalización aplicada no aporta valor predictivo adicional** en esta etapa.

---

**3. ROC y EMA son los features más estables**

- **roc_60_raw** y **ema_60_raw**:
  - Lideran el ranking tanto en **H = 60** como en **H = 90**.
  - Muestran **mejor desempeño en OOS que en IS**.
- Confirma que **tendencia + momentum estructural** constituyen el núcleo informativo.

---

**4. Momentum corto aporta señal incremental**

- `momentum_10_raw`, `momentum_5_raw`, `momentum_3_raw`:
  - IC inferior a ROC/EMA.
  - Señal **consistente y positiva en OOS**.
- Son útiles como **features complementarias**, no principales.

---
**5. `close_raw` es informativo, pero de otra naturaleza**

- Presenta **IC negativo estable** en IS y OOS.
- No compite con los indicadores derivados, pero:
  - Aporta **contexto direccional base** del precio.
- Justifica su inclusión como **feature estructural**, no como predictor directo.

---

**Conclusión operativa**

- **Mantener los features raw** como base de entrada al modelo.
- **Descartar por ahora** las versiones `*_z_exp`.
- El siguiente paso no es normalizar más, sino:
  **contextualizar estadísticamente intradía**  
  (percentiles intradía, z-score por día, desviación respecto al régimen).


### **8.2. Introducir interacciones mínimas**


Hasta ahora, los indicadores se utilizaron como **features individuales**.  
El siguiente paso metodológico consiste en introducir **interacciones simples y controladas**, cuyo objetivo es:

- capturar **relaciones estructurales** (tendencia + velocidad),
- **sin aumentar complejidad innecesaria**,
- manteniendo interpretabilidad y estabilidad OOS.

Estas interacciones **no crean nueva información**, sino que **reexpresan la existente en contexto**.

### **8.2.1. Interacciones recomendadas por jornada completa y por ventanas**

**1. Interacciones válidas para todo el día (Full Day)**

- Indicadores disponibles:
  - `ema_60`
  - `roc_60`

- Interacciones propuestas

  - **`ema_slope_1 = ema_60 - ema_60_lag1`**  
    *Justificación*: la EMA no aporta por su nivel, sino por su **dirección inmediata** (pendiente).

  - **`roc60_x_emaSlope = roc_60 * ema_slope_1`**  
    *Justificación*: combina **dirección de tendencia** con **velocidad del movimiento**.  
    Penaliza momentum contra-tendencia y refuerza momentum alineado.

  - **`roc60_abs = |roc_60|`**  
    *Justificación*: mide **intensidad del movimiento**, independientemente del signo.

---

**2. Interacciones para la ventana de gestación (08:00 - 09:00)**

  - Indicadores disponibles:
    - `ema_60`
    - `roc_60`
    - `momentum_10`, `momentum_5`, `momentum_3`
    - `roc_20`, `roc_30`

  - Interacciones propuestas

    - Tendencia × Momentum
      - **`mom10_x_emaSlope = momentum_10 * ema_slope_1`**  
        *Justificación*: identifica **impulso temprano alineado con la tendencia** dominante.

    - Aceleración multi-horizonte
      - **`roc20_minus_roc60 = roc_20 - roc_60`**  
      - **`roc30_minus_roc60 = roc_30 - roc_60`**  
        *Justificación*: mide **aceleración relativa** del precio frente al movimiento estructural.

    - Diferenciales de momentum
      - **`mom3_minus_mom10 = momentum_3 - momentum_10`**  
      - **`mom5_minus_mom10 = momentum_5 - momentum_10`**  
        *Justificación*: detecta **cambios de velocidad** entre escalas cortas y medias.

---

**3. Interacciones para la ventana de expansión / ejecución (09:00 - 10:00)**

- Indicadores disponibles:
  - `ema_60`
  - `roc_60`
  - `momentum_3`, `momentum_5`
  - `roc_20`

- Interacciones propuestas

  - **`ema_slope_1`**  
    *Justificación*: sigue siendo el filtro direccional principal.

  - **`mom5_x_emaSlope = momentum_5 * ema_slope_1`**  
    *Justificación*: refuerza **momentum operativo** solo cuando está alineado.

  - **`roc20_minus_roc60 = roc_20 - roc_60`**  
    *Justificación*: identifica **continuidad o agotamiento** durante la ejecución.

  - **`mom3_minus_mom5 = momentum_3 - momentum_5`**  
    *Justificación*: captura **micro-aceleraciones** útiles para timing.

---

**4. Criterio metodológico clave (muy importante)**

> **Las interacciones pueden calcularse para todo el día**,  
> pero su **evaluación (IC IS/OOS)** debe hacerse **exclusivamente dentro de la ventana donde tienen sentido operativo**.

  - Esto implica:
    - mismo dataset base,
    - mismas features,
    - **IC condicionado por ventana temporal**.

  - Este enfoque:
    - evita leakage,
    - mantiene consistencia del pipeline,
    - permite reutilizar features sin duplicar lógica.

---

**Síntesis**

- Las interacciones propuestas son **mínimas, interpretables y alineadas con el libro**.
- No se introducen combinaciones arbitrarias ni polinomios.
- Cada interacción tiene **una justificación económica y temporal clara**.
- El enfoque preserva estabilidad OOS y control metodológico.


### **8.2.2. Implementación**

In [145]:
import numpy as np
import pandas as pd

# ============================================================
# Interacciones mínimas (full_day / gestation / execution)
# ------------------------------------------------------------
# - Calcula interacciones para TODO el dataset (toda la jornada).
# - Luego, usted evalúa IC por ventana (full_day, gestation, execution)
#   filtrando por horario, como ya hace en su pipeline.
# ============================================================

def add_minimal_interactions(
    df: pd.DataFrame,
    *,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    close_col: str = "close",
    # Indicadores base (se asume que ya existen en df)
    ema_col: str = "ema_60",
    roc60_col: str = "roc_60",
    roc20_col: str = "roc_20",
    roc30_col: str = "roc_30",
    mom3_col: str = "momentum_3",
    mom5_col: str = "momentum_5",
    mom10_col: str = "momentum_10",
    # Nombres de salida
    prefix: str = "",
) -> tuple[pd.DataFrame, list[str]]:
    """
    Crea interacciones mínimas y controladas.
    Devuelve:
      - df_out: dataframe con nuevas columnas
      - created_cols: lista con los nombres creados

    Nota:
      - ema_60_lag1 se calcula por día (shift dentro del día).
      - Las columnas se crean aunque luego usted evalúe IC solo
        en la ventana correspondiente.
    """
    df_out = df.copy()

    # -------------------------
    # Helpers internos
    # -------------------------
    def _col(name: str) -> str:
        return f"{prefix}{name}" if prefix else name

    needed = [
        date_col, minute_col,
        ema_col, roc60_col,
    ]
    missing = [c for c in needed if c not in df_out.columns]
    if missing:
        raise KeyError(f"Faltan columnas requeridas: {missing}")

    created_cols: list[str] = []

    # ============================================================
    # 0) Lags / pendientes (por día)
    # ============================================================
    # ema_60_lag1 = EMA t-1 dentro del día
    ema_lag1 = _col(f"{ema_col}_lag1")
    df_out[ema_lag1] = df_out.groupby(date_col, sort=False)[ema_col].shift(1)
    created_cols.append(ema_lag1)

    # ema_slope_1 = ema - ema_lag1
    ema_slope_1 = _col("ema_slope_1")
    df_out[ema_slope_1] = df_out[ema_col] - df_out[ema_lag1]
    created_cols.append(ema_slope_1)

    # ============================================================
    # 1) Full day (siempre disponibles: EMA_60, ROC_60)
    # ============================================================
    roc60_abs = _col("roc60_abs")
    df_out[roc60_abs] = df_out[roc60_col].abs()
    created_cols.append(roc60_abs)

    roc60_x_emaSlope = _col("roc60_x_emaSlope")
    df_out[roc60_x_emaSlope] = df_out[roc60_col] * df_out[ema_slope_1]
    created_cols.append(roc60_x_emaSlope)

    # ============================================================
    # 2) Gestation (08:00-09:00): usa ROC_20/ROC_30 y Momentum_*
    #    (se crean igual para todo el día; su IC se evalúa en ventana)
    # ============================================================
    if roc20_col in df_out.columns:
        roc20_minus_roc60 = _col("roc20_minus_roc60")
        df_out[roc20_minus_roc60] = df_out[roc20_col] - df_out[roc60_col]
        created_cols.append(roc20_minus_roc60)

    if roc30_col in df_out.columns:
        roc30_minus_roc60 = _col("roc30_minus_roc60")
        df_out[roc30_minus_roc60] = df_out[roc30_col] - df_out[roc60_col]
        created_cols.append(roc30_minus_roc60)

    # Momentum diffs
    if mom3_col in df_out.columns and mom10_col in df_out.columns:
        mom3_minus_mom10 = _col("mom3_minus_mom10")
        df_out[mom3_minus_mom10] = df_out[mom3_col] - df_out[mom10_col]
        created_cols.append(mom3_minus_mom10)

    if mom5_col in df_out.columns and mom10_col in df_out.columns:
        mom5_minus_mom10 = _col("mom5_minus_mom10")
        df_out[mom5_minus_mom10] = df_out[mom5_col] - df_out[mom10_col]
        created_cols.append(mom5_minus_mom10)

    # Trend x momentum (gestation)
    if mom10_col in df_out.columns:
        mom10_x_emaSlope = _col("mom10_x_emaSlope")
        df_out[mom10_x_emaSlope] = df_out[mom10_col] * df_out[ema_slope_1]
        created_cols.append(mom10_x_emaSlope)

    # ============================================================
    # 3) Execution (09:00-10:00): Momentum_3/Momentum_5, ROC_20
    # ============================================================
    if mom5_col in df_out.columns:
        mom5_x_emaSlope = _col("mom5_x_emaSlope")
        df_out[mom5_x_emaSlope] = df_out[mom5_col] * df_out[ema_slope_1]
        created_cols.append(mom5_x_emaSlope)

    if mom3_col in df_out.columns and mom5_col in df_out.columns:
        mom3_minus_mom5 = _col("mom3_minus_mom5")
        df_out[mom3_minus_mom5] = df_out[mom3_col] - df_out[mom5_col]
        created_cols.append(mom3_minus_mom5)

    return df_out, created_cols




In [146]:
mnq_with_interactions, interaction_cols = add_minimal_interactions(
    mnq_features_targets,
    date_col="date",
    minute_col="minute_of_day",
    ema_col="ema_60",
    roc60_col="roc_60",
    roc20_col="roc_20",
    roc30_col="roc_30",
    mom3_col="momentum_3",
    mom5_col="momentum_5",
    mom10_col="momentum_10",
)

print("Interacciones creadas:")
print(interaction_cols)


Interacciones creadas:
['ema_60_lag1', 'ema_slope_1', 'roc60_abs', 'roc60_x_emaSlope', 'roc20_minus_roc60', 'roc30_minus_roc60', 'mom3_minus_mom10', 'mom5_minus_mom10', 'mom10_x_emaSlope', 'mom5_x_emaSlope', 'mom3_minus_mom5']


In [147]:
features_interactions_to_test = technical_indicators_features + interaction_cols
#features_interactions_to_test

In [148]:
# ============================================================
# IC tables IS vs OOS por ventana (con cache en Drive)
# ============================================================

ic_table_full_day_interactions = load_or_compute_ic_table(
    name="ic_table_full_day_interactions",
    df=mnq_with_interactions,
    indicator_columns=features_interactions_to_test,
    horizons=(60, 90),
    window_start=start_time_full_day,
    window_end=final_time_full_day,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

ic_table_gestation_interactions = load_or_compute_ic_table(
    name="ic_table_gestation_interactions",
    df=mnq_with_interactions,
    indicator_columns=features_interactions_to_test,
    horizons=(60, 90),
    window_start=start_time_gestation_window,
    window_end=final_time_gestation_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

ic_table_execution_interactions = load_or_compute_ic_table(
    name="ic_table_execution_interactions",
    df=mnq_with_interactions,
    indicator_columns=features_interactions_to_test,
    horizons=(60, 90),
    window_start=start_time_execution_window,
    window_end=final_time_execution_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/processed/ic_tables/ic_table_full_day_interactions.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/processed/ic_tables/ic_table_gestation_interactions.parquet
[cache] Computed & saved: /content/drive/MyDrive/neural_profit/data/processed/ic_tables/ic_table_execution_interactions.parquet


### **8.2.3. Evaluación de interacciones por ventana temporal**

#### **Full day**

In [149]:
print('\nic_table_full_day_interactions\n')
display (ic_table_full_day_interactions)


ic_table_full_day_interactions



,indicator,horizon,IC_IS,IC_OOS,ret_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,-0.193578,-0.189173,0.004405,281064,229712,,0.189173
1,ema_60_lag1,60,-0.189416,-0.183959,0.005456,280347,229126,,0.183959
2,roc_60,60,-0.187156,-0.177858,0.009298,281064,229712,,0.177858
3,roc_30,60,-0.135632,-0.135081,0.000551,281064,229712,,0.135081
4,roc_20,60,-0.113311,-0.114449,-0.001138,281064,229712,,0.114449
5,roc20_minus_roc60,60,0.121940,0.102249,-0.019691,281064,229712,,0.102249
6,momentum_10,60,-0.085062,-0.084314,0.000748,281064,229712,,0.084314
7,roc30_minus_roc60,60,0.100678,0.080945,-0.019733,281064,229712,,0.080945
8,mom3_minus_mom10,60,0.061595,0.065114,0.003519,281064,229712,,0.065114
9,momentum_5,60,-0.063818,-0.062301,0.001517,281064,229712,,0.062301


- **No aportan valor adicional**: las interacciones presentan un **IC OOS claramente menor** que los indicadores base.
- **Dominan los indicadores simples**: `ema_60`, `roc_60`, `roc_30`, `roc_20`.
- Las interacciones de tipo *slope* y *products* (`*_x_emaSlope`, `ema_slope_1`) se comportan como **ruido** en esta ventana.
- Únicas interacciones “aceptables”:
  - `mom3_minus_mom10`
  - `mom5_minus_mom10`  
  con **IC bajo pero estable**.

**Decisión**: en *Full Day*, **no utilizar interacciones**.

#### **Gestation (08:00–09:00)**

In [150]:
print('\nic_table_gestation_interactions\n')
display (ic_table_gestation_interactions)


ic_table_gestation_interactions



,indicator,horizon,IC_IS,IC_OOS,ret_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,-0.354846,-0.448972,-0.094126,43737,35746,,0.448972
1,ema_60_lag1,60,-0.321427,-0.409333,-0.087907,43737,35746,,0.409333
2,roc_30,60,-0.308842,-0.395218,-0.086377,43737,35746,,0.395218
3,roc_60,60,-0.319006,-0.383793,-0.064787,43737,35746,,0.383793
4,roc_20,60,-0.277677,-0.373996,-0.096319,43737,35746,,0.373996
5,momentum_10,60,-0.221873,-0.294612,-0.072739,43737,35746,,0.294612
6,momentum_5,60,-0.174053,-0.218417,-0.044364,43737,35746,,0.218417
7,mom3_minus_mom10,60,0.150388,0.209750,0.059362,43737,35746,,0.209750
8,momentum_3,60,-0.140842,-0.168313,-0.027471,43737,35746,,0.168313
9,mom5_minus_mom10,60,0.116622,0.163700,0.047078,43737,35746,,0.163700



- En esta ventana **sí emergen interacciones útiles**.
- Las diferencias de momentum:
  - `mom3_minus_mom10`
  - `mom5_minus_mom10`
  - `mom3_minus_mom5`  
  muestran **IC OOS elevado** y **mejora clara IS → OOS**.
- Capturan **aceleración/desaceleración temprana**, coherente con la lógica de gestación.
- Las interacciones con EMA (`*_x_emaSlope`) continúan siendo **débiles**.

**Decisión**:  Mantener **solo interacciones de tipo diferencia de momentum** en *Gestation*.

#### **Execution (09:00–10:00)**

In [151]:
print('\nic_table_execution_interactions\n')
display (ic_table_execution_interactions)


ic_table_execution_interactions



,indicator,horizon,IC_IS,IC_OOS,ret_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,-0.594698,-0.556157,0.038542,43737,35746,,0.556157
1,roc_30,60,-0.555345,-0.516905,0.038440,43737,35746,,0.516905
2,ema_60_lag1,60,-0.540886,-0.500600,0.040285,43737,35746,,0.500600
3,roc_20,60,-0.522272,-0.491503,0.030770,43737,35746,,0.491503
4,roc_60,60,-0.542577,-0.484769,0.057807,43737,35746,,0.484769
5,momentum_10,60,-0.435144,-0.399036,0.036108,43737,35746,,0.399036
6,momentum_5,60,-0.335210,-0.309230,0.025980,43737,35746,,0.309230
7,mom3_minus_mom10,60,0.302809,0.281376,-0.021433,43737,35746,,0.281376
8,momentum_3,60,-0.270437,-0.252229,0.018208,43737,35746,,0.252229
9,mom5_minus_mom10,60,0.234117,0.222986,-0.011132,43737,35746,,0.222986


- Interacciones de momentum **muy fuertes**:
  - `mom3_minus_mom10`
  - `mom5_minus_mom10`
  - `mom3_minus_mom5`
- **IC OOS alto y estable**, comparable al de los indicadores base.
- Aportan **timing fino**, no dirección estructural.
- Interacciones con ROC (`roc*_minus_roc60`) **no escalan bien**.
- Productos con EMA siguen siendo **irrelevantes**.

**Decisión**:  Usar **momentum-diff** como complemento de *timing* en *Execution*.


### **8.2.4. Decisión metodológica sobre interacciones y features técnicos**

A partir del análisis de **IC IS vs OOS** y su evaluación por ventanas temporales (*Full Day, Gestation y Execution*), se adopta la siguiente decisión respecto al uso de **interacciones entre indicadores técnicos** y la **conservación de los indicadores base**.

---

**1. Indicadores técnicos base (se mantienen)**

**Features base**
- `ema_60`
- `roc_20`, `roc_30`, `roc_60`
- `momentum_3`, `momentum_5`, `momentum_10`

**Justificación**
- Presentan **IC consistente y coherente entre IS y OOS** en todas las ventanas.
- Capturan **información estructural primaria del mercado**:
  - EMA → tendencia dominante.
  - ROC / Momentum → dirección y velocidad absoluta del movimiento.
- Son **interpretables, estables y alineados con la lógica del precio**.
- Funcionan como el **nivel absoluto de la señal**, base necesaria para cualquier refinamiento posterior.

En conclusión, los indicadores técnicos base constituyen el **núcleo del feature set** y no deben ser reemplazados.

---

**2. Interacciones seleccionadas (uso condicionado)**

**Interacciones válidas**
- `mom3_minus_mom10`
- `mom5_minus_mom10`
- `mom3_minus_mom5`

**Justificación**
- Capturan **dinámica relativa**, no redundante:
  - aceleración vs desaceleración,
  - cambios de régimen intradía,
  - timing fino del movimiento.
- Muestran **mejor comportamiento OOS** en ventanas donde el mercado entra en fase activa.
- Aportan información **ortogonal** al momentum absoluto.

---

**3. Interacciones descartadas**

**Descartadas de forma sistemática**
- Productos con pendiente de EMA (`*_x_emaSlope`)
- `ema_slope_1`
- `roc60_abs`
- Diferencias de ROC (`roc*_minus_roc60`)

**Motivo**
- IC bajo o inestable en OOS.
- No aportan señal adicional clara.
- Introducen ruido y complejidad sin ganancia predictiva.

---

**4. Regla metodológica**

- **Indicador técnico base**  
  → responde a *“qué tan fuerte es el movimiento”*.
- **Interacción (diferencia)**  
  → responde a *“cómo está cambiando ese movimiento”*.

Eliminar el indicador base y quedarse solo con la interacción **rompe esta jerarquía** y vuelve el feature dependiente del contexto, perdiendo estabilidad.

---
**5. Implementación práctica por ventana**

- **Full Day**
  - Indicadores base: ✅
  - Interacciones: ❌

- **Ventana de Gestation**
  - Indicadores base: ✅
  - Interacciones de momentum-diff: ✅

- **Ventana de Execution**
  - Indicadores base: ✅
  - Interacciones de momentum-diff: ✅

---

**Conclusión final**

- **Indicadores técnicos base** = señal estructural.
- **Interacciones seleccionadas** = refinamiento temporal y de régimen.

- No compiten: **se complementan**.  
- El feature set final mantiene **estabilidad, interpretabilidad y coherencia IS vs OOS**, alineado con la lógica mencionada por Jansen y del mercado.


### **8.3. Verificar estabilidad por horizonte**


#### **8.3.1. Introducción**


En predicción intradía, **un mismo indicador no necesariamente funciona igual para distintos horizontes de predicción**.  
Un feature puede capturar dinámicas de corto plazo (por ejemplo, impulsos rápidos), pero perder relevancia cuando el horizonte se extiende, o viceversa.

Por este motivo, no basta con que un indicador tenga **IC positivo**:  es necesario evaluar **si su relación con el target es estable cuando cambia el horizonte** \(H\).

Este análisis permite distinguir entre:
- señales **estructurales** del mercado, y
- señales **dependientes del horizonte** o directamente inestables.

El objetivo es que para cada feature, se busca determinar si:

- **Generaliza entre horizontes** (H = 60 y H = 90),
- Es **específica de un horizonte**, o
- Es **inestable** y debe descartarse.

En esta etapa **no se maximiza el IC**, sino que se evalúa **consistencia out-of-sample**.

**Criterios de clasificación**

Para cada feature \(f\):

1. Feature **estable**
    - IC OOS positivo en **H = 60 y H = 90**
    - Mantiene el signo
    - La variación de magnitud al aumentar H es moderada y explicable
    > Puede usarse en ambos horizontes.

2. Feature **especializada**
    - IC OOS positivo solo en **un horizonte**
    - IC OOS débil o cercano a cero en el otro
    - Sin comportamiento errático

    > Se utiliza únicamente en el horizonte donde aporta señal.

3. Feature **inestable**
    - IC OOS negativo
    - Cambios de signo sin patrón
    - Ruptura clara al variar el horizonte

    > Se descarta.

**Intuición clave**

- Un feature **estable** refleja una dinámica persistente del mercado.
- Un feature **especializado** captura efectos de escala temporal específica.
- Un feature **inestable** suele ser ruido o sobreajuste.

Este enfoque evita forzar indicadores fuera de su dominio natural de validez.

**Resultado esperado**

- Clasificar los features en lugar de eliminarlos arbitrariamente.
- Definir un **set coherente por horizonte**.
- Preparar el terreno para modelos que generalicen mejor out-of-sample.


#### **8.3.2. Implementación**


In [156]:
import numpy as np
import pandas as pd

# ============================================================
# IC IS vs OOS usando split_fe (no fechas)
# ============================================================
def compute_ic_is_oos_by_split(
    df: pd.DataFrame,
    features: list[str],
    target: str,
    *,
    split_col: str = "split_fe",
    is_label: str = "IS",
    oos_label: str = "OOS",
    method: str = "spearman",
    min_obs: int = 200,
) -> pd.DataFrame:
    """
    Calcula IC (Spearman/ Pearson) feature vs target por separado en IS y OOS
    usando la columna split_fe.

    Retorna columnas:
      feature, n_is, ic_is, n_oos, ic_oos, delta_oos_minus_is
    """
    data = df.copy()

    if split_col not in data.columns:
        raise KeyError(f"Falta columna '{split_col}' en el DataFrame.")
    if target not in data.columns:
        raise KeyError(f"Target '{target}' no existe en el DataFrame.")

    df_is = data.loc[data[split_col] == is_label, :]
    df_oos = data.loc[data[split_col] == oos_label, :]

    rows = []
    for f in features:
        if f not in data.columns:
            raise KeyError(f"Feature '{f}' no existe en el DataFrame.")

        # --- IS ---
        tmp_is = df_is[[f, target]].dropna()
        n_is = len(tmp_is)
        ic_is = np.nan
        if n_is >= min_obs:
            ic_is = tmp_is[f].corr(tmp_is[target], method=method)

        # --- OOS ---
        tmp_oos = df_oos[[f, target]].dropna()
        n_oos = len(tmp_oos)
        ic_oos = np.nan
        if n_oos >= min_obs:
            ic_oos = tmp_oos[f].corr(tmp_oos[target], method=method)

        rows.append({
            "feature": f,
            "n_is": n_is,
            "ic_is": ic_is,
            "n_oos": n_oos,
            "ic_oos": ic_oos,
            "ret_oos_minus_is": (ic_oos - ic_is)
            if (pd.notna(ic_oos) and pd.notna(ic_is)) else np.nan
        })

    out = pd.DataFrame(rows)
    # Orden: mayor |IC_OOS| primero
    out["abs_ic_oos"] = out["ic_oos"].abs()
    out = out.sort_values("abs_ic_oos", ascending=False).reset_index(drop=True)
    return out


# ============================================================
# Estabilidad por horizonte (H=60 vs H=90) usando split_fe
# ============================================================
def evaluate_feature_stability_by_horizon_split(
    df: pd.DataFrame,
    features: list[str],
    *,
    target_h60: str = "ret_60",
    target_h90: str = "ret_90",
    split_col: str = "split_fe",
    is_label: str = "IS",
    oos_label: str = "OOS",
    method: str = "spearman",
    min_obs: int = 200,
    min_abs_ic: float = 0.01,
    eps: float = 1e-12,
) -> pd.DataFrame:
    """
    Calcula IC IS/OOS para H=60 y H=90 y clasifica cada feature como:
      - stable: |IC_OOS|>=min_abs_ic en ambos horizontes + mismo signo
      - specialized_h60: señal en H60 (>=thr) y débil en H90 (<thr)
      - specialized_h90: señal en H90 (>=thr) y débil en H60 (<thr)
      - unstable: lo demás (incluye flips con señal relevante)

    Devuelve:
      feature, stability_label, ic_is_60/ic_oos_60/delta_60, ic_is_90/ic_oos_90/delta_90,
      ratio_oos_90_over_60, sign_flip_h60_h90, abs_min_oos, n_is/n_oos
    """
    ic60 = compute_ic_is_oos_by_split(
        df=df,
        features=features,
        target=target_h60,
        split_col=split_col,
        is_label=is_label,
        oos_label=oos_label,
        method=method,
        min_obs=min_obs,
    ).rename(columns={
        "ic_is": "ic_is_60",
        "ic_oos": "ic_oos_60",
        "ret_oos_minus_is": "ret_60",
        "n_is": "n_is_60",
        "n_oos": "n_oos_60",
    })

    ic90 = compute_ic_is_oos_by_split(
        df=df,
        features=features,
        target=target_h90,
        split_col=split_col,
        is_label=is_label,
        oos_label=oos_label,
        method=method,
        min_obs=min_obs,
    ).rename(columns={
        "ic_is": "ic_is_90",
        "ic_oos": "ic_oos_90",
        "ret_oos_minus_is": "ret_90",
        "n_is": "n_is_90",
        "n_oos": "n_oos_90",
    })

    out = ic60.merge(ic90, on="feature", how="inner")

    out["abs_oos_60"] = out["ic_oos_60"].abs()
    out["abs_oos_90"] = out["ic_oos_90"].abs()
    out["abs_min_oos"] = out[["abs_oos_60", "abs_oos_90"]].min(axis=1)

    # ratio de magnitudes (cuánto “queda” al pasar de 60 -> 90)
    out["ratio_oos_90_over_60"] = out["abs_oos_90"] / (out["abs_oos_60"] + eps)

    # signos y flips
    out["sign_oos_60"] = np.sign(out["ic_oos_60"])
    out["sign_oos_90"] = np.sign(out["ic_oos_90"])
    out["sign_flip_h60_h90"] = (
        (out["sign_oos_60"] != 0) & (out["sign_oos_90"] != 0) &
        (out["sign_oos_60"] != out["sign_oos_90"])
    )

    def _label(row) -> str:
        o60, o90 = row["ic_oos_60"], row["ic_oos_90"]
        if pd.isna(o60) or pd.isna(o90):
            return "unknown"

        strong60 = abs(o60) >= min_abs_ic
        strong90 = abs(o90) >= min_abs_ic
        flip = row["sign_flip_h60_h90"]

        if strong60 and strong90 and (not flip):
            return "stable"
        if strong60 and (not strong90):
            return "specialized_h60"
        if strong90 and (not strong60):
            return "specialized_h90"

        # Si hay flip con señal relevante en ambos, lo marcamos inestable
        if flip and strong60 and strong90:
            return "unstable"

        # Ambos débiles o mezcla no consistente
        return "unstable"

    out["stability_label"] = out.apply(_label, axis=1)

    # Orden: stable -> specialized -> unstable; y por fuerza mínima OOS
    order = {"stable": 0, "specialized_h60": 1, "specialized_h90": 2, "unstable": 3, "unknown": 4}
    out["_ord"] = out["stability_label"].map(order).fillna(99).astype(int)
    out = out.sort_values(by=["_ord", "abs_min_oos"], ascending=[True, False]).drop(columns="_ord")

    cols = [
        "feature", "stability_label",
        "ic_is_60", "ic_oos_60", "ret_60",
        "ic_is_90", "ic_oos_90", "ret_90",
        "abs_oos_60", "abs_oos_90", "abs_min_oos",
        "ratio_oos_90_over_60",
        "sign_oos_60", "sign_oos_90", "sign_flip_h60_h90",
        "n_is_60", "n_oos_60", "n_is_90", "n_oos_90",
    ]
    return out[cols]


# ============================================================
# Helper: filtrar por ventana intradía (opcional)
# ============================================================
def filter_by_minute_window(
    df: pd.DataFrame,
    *,
    minute_col: str = "minute_of_day",
    start_minute: int | None = None,
    end_minute: int | None = None,
) -> pd.DataFrame:
    """
    Filtra df por minute_of_day.
    - start_minute/end_minute son inclusivos.
    """
    out = df
    if start_minute is not None:
        out = out[out[minute_col] >= start_minute]
    if end_minute is not None:
        out = out[out[minute_col] <= end_minute]
    return out




In [157]:
# ============================================================
# CÓMO USARLO
# ============================================================

# 1) Define features
features_to_check = [
    "close", "ema_60",
    "roc_20", "roc_30", "roc_60",
    "momentum_3", "momentum_5", "momentum_10",
    "mom3_minus_mom10", "mom5_minus_mom10", "mom3_minus_mom5",
]

# 2) Full day (sin filtrar)
stability_full_day = evaluate_feature_stability_by_horizon_split(
    df=mnq_with_interactions,
    features=features_to_check,
    target_h60="ret_60",
    target_h90="ret_90",
    split_col="split_fe",
    is_label="IS",
    oos_label="OOS",
    method="spearman",
    min_obs=200,
    min_abs_ic=0.01,   # ajusta a tu criterio
)

print("\nstability_full_day\n")
display(stability_full_day)

# 3) Gestation (ejemplo: 08:00–09:00) -> necesitas tus minutos exactos
df_gestation = filter_by_minute_window(
    mnq_with_interactions,
    minute_col="minute_of_day",
    start_minute=8*60,   # ejemplo
    end_minute=9*60,     # ejemplo
)

stability_gestation = evaluate_feature_stability_by_horizon_split(
    df=df_gestation,
    features=features_to_check,
    target_h60="ret_60",
    target_h90="ret_90",
    split_col="split_fe",
    is_label="IS",
    oos_label="OOS",
    method="spearman",
    min_obs=200,
    min_abs_ic=0.01,
)

print("\nstability_gestation\n")
display(stability_gestation)

# 4) Execution (ejemplo: 09:00–10:00)
df_execution = filter_by_minute_window(
    mnq_with_interactions,
    minute_col="minute_of_day",
    start_minute=9*60,   # ejemplo
    end_minute=10*60,    # ejemplo
)

stability_execution = evaluate_feature_stability_by_horizon_split(
    df=df_execution,
    features=features_to_check,
    target_h60="ret_60",
    target_h90="ret_90",
    split_col="split_fe",
    is_label="IS",
    oos_label="OOS",
    method="spearman",
    min_obs=200,
    min_abs_ic=0.01,
)

print("\nstability_execution\n")
display(stability_execution)



stability_full_day



,feature,stability_label,ic_is_60,ic_oos_60,ret_60,ic_is_90,ic_oos_90,ret_90,abs_oos_60,abs_oos_90,abs_min_oos,ratio_oos_90_over_60,sign_oos_60,sign_oos_90,sign_flip_h60_h90,n_is_60,n_oos_60,n_is_90,n_oos_90
0,close,stable,-0.043304,-0.043476,-0.000171,-0.051420,-0.054131,-0.002711,0.043476,0.054131,0.043476,1.245093,-1.0,-1.0,False,281064,229712,281064,229712
1,roc_60,stable,0.005668,0.031913,0.026246,0.009021,0.030724,0.021703,0.031913,0.030724,0.030724,0.962728,1.0,1.0,False,281064,229712,281064,229712
2,ema_60,stable,0.010691,0.029803,0.019113,0.010930,0.029764,0.018834,0.029803,0.029764,0.029764,0.998678,1.0,1.0,False,281064,229712,281064,229712
3,roc_30,stable,0.007810,0.019952,0.012142,0.008917,0.023236,0.014319,0.019952,0.023236,0.019952,1.164625,1.0,1.0,False,281064,229712,281064,229712
4,roc_20,stable,0.009951,0.016372,0.006421,0.006671,0.019300,0.012628,0.016372,0.019300,0.016372,1.178798,1.0,1.0,False,281064,229712,281064,229712
5,momentum_10,stable,0.008253,0.014951,0.006699,0.006128,0.013512,0.007384,0.014951,0.013512,0.013512,0.903746,1.0,1.0,False,281064,229712,281064,229712
6,mom3_minus_mom10,stable,-0.008623,-0.012569,-0.003946,-0.006887,-0.011281,-0.004394,0.012569,0.011281,0.011281,0.897517,-1.0,-1.0,False,281064,229712,281064,229712
8,mom5_minus_mom10,specialized_h60,-0.007629,-0.010176,-0.002547,-0.006336,-0.009256,-0.002919,0.010176,0.009256,0.009256,0.909580,-1.0,-1.0,False,281064,229712,281064,229712
7,momentum_5,specialized_h60,0.004779,0.011251,0.006472,0.003068,0.009043,0.005975,0.011251,0.009043,0.009043,0.803781,1.0,1.0,False,281064,229712,281064,229712
9,momentum_3,unstable,0.002191,0.008322,0.006130,0.001049,0.006347,0.005298,0.008322,0.006347,0.006347,0.762630,1.0,1.0,False,281064,229712,281064,229712



stability_gestation



,feature,stability_label,ic_is_60,ic_oos_60,ret_60,ic_is_90,ic_oos_90,ret_90,abs_oos_60,abs_oos_90,abs_min_oos,ratio_oos_90_over_60,sign_oos_60,sign_oos_90,sign_flip_h60_h90,n_is_60,n_oos_60,n_is_90,n_oos_90
0,close,stable,-0.052040,-0.057808,-0.005768,-0.061376,-0.079521,-0.018145,0.057808,0.079521,0.057808,1.375610,-1.0,-1.0,False,43737,35746,43737,35746
1,roc_20,specialized_h90,0.002645,-0.007126,-0.009771,-0.018166,0.025596,0.043762,0.007126,0.025596,0.007126,3.591703,-1.0,1.0,True,43737,35746,43737,35746
3,momentum_10,specialized_h90,0.009745,-0.003827,-0.013572,-0.007613,0.011440,0.019053,0.003827,0.011440,0.003827,2.989506,-1.0,1.0,True,43737,35746,43737,35746
7,ema_60,specialized_h90,-0.022307,-0.001816,0.020491,-0.039402,0.032693,0.072095,0.001816,0.032693,0.001816,18.001682,-1.0,1.0,True,43737,35746,43737,35746
8,roc_60,specialized_h90,-0.043486,0.001607,0.045093,-0.042859,0.035603,0.078462,0.001607,0.035603,0.001607,22.157685,1.0,1.0,False,43737,35746,43737,35746
9,roc_30,specialized_h90,-0.018034,-0.000496,0.017538,-0.033737,0.037024,0.070761,0.000496,0.037024,0.000496,74.672690,-1.0,1.0,True,43737,35746,43737,35746
10,mom3_minus_mom10,specialized_h90,-0.006542,-0.000429,0.006113,0.003761,-0.010353,-0.014114,0.000429,0.010353,0.000429,24.145136,-1.0,-1.0,False,43737,35746,43737,35746
2,momentum_3,unstable,0.006541,0.006517,-0.000024,-0.003084,0.008866,0.011951,0.006517,0.008866,0.006517,1.360467,1.0,1.0,False,43737,35746,43737,35746
4,mom5_minus_mom10,unstable,-0.005087,-0.003672,0.001415,0.001977,-0.008553,-0.010530,0.003672,0.008553,0.003672,2.329327,-1.0,-1.0,False,43737,35746,43737,35746
5,momentum_5,unstable,0.006811,0.003249,-0.003561,-0.009206,0.009713,0.018919,0.003249,0.009713,0.003249,2.989286,1.0,1.0,False,43737,35746,43737,35746



stability_execution



,feature,stability_label,ic_is_60,ic_oos_60,ret_60,ic_is_90,ic_oos_90,ret_90,abs_oos_60,abs_oos_90,abs_min_oos,ratio_oos_90_over_60,sign_oos_60,sign_oos_90,sign_flip_h60_h90,n_is_60,n_oos_60,n_is_90,n_oos_90
0,close,stable,-0.083779,-0.040375,0.043404,-0.099177,-0.035708,0.063469,0.040375,0.035708,0.035708,0.884398,-1.0,-1.0,False,43737,35746,43737,35746
1,roc_60,stable,0.022461,0.016969,-0.005492,0.024159,0.018527,-0.005632,0.016969,0.018527,0.016969,1.091830,1.0,1.0,False,43737,35746,43737,35746
2,ema_60,stable,0.017262,0.010900,-0.006362,0.021621,0.019379,-0.002242,0.010900,0.019379,0.010900,1.777899,1.0,1.0,False,43737,35746,43737,35746
3,momentum_10,stable,0.005828,0.010664,0.004836,0.015770,0.018650,0.002880,0.010664,0.018650,0.010664,1.748840,1.0,1.0,False,43737,35746,43737,35746
4,mom3_minus_mom10,stable,-0.008225,-0.010301,-0.002076,-0.018001,-0.016477,0.001524,0.010301,0.016477,0.010301,1.599608,-1.0,-1.0,False,43737,35746,43737,35746
6,momentum_5,specialized_h90,0.003791,0.005943,0.002151,0.009581,0.012166,0.002584,0.005943,0.012166,0.005943,2.047211,1.0,1.0,False,43737,35746,43737,35746
7,mom5_minus_mom10,specialized_h90,-0.007353,-0.005888,0.001464,-0.016904,-0.012982,0.003922,0.005888,0.012982,0.005888,2.204746,-1.0,-1.0,False,43737,35746,43737,35746
8,roc_20,specialized_h90,0.027883,0.005201,-0.022682,0.035363,0.017201,-0.018162,0.005201,0.017201,0.005201,3.307480,1.0,1.0,False,43737,35746,43737,35746
10,roc_30,specialized_h90,0.035886,-0.000180,-0.036066,0.040562,0.015553,-0.025010,0.000180,0.015553,0.000180,86.442623,-1.0,1.0,True,43737,35746,43737,35746
5,mom3_minus_mom5,unstable,-0.000603,-0.006192,-0.005589,-0.005929,-0.009117,-0.003188,0.006192,0.009117,0.006192,1.472465,-1.0,-1.0,False,43737,35746,43737,35746


#### **8.3.3.Conclusiones rápidas - Estabilidad por horizonte y ventana**


**1. Indicadores válidos durante **todo el día (Full Day)**

- **`ema_60` y `roc_60`**
  - Son los **indicadores más estables**:
    - Mantienen el signo entre H=60 y H=90.
    - Presentan magnitudes IC similares entre horizontes.
  - Capturan **estructura de tendencia y momentum de fondo**.
  - **Se usan como features globales**, válidos durante toda la jornada.

- **`close`**
  - Mantiene estabilidad por horizonte.
  - Actúa como **ancla de precio**, no como señal direccional.
  - Se conserva como feature base.

➡️ **Conclusión**:  
`ema_60` y `roc_60` forman el **núcleo estructural** del modelo y se utilizan **en todo el día**.

---

**2. Indicadores activos en **ventanas operativas** (no todo el día)**

- Ventana de gestación

  - **`ema_60`**
  - **`roc_60`**
  - **`roc_20`, `roc_30`**
  - **momentums (`momentum_10`, `momentum_5`, `momentum_3`)**
  - **interacciones de momentum**

  Características:
  - Aportan señal **dependiente del horizonte** (principalmente H=90).
  - No deben forzarse fuera de esta ventana.
  - Funcionan como **precondición de movimiento**, no como timing fino.

---

- Ventana de ejecución
  - **`ema_60` y derivados (`ema_60_lag1`, `ema_slope_1`)**
  - **`roc_60`, `roc_30`, `roc_20`**
  - **momentums e interacciones**:
    - `mom3_minus_mom10`
    - `mom5_minus_mom10`
    - `mom3_minus_mom5`

  Características:
  - IC más alto y más consistente.
  - Mejor comportamiento OOS.
  - Reflejan **dinámica activa del precio**.


**Conclusión**:  

Las **interacciones y momentum** **sí se utilizan**, pero **solo en ventanas**, no como señales globales.

---

**3. Decisión final de uso por tipo de feature**

| Tipo de feature | Uso |
|---|---|
| `ema_60`, `roc_60` | Todo el día |
| `roc_20`, `roc_30` | Solo ventanas |
| `momentum_*` | Solo ventanas |
| Interacciones | Solo ventanas |
| `close` | Feature base |

---

**4. Lectura metodológica**

- No hay contradicción entre estabilidad y especialización:
  - **Estables** → estructura (todo el día).
  - **Especializados** → dinámica local (ventanas).
- El error sería usar indicadores de ventana como señales globales.
- El pipeline queda alineado con:
  - dinámica intradía real,
  - control de leakage,
  - generalización out-of-sample.

- La interpretación correcta es:
  > *EMA y ROC largos estructuran el día;  
  > ROC cortos, momentum e interacciones explican **cuándo** y **cómo** se mueve el precio dentro del día.*



## **9. Consolidación de features**


A partir del análisis estadístico y estructural realizado en este stage, se concluye
que el conjunto de features adecuado para el entrenamiento del modelo está compuesto por:

- **Variables OHLC**:
  - `close`
- **Indicadores técnicos seleccionados**:
  - `ema_60`
  - `roc_20`
  - `roc_30`
  - `roc_60`
  - `momentum_3`
  - `momentum_5`
  - `momentum_10`
- **Interaccioones seleccionadas**:
  - `mom3_minus_mom10`
  - `mom5_minus_mom10`
  - `mom3_minus_mom5`





### **9.1. Justificación del set final de features**


A partir del análisis estadístico, estructural y de estabilidad realizado a lo largo de este stage, se define un conjunto de features que equilibra **capacidad predictiva, estabilidad out-of-sample e interpretabilidad**, evitando redundancia y sobreajuste.

**1. Variable OHLC seleccionada**

  - **`close`**

    - Las variables OHLC presentan **correlación extremadamente alta entre sí** (> 0.99), por lo que aportan información prácticamente redundante.
    - `close` concentra la información relevante del precio:
      - Es el valor más utilizado por los indicadores técnicos.
      - Resume el consenso del mercado al cierre de cada minuto.
    - Su IC es **estable entre horizontes (H=60 y H=90)** y consistente entre IS y OOS.

  - **Decisión**: se utiliza únicamente `close` como variable de precio base.

**2. Indicadores técnicos seleccionados**

Los indicadores fueron elegidos por cumplir simultáneamente:

  - IC out-of-sample distinto de cero,
  - coherencia direccional IS vs OOS,
  - estabilidad entre horizontes o especialización explicable,
  - alineación con la lógica de mercado (tendencia + momentum).

  Los indicadores técnicos seleccionados son:

  - `ema_60`

    - Captura la **tendencia intradía dominante**.
    - Es estable en full day y relevante en ventanas operativas.
    - Funciona como **filtro estructural**, no como señal puntual.

  - `roc_20`, `roc_30`, `roc_60`

    - Representan **velocidad y magnitud del movimiento del precio** en distintas escalas.
    - `roc_60`: estructura de medio plazo (usable todo el día).
    - `roc_20` y `roc_30`: dinámicas más cortas, relevantes en ventanas específicas.
    - Muestran **buena generalización OOS** y coherencia entre horizontes.

  - `momentum_3`, `momentum_5`, `momentum_10`

    - Capturan impulsos de **muy corto plazo**, especialmente útiles en ventanas de gestación y ejecución.
    - Individualmente pueden ser ruidosos, pero:
      - mantienen señal consistente en contextos específicos,
      - son la base para interacciones más informativas.

**3. Interacciones seleccionadas**

Las interacciones no se introducen para aumentar complejidad, sino para **capturar relaciones relativas entre escalas temporales**, que mostraron mayor estabilidad que algunos indicadores crudos.

  - `mom3_minus_mom10`

    - Mide aceleración corta vs impulso base.
    - Detecta cambios tempranos de régimen.
    - Mostró buena estabilidad en execution.

  - `mom5_minus_mom10`

    - Captura divergencias entre momentum corto y medio.
    - Útil para distinguir continuidad vs agotamiento.

  - `mom3_minus_mom5`

    - Refina el análisis del impulso inmediato.
    - Aporta señal complementaria en ventanas operativas.

Las interacciones **no reemplazan** a los indicadores base, sino que los **contextualizan**.

**4. Cómo se utilizarán en el modelo**

- **Indicadores base (`ema`, `roc`, `momentum`)**:
  - Proveen información primaria de tendencia y velocidad.
  - Se utilizarán como features directos (eventualmente normalizados intradía).

- **Interacciones de momentum**:
  - Actúan como features de **contexto relativo**.
  - Ayudan al modelo a distinguir:
    - impulso genuino vs ruido,
    - aceleración vs desaceleración.

- **Separación por ventanas (full day / gestation / execution)**:
  - Los features se calculan para todo el día,
  - pero su **evaluación y peso** se interpretan según la ventana donde demostraron mayor valor.

### **9.2. Decisión final y set consolidado de features**


El set final de features:

- es **compacto** (sin redundancias),
- **estable out-of-sample**,
- **interpretable desde la lógica de mercado**,
- y está preparado para **modelos que generalicen**, no para optimizar IC in-sample.

Este conjunto constituye una base sólida para el entrenamiento de modelos intradía con control explícito de generalización y sin leakage.

| Tipo de feature                | Variable(s)                                      | Rol económico principal                                                                 |
|--------------------------------|--------------------------------------------------|------------------------------------------------------------------------------------------|
| Precio intradía                | `close`                                         | Estado representativo del precio; consenso del mercado en cada minuto                    |
| Tendencia intradía             | `ema_60`                                        | Dirección y nivel tendencial dominante del día; filtro estructural del mercado           |
| Momentum de muy corto plazo    | `momentum_3`, `momentum_5`                      | Impulso inmediato; captura aceleraciones y micro-movimientos del precio                 |
| Momentum de corto plazo        | `momentum_10`                                   | Intensidad del desplazamiento reciente; referencia base de impulso                       |
| Velocidad de corto plazo       | `roc_20`, `roc_30`                              | Rapidez y magnitud del movimiento en escalas cortas; útil en ventanas operativas         |
| Velocidad de medio plazo       | `roc_60`                                       | Dinámica estructural del movimiento; estabilidad direccional a lo largo del día          |
| Interacción de momentum        | `mom3_minus_mom10`                              | Aceleración relativa: impulso corto frente a impulso base                                |
| Interacción de momentum        | `mom5_minus_mom10`                              | Divergencia entre momentum corto y medio; detección de agotamiento o continuidad         |
| Interacción de momentum        | `mom3_minus_mom5`                               | Refinamiento del impulso inmediato; cambios tempranos en la dinámica intradía            |


### **9.3. Implicancia para el pipeline**


Este set consolidado constituye el núcleo de features técnicas del modelo y será utilizado en las etapas posteriores para:

- entrenamiento y validación del modelo predictivo,
- análisis de contribución por feature,
- y evaluación de desempeño económico.

La consolidación reduce la dimensionalidad, mejora la interpretabilidad y alinea el diseño del modelo con la estructura temporal y económica observada en los datos.

### **9.4. Implementación**


In [ ]:
mnq_intraday_labeled = load_data()

In [ ]:
def info_dataset_final(df):
    print("Información del dataset:\n")

    # -------------------------
    # Días y registros
    # -------------------------
    num_dias = df["date"].nunique()
    print(f"\tCantidad de días: {num_dias}")

    validos_por_dia = (
        df.dropna(subset=["close"])
          .groupby("date")
          .size()
    )
    promedio_por_fecha = validos_por_dia.mean()
    print(f"\tRegistros por día: {int(promedio_por_fecha)}")

    # -------------------------
    # Horarios (desde minute_of_day)
    # -------------------------
    if "minute_of_day" in df.columns:
        min_minute = int(df["minute_of_day"].min())
        max_minute = int(df["minute_of_day"].max())

        primer_hora = f"{min_minute // 60:02d}:{min_minute % 60:02d}"
        ultima_hora = f"{max_minute // 60:02d}:{max_minute % 60:02d}"
    else:
        primer_hora = None
        ultima_hora = None

    print(f"\tHora diaria de inicio: {primer_hora}")
    print(f"\tHora diaria de final: {ultima_hora}")

    # -------------------------
    # Columnas del dataset
    # -------------------------
    print("\n\tColumnas del dataset:")
    for c in df.columns:
        print(f"\t- {c}")



In [ ]:
info_dataset_final(mnq_intraday_labeled)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 481
	Hora diaria de inicio: 06:30
	Hora diaria de final: 14:30

	Columnas del dataset:
	- date
	- open
	- high
	- low
	- close
	- volume
	- delta_pts_60
	- trade_60
	- target_op_60
	- target_tail_60
	- delta_pts_90
	- trade_90
	- target_op_90
	- target_tail_90
	- minute_of_day


In [ ]:
import pandas as pd
from typing import List, Tuple
from ta.momentum import ROCIndicator


def build_mnq_features_targets_full(
    mnq_intraday_labeled: pd.DataFrame,
    *,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    close_col: str = "close",
    target_cols: Tuple[str, str] = ("delta60", "delta90"),
    tz: str | None = "America/New_York",   # se deja, pero NO se usa para el índice
    keep_datetime_col: bool = True,
) -> Tuple[pd.DataFrame, List[str], List[str]]:

    # ------------------------------------------------------------
    # Preservar el índice original (NO tocarlo)
    # ------------------------------------------------------------
    original_index = mnq_intraday_labeled.index

    # ------------------------------------------------------------
    # 0) Normalización de nombres (si vinieran con delta_pts_*)
    # ------------------------------------------------------------
    df = mnq_intraday_labeled.copy()
    df = df.rename(columns={"delta_pts_60": "delta60", "delta_pts_90": "delta90"})

    # -------------------------
    # Validación columnas mínimas
    # -------------------------
    required_base = [date_col, minute_col, close_col, *list(target_cols)]
    missing = [c for c in required_base if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas en mnq_intraday_labeled: {missing}")

    # ------------------------------------------------------------
    # 2) Indicadores técnicos (por día)
    # ------------------------------------------------------------
    technical_indicators_features: List[str] = [
        "ema60",
        "mom3",
        "mom5",
        "mom10",
        "roc20",
        "roc30",
        "roc60",
    ]

    def _apply_per_day(g: pd.DataFrame) -> pd.DataFrame:
        g = g.copy()

        g["ema60"] = g[close_col] / g[close_col].ewm(span=60, adjust=False).mean() - 1.0

        g["mom3"] = g[close_col].pct_change(3)
        g["mom5"] = g[close_col].pct_change(5)
        g["mom10"] = g[close_col].pct_change(10)

        g["roc20"] = ROCIndicator(close=g[close_col], window=20).roc()
        g["roc30"] = ROCIndicator(close=g[close_col], window=30).roc()
        g["roc60"] = ROCIndicator(close=g[close_col], window=60).roc()

        return g

    # Groupby por date (columna), no por índice
    df = df.groupby(df[date_col], group_keys=False, sort=False).apply(_apply_per_day)

    # ✅ Garantía dura: restaurar el índice EXACTO del input
    # (si por cualquier motivo apply lo alteró)
    if not df.index.equals(original_index):
        df = df.copy()
        df.index = original_index

    # ------------------------------------------------------------
    # 3) Interacciones mínimas
    # ------------------------------------------------------------
    def calculate_interactions_features(
        df_in: pd.DataFrame,
        *,
        date_col: str = "date",
        minute_col: str = "minute_of_day",
        mom3_col: str = "mom3",
        mom5_col: str = "mom5",
        mom10_col: str = "mom10",
        prefix: str = "",
    ) -> Tuple[pd.DataFrame, List[str]]:

        df_out = df_in.copy()

        def _col(name: str) -> str:
            return f"{prefix}{name}" if prefix else name

        needed = [date_col, minute_col, mom3_col, mom5_col, mom10_col]
        miss = [c for c in needed if c not in df_out.columns]
        if miss:
            raise KeyError(f"Faltan columnas requeridas para interacciones: {miss}")

        created_cols: List[str] = []

        c = _col("mom3_mom10")
        df_out[c] = df_out[mom3_col] - df_out[mom10_col]
        created_cols.append(c)

        c = _col("mom5_mom10")
        df_out[c] = df_out[mom5_col] - df_out[mom10_col]
        created_cols.append(c)

        c = _col("mom3_mom5")
        df_out[c] = df_out[mom3_col] - df_out[mom5_col]
        created_cols.append(c)

        return df_out, created_cols

    df, interactions_features = calculate_interactions_features(
        df,
        date_col=date_col,
        minute_col=minute_col,
        mom3_col="mom3",
        mom5_col="mom5",
        mom10_col="mom10",
    )

    # ------------------------------------------------------------
    # 4) Dataset final (selección de columnas)
    # ------------------------------------------------------------
    final_cols: List[str] = [
        date_col,
        minute_col,
        close_col,
        *technical_indicators_features,
        *interactions_features,
        *list(target_cols),
    ]

    # Mantengo esto solo si la columna "datetime" EXISTE; si no, no la invento
    if keep_datetime_col and "datetime" in df.columns:
        final_cols = ["datetime"] + final_cols

    missing_final = [c for c in final_cols if c not in df.columns]
    if missing_final:
        raise ValueError(f"No se pudieron construir todas las columnas finales: {missing_final}")

    mnq_features_targets = df.loc[:, final_cols].copy()

    # ✅ Garantía final: el índice sale idéntico al que entró
    if not mnq_features_targets.index.equals(original_index):
        mnq_features_targets.index = original_index

    return mnq_features_targets, technical_indicators_features, interactions_features



In [ ]:
mnq_features_targets, tech_feats, inter_feats = build_mnq_features_targets_full(
    mnq_intraday_labeled=mnq_intraday_labeled
)

mnq_features_targets.head()

,date,minute_of_day,close,ema60,mom3,mom5,mom10,roc20,roc30,roc60,mom3_mom10,mom5_mom10,mom3_mom5,delta60,delta90
datetime,,,,,,,,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,390,8727.75,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.00,6.00
2019-12-23 06:31:00-05:00,2019-12-23,391,8726.50,-0.000139,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.25,7.50
2019-12-23 06:32:00-05:00,2019-12-23,392,8725.25,-0.000273,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.75,8.00
2019-12-23 06:33:00-05:00,2019-12-23,393,8726.00,-0.000180,-0.000201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.50,8.00
2019-12-23 06:34:00-05:00,2019-12-23,394,8726.00,-0.000175,-0.000057,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.00,7.75


In [ ]:
print(f"OHLVC feats: ['close'] \n")
print(f"Tech feats: {tech_feats} \n")
print(f"Interaction feats: {inter_feats} \n")
print(f"Targets: {targets}")

OHLVC feats: ['close'] 

Tech feats: ['ema60', 'mom3', 'mom5', 'mom10', 'roc20', 'roc30', 'roc60'] 

Interaction feats: ['mom3_mom10', 'mom5_mom10', 'mom3_mom5'] 

Targets: ['delta_60', 'delta_90']


In [ ]:
info_dataset_final(mnq_features_targets)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 481
	Hora diaria de inicio: 06:30
	Hora diaria de final: 14:30

	Columnas del dataset:
	- date
	- minute_of_day
	- close
	- ema60
	- mom3
	- mom5
	- mom10
	- roc20
	- roc30
	- roc60
	- mom3_mom10
	- mom5_mom10
	- mom3_mom5
	- delta60
	- delta90


## **10. Flags de activación por ventanas operativas**

### **10.1. Introducción conceptual**


**1. Motivación principal: valor predictivo dependiente del horario**

El dataset incluye features cuyo valor predictivo no es homogéneo a lo largo de la jornada.

En particular:

- Algunos indicadores (por ejemplo, ciertos momentum y ROC) muestran mayor capacidad predictiva durante las ventanas de:

  - gestation (08:00-09:00)
  - execution (09:00-10:00)

- Fuera de esas ventanas, esas mismas features:
  - pierden señal,
  - se vuelven ruidosas,
  - o directamente dejan de ser informativas para la toma de decisiones.

Sin embargo, el modelo se entrena usando ventanas deslizantes a lo largo de todo el full day.
Por lo tanto, es necesario informarle explícitamente al modelo en qué contextos temporales una feature es relevante.

**2. Por qué se utilizan flags y no filtros duros**

En lugar de:

- eliminar features fuera de ciertas horas, o
- entrenar modelos distintos por franja horaria,

se introducen flags de activación para permitir que el modelo:

- vea toda la jornada (maximizando datos),
- pero aprenda cuándo una feature es confiable.

Cada flag responde a la pregunta:

>“¿Esta feature tiene sentido predictivo en este momento del día?”

**3. Qué aprende el modelo con este esquema**

Cada muestra contiene pares del tipo:

$$  
𝑋_t = [roc20, roc20_{active}, mom3, mom3_{active}, ... ]
$$

Durante el entrenamiento, el modelo aprende patrones como:

- Cuando `roc20_active` = 1

  → `roc20` suele correlacionar mejor con el target.

- Cuando `roc20_active` = 0

  → `roc20` pierde poder explicativo y debe ser atenuado o ignorado.

De este modo, el modelo modula dinámicamente la importancia de cada feature en función del contexto horario, sin reglas hard-coded.

**4. Por qué no es suficiente “poner ceros”**

Asignar 0 a una feature fuera de su ventana introduce ambigüedad:

- `roc20` = 0 puede significar:

  - un valor legítimo del indicador, o
  - una feature fuera de su régimen predictivo.

Con flags explícitos:
  - `roc20` = 0
  - `roc20_active` = 0

el modelo puede distinguir claramente entre:
  - “valor bajo pero válido”
  - “valor no relevante en este horario”

**5. Interpretación conceptual**

El flag actúa como un contextualizador temporal, no como una regla.

- Feature → señal numérica
- Flag → indica si la señal está en su régimen de mayor valor predictivo

El modelo aprende implícitamente:

> “Esta feature solo merece atención cuando estoy dentro de la ventana operativa adecuada.”

**6. Condiciones necesarias para que el enfoque sea válido**

1. Cada feature dependiente del horario tiene su flag correspondiente.
2. Los flags se mantienen como variables binarias (0/1) y no se escalan.
3. El orden de columnas es estrictamente consistente en todo el pipeline.

**Conclusión**

El uso de flags permite entrenar un modelo con cobertura completa del día, sin perder la capacidad de:

- capturar regímenes horarios de mayor valor predictivo,
- diferenciar señal estructural de ruido intradía,
- y alinear el aprendizaje con las ventanas reales de operación y ejecución.

Este diseño es coherente con un enfoque full-day learning y window-aware decision making.

### **10.2. Implementación**


In [ ]:
execution_features = ['mom3', 'mom5', 'roc20', 'mom3_mom5']
gestation_features = ['mom10','roc30','mom3_mom10','mom5_mom10'] + execution_features

In [ ]:
#print(f'gestation_window: {start_time_gestation_window}-{final_time_gestation_window}')
#print(f'gestation_features: {gestation_features}')

#print(f'execution_window: {start_time_execution_window}-{final_time_execution_window}')
#print(f'execution_features: {execution_features}' )

In [ ]:
windows = {
    "gestation": (start_time_gestation_window, final_time_gestation_window),
    "execution": (start_time_execution_window, final_time_execution_window)
}

window_features = {
    "gestation": gestation_features,
    "execution": execution_features,
}

In [ ]:
import pandas as pd
from typing import Iterable, Dict, Tuple

def add_time_window_feature_flags(
    df: pd.DataFrame,
    *,
    windows: Dict[str, Tuple[str, str]],
    window_features: Dict[str, Iterable[str]],
    flag_suffix: str = "_active",
) -> pd.DataFrame:
    """
    Agrega flags binarios por feature indicando si el timestamp (índice datetime)
    cae dentro de la ventana temporal asociada a esa feature.

    - NO modifica el índice.
    - NO altera valores de las features.
    - Devuelve una copia del DataFrame con nuevas columnas *_active.

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset con índice DatetimeIndex.
    windows : dict
        Mapeo nombre_ventana -> (start_time, end_time) en formato 'HH:MM'.
        Ej: {'gestation': ('08:00','09:00'), 'execution': ('09:00','10:00')}
    window_features : dict
        Mapeo nombre_ventana -> iterable de features que aplican a esa ventana.
    flag_suffix : str
        Sufijo para las columnas flag. Default '_active'.

    Retorna
    -------
    pd.DataFrame
        Copia del df con flags agregados.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El DataFrame debe tener un índice DatetimeIndex.")

    out = df.copy()

    # Hora del índice sin tocar timezone
    idx_time = out.index.strftime("%H:%M")

    for window_name, (start, end) in windows.items():
        if window_name not in window_features:
            continue

        is_in_window = (idx_time >= start) & (idx_time < end)

        for feat in window_features[window_name]:
            if feat not in out.columns:
                raise ValueError(f"La feature '{feat}' no existe en el DataFrame.")

            flag_col = f"{feat}{flag_suffix}"
            # Si la feature aparece en múltiples ventanas, OR lógico
            if flag_col in out.columns:
                out[flag_col] = out[flag_col] | is_in_window.astype("int8")
            else:
                out[flag_col] = is_in_window.astype("int8")

    return out



In [ ]:
mnq_features_targets = add_time_window_feature_flags(
    mnq_features_targets,
    windows=windows,
    window_features=window_features,
)

In [ ]:
info_dataset_final(mnq_features_targets)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 481
	Hora diaria de inicio: 06:30
	Hora diaria de final: 14:30

	Columnas del dataset:
	- date
	- minute_of_day
	- close
	- ema60
	- mom3
	- mom5
	- mom10
	- roc20
	- roc30
	- roc60
	- mom3_mom10
	- mom5_mom10
	- mom3_mom5
	- delta60
	- delta90
	- mom10_active
	- roc30_active
	- mom3_mom10_active
	- mom5_mom10_active
	- mom3_active
	- mom5_active
	- roc20_active
	- mom3_mom5_active


In [ ]:
mnq_features_targets.columns

Index(['date', 'minute_of_day', 'close', 'ema60', 'mom3', 'mom5', 'mom10',
       'roc20', 'roc30', 'roc60', 'mom3_mom10', 'mom5_mom10', 'mom3_mom5',
       'delta60', 'delta90', 'mom10_active', 'roc30_active',
       'mom3_mom10_active', 'mom5_mom10_active', 'mom3_active', 'mom5_active',
       'roc20_active', 'mom3_mom5_active'],
      dtype='object')

In [ ]:
base_order = [
    'date',
    'minute_of_day',
]

features_order = [
    'close',
    'ema60',
    'roc60',

    'roc30', 'roc30_active',
    'roc20', 'roc20_active',

    'mom10', 'mom10_active',
    'mom5',  'mom5_active',
    'mom3',  'mom3_active',

    'mom5_mom10', 'mom5_mom10_active',
    'mom3_mom10', 'mom3_mom10_active',
    'mom3_mom5',  'mom3_mom5_active',
]

target_order = [
    'delta60',
    'delta90',
]

In [ ]:
import pandas as pd
from typing import List

def reorder_mnq_columns(
    df: pd.DataFrame,
    *,
    base_order: List[str],
    features_order: List[str],
    target_order: List[str],
) -> pd.DataFrame:
    """
    Reordena las columnas del dataset mnq_features_targets según un orden explícito.

    - NO modifica el índice.
    - NO altera valores.
    - Valida que todas las columnas requeridas existan.
    - Devuelve una copia del DataFrame con el nuevo orden.

    Orden final:
        base_order + features_order + target_order
    """
    required_cols = base_order + features_order + target_order

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"Faltan columnas requeridas en el DataFrame: {missing}"
        )

    return df[required_cols].copy()


In [ ]:
mnq_features_targets = reorder_mnq_columns(
    mnq_features_targets,
    base_order=[
        'date',
        'minute_of_day',
    ],
    features_order=[
        'close',
        'ema60',
        'roc60',

        'roc30', 'roc30_active',
        'roc20', 'roc20_active',

        'mom10', 'mom10_active',
        'mom5',  'mom5_active',
        'mom3',  'mom3_active',

        'mom5_mom10', 'mom5_mom10_active',
        'mom3_mom10', 'mom3_mom10_active',
        'mom3_mom5',  'mom3_mom5_active',
    ],
    target_order=[
        'delta60',
        'delta90',
    ],
)


In [ ]:
mnq_features_targets

,date,minute_of_day,close,ema60,roc60,roc30,roc30_active,roc20,roc20_active,mom10,...,mom3,mom3_active,mom5_mom10,mom5_mom10_active,mom3_mom10,mom3_mom10_active,mom3_mom5,mom3_mom5_active,delta60,delta90
datetime,,,,,,,,,,,,,,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,390,8727.75,0.000000,NaN,NaN,0,NaN,0,NaN,...,NaN,0,NaN,0,NaN,0,NaN,0,9.00,6.00
2019-12-23 06:31:00-05:00,2019-12-23,391,8726.50,-0.000139,NaN,NaN,0,NaN,0,NaN,...,NaN,0,NaN,0,NaN,0,NaN,0,9.25,7.50
2019-12-23 06:32:00-05:00,2019-12-23,392,8725.25,-0.000273,NaN,NaN,0,NaN,0,NaN,...,NaN,0,NaN,0,NaN,0,NaN,0,9.75,8.00
2019-12-23 06:33:00-05:00,2019-12-23,393,8726.00,-0.000180,NaN,NaN,0,NaN,0,NaN,...,-0.000201,0,NaN,0,NaN,0,NaN,0,8.50,8.00
2019-12-23 06:34:00-05:00,2019-12-23,394,8726.00,-0.000175,NaN,NaN,0,NaN,0,NaN,...,-0.000057,0,NaN,0,NaN,0,NaN,0,8.00,7.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 14:26:00-04:00,2025-06-13,866,21719.50,-0.001801,-0.357839,-0.309818,0,-0.193001,0,-0.000207,...,0.000438,0,-0.000092,0,0.000645,0,0.000737,0,-90.00,-102.00
2025-06-13 14:27:00-04:00,2025-06-13,867,21698.50,-0.002675,-0.419917,-0.406206,0,-0.241368,0,-0.000714,...,-0.001116,0,-0.000104,0,-0.000403,0,-0.000299,0,-69.25,-74.75
2025-06-13 14:28:00-04:00,2025-06-13,868,21679.25,-0.003444,-0.493419,-0.488852,0,-0.293198,0,-0.001577,...,-0.001750,0,0.000161,0,-0.000172,0,-0.000333,0,-52.25,-57.50


In [ ]:
OUT_PARQUET

PosixPath('/content/drive/MyDrive/neural_profit/data/features/mnq_features_target.parquet')

In [ ]:
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

# Guardar dataset
mnq_features_targets.to_parquet(OUT_PARQUET, index=True)